In [ ]:
%pip install -q "trl" "bitsandbytes"

In [ ]:
import os
import subprocess
from pathlib import Path

BASE = Path("/kaggle/working/api_pentest_research")
BASE.mkdir(parents=True, exist_ok=True)

repositories = {
    "ATRDF": {
        "url": "https://github.com/ArielCyber/Cisco_Ariel_Uni_API_security_challenge.git",
        "path": BASE / "ATRDF"
    },
    "WFD": {
        "url": "https://github.com/WebFuzzing/Dataset.git",
        "path": BASE / "WFD"
    }
}

for name, repo in repositories.items():
    path = repo["path"]

    if path.exists() and any(path.iterdir()):
        print(f"{name}: directory already exists, skipping clone")
        continue

    command = ["git", "clone", "--depth", "1"]

    if name == "WFD":
        command += ["--branch", "v4.1.0"]

    command += [repo["url"], str(path)]

    print(f"\nDownloading {name}...")
    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode == 0:
        print(f"{name}: download successful")
    else:
        print(f"{name}: download failed")
        print(result.stderr[-2000:])

print("\n--- DOWNLOADED FILE INVENTORY ---")

for name, repo in repositories.items():
    path = repo["path"]
    print(f"\n{name}")

    if not path.exists():
        print("Repository not found")
        continue

    files = sorted(p for p in path.rglob("*") if p.is_file())
    print("File count:", len(files))

    for file in files[:35]:
        print(f"{file.relative_to(path)} ({file.stat().st_size / 1024:.1f} KB)")

    for filename in ["README.md", "LICENSE", "LICENSE.md", "CITATION.cff"]:
        file = path / filename
        if file.exists():
            print(f"\n--- {filename} ---")
            print(file.read_text(errors="replace")[:5000])

print("\nResearch directory:", BASE)

In [ ]:
from pathlib import Path
import shutil
import subprocess

BASE = Path("/kaggle/working/api_pentest_research")
ATRDF = BASE / "ATRDF"
WFD = BASE / "WFD"

print("=== ATRDF ARCHIVES ===")
archives = sorted((ATRDF / "Datasets").glob("*.7z"))

for archive in archives:
    print(f"{archive.name}: {archive.stat().st_size / (1024**2):.2f} MB")

print("\n=== EXTRACTION TOOLS ===")
for tool in ["7z", "7zz", "7zr", "unar", "unzip"]:
    print(f"{tool}: {shutil.which(tool)}")

print("\n=== WFD TOP-LEVEL DIRECTORIES ===")
for path in sorted(WFD.iterdir()):
    if path.name != ".git":
        kind = "DIR " if path.is_dir() else "FILE"
        print(kind, path.name)

print("\n=== WFD DATA / DOCUMENTATION FILES ===")
count = 0

for path in WFD.rglob("*"):
    if not path.is_file() or ".git" in path.parts:
        continue

    if path.suffix.lower() in {
        ".yaml", ".yml", ".json", ".csv", ".tsv",
        ".md", ".xml", ".graphql", ".proto"
    }:
        print(path.relative_to(WFD))
        count += 1

        if count >= 60:
            print("... listing limited to 60 files")
            break

print("\n=== ARCHIVE EXTRACTION LIBRARY ===")
try:
    import py7zr
    print("py7zr available:", py7zr.__version__)
except ImportError:
    print("py7zr is not installed")

In [ ]:
from pathlib import Path
import subprocess

BASE = Path("/kaggle/working/api_pentest_research")
ARCHIVE_DIR = BASE / "ATRDF" / "Datasets"
EXTRACT_DIR = BASE / "ATRDF" / "extracted"
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

for archive in sorted(ARCHIVE_DIR.glob("*.7z")):
    output_dir = EXTRACT_DIR / archive.stem
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"\nExtracting {archive.name}...")

    result = subprocess.run(
        ["7z", "x", "-y", f"-o{output_dir}", str(archive)],
        capture_output=True,
        text=True
    )

    if result.returncode == 0:
        print("Extraction successful")
    else:
        print("Extraction failed:")
        print(result.stderr[-1500:])

print("\n=== EXTRACTED FILES ===")

for path in sorted(EXTRACT_DIR.rglob("*")):
    if path.is_file():
        print(
            f"{path.relative_to(EXTRACT_DIR)} "
            f"| {path.stat().st_size / 1024:.1f} KB"
        )

In [ ]:
import json
from pathlib import Path

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for file in sorted(BASE.rglob("*.json")):
    print("\n" + "=" * 80)
    print("FILE:", file.relative_to(BASE))
    print("SIZE (MB):", round(file.stat().st_size / (1024**2), 2))

    try:
        with file.open("r", encoding="utf-8") as f:
            first = f.read(2_000_000)

        data = json.loads(first) if first.strip().endswith(("}", "]")) else None

        if data is None:
            with file.open("r", encoding="utf-8") as f:
                data = json.load(f)

        print("TOP-LEVEL TYPE:", type(data).__name__)

        if isinstance(data, list):
            print("RECORD COUNT:", len(data))
            if data:
                print("FIRST RECORD TYPE:", type(data[0]).__name__)
                if isinstance(data[0], dict):
                    print("KEYS:", list(data[0].keys()))
                    print("SAMPLE:", json.dumps(data[0], ensure_ascii=False)[:1800])

        elif isinstance(data, dict):
            print("TOP-LEVEL KEYS:", list(data.keys())[:40])
            for key, value in list(data.items())[:3]:
                print(f"VALUE {key!r}: type={type(value).__name__}")
                if isinstance(value, list) and value:
                    print("FIRST ITEM:", json.dumps(value[0], ensure_ascii=False)[:1800])
                elif isinstance(value, dict):
                    print("INNER KEYS:", list(value.keys())[:30])

    except Exception as e:
        print("Could not parse with sample method:", repr(e))
        print("Trying streaming-style preview...")
        with file.open("r", encoding="utf-8", errors="replace") as f:
            print(f.read(1200))

In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for file in sorted(BASE.rglob("*.json")):
    print("\n" + "=" * 70)
    print("FILE:", file.relative_to(BASE))

    with file.open("r", encoding="utf-8") as f:
        records = json.load(f)

    print("Total records:", len(records))

    tags = Counter()
    methods = Counter()
    status_codes = Counter()
    missing_request = 0
    missing_response = 0

    for record in records:
        request = record.get("request")
        response = record.get("response")

        if not isinstance(request, dict):
            missing_request += 1
            request = {}

        if not isinstance(response, dict):
            missing_response += 1
            response = {}

        tag = request.get("Attack_Tag", record.get("Attack_Tag"))
        tags[str(tag) if tag is not None else "NO_TAG"] += 1

        methods[str(request.get("method", "UNKNOWN")).upper()] += 1
        status_codes[str(response.get("status_code", "UNKNOWN"))] += 1

    print("Attack tags:", dict(tags.most_common(20)))
    print("HTTP methods:", dict(methods))
    print("Response codes:", dict(status_codes.most_common(15)))
    print("Missing request objects:", missing_request)
    print("Missing response objects:", missing_response)

    del records

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research/ATRDF")

terms = [
    "Attack_Tag",
    "dataset_1_train",
    "validation",
    "benign",
    "malicious",
    "license",
    "licence"
]

for term in terms:
    print(f"\n{'=' * 25} {term} {'=' * 25}")
    matches = 0

    for file in ROOT.rglob("*"):
        if not file.is_file() or ".git" in file.parts:
            continue
        if file.suffix.lower() not in {".md", ".py", ".txt", ".yaml", ".yml", ".json", ".cff"}:
            continue
        try:
            if file.stat().st_size > 5_000_000:
                continue
            lines = file.read_text(encoding="utf-8", errors="ignore").splitlines()
            for line_no, line in enumerate(lines, 1):
                if term.lower() in line.lower():
                    print(f"{file.relative_to(ROOT)}:{line_no}: {line.strip()[:240]}")
                    matches += 1
                    if matches >= 12:
                        break
        except Exception:
            pass

        if matches >= 12:
            break

    if matches == 0:
        print("No matches found.")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research/ATRDF")
readme = ROOT / "README.md"

lines = readme.read_text(encoding="utf-8", errors="ignore").splitlines()

print("=== README: class mapping and submission instructions ===")
for start, end in [(145, 180), (1, 25), (115, 145)]:
    print(f"\n--- Lines {start}-{end} ---")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")

print("\n=== Repository files (excluding extracted data) ===")
count = 0
for path in sorted(ROOT.rglob("*")):
    if not path.is_file() or "extracted" in path.parts or ".git" in path.parts:
        continue
    print(path.relative_to(ROOT))
    count += 1
    if count >= 100:
        print("Output limited to 100 files.")
        break

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research/ATRDF")
NOTEBOOK = ROOT / "baseline_model.ipynb"

if not NOTEBOOK.exists():
    print("Baseline notebook not found:", NOTEBOOK)
else:
    with NOTEBOOK.open("r", encoding="utf-8") as f:
        notebook = json.load(f)

    for i, cell in enumerate(notebook.get("cells", [])):
        source = "".join(cell.get("source", []))
        if not source.strip():
            continue

        print(f"\n{'=' * 24} CELL {i} ({cell.get('cell_type')}) {'=' * 24}")
        print(source[:5000])

print("\n=== Licence and citation files ===")
for name in ["LICENSE", "LICENSE.md", "CITATION.cff"]:
    path = ROOT / name
    if path.exists():
        print(f"\n--- {name} ---")
        print(path.read_text(encoding="utf-8", errors="ignore")[:5000])

In [ ]:
import json
from pathlib import Path
from collections import Counter

DATA = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")
FILE = DATA / "dataset_1_train" / "dataset_1_train.json"

with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

print("Dataset:", FILE.name)
print("Total records:", len(records))

request_fields = Counter()
response_fields = Counter()
label_counts = Counter()
methods = Counter()
status_codes = Counter()
request_signatures = Counter()

for record in records:
    req = record.get("request") or {}
    resp = record.get("response") or {}

    request_fields.update(req.keys())
    response_fields.update(resp.keys())

    tag = req.get("Attack_Tag")
    label = "Benign" if tag is None or str(tag).strip() == "" else "Malware"
    label_counts[label] += 1

    attack_type = str(tag).strip() if tag is not None and str(tag).strip() else "Benign"
    methods[str(req.get("method", "UNKNOWN")).upper()] += 1
    status_codes[str(resp.get("status_code", "UNKNOWN"))] += 1

    signature = (
        str(req.get("method", "")),
        str(req.get("url", "")),
        str(req.get("body", ""))
    )
    request_signatures[signature] += 1

print("\nRequest fields:")
print(sorted(request_fields))

print("\nResponse fields:")
print(sorted(response_fields))

print("\nBinary label counts:")
print(dict(label_counts))

print("\nAttack-type counts:")
print(dict(Counter(
    (str(r.get("request", {}).get("Attack_Tag")).strip()
     if r.get("request", {}).get("Attack_Tag") is not None
     and str(r.get("request", {}).get("Attack_Tag")).strip()
     else "Benign")
    for r in records
)))

print("\nHTTP methods:", dict(methods))
print("Status codes:", dict(status_codes))
print("Repeated request signatures:", sum(n > 1 for n in request_signatures.values()))
print("Records belonging to repeated signatures:",
      sum(n for n in request_signatures.values() if n > 1))

print("\nSample record keys and field types:")
sample = records[0]
for section in ["request", "response"]:
    print(f"\n{section}:")
    for key, value in sample.get(section, {}).items():
        if key == "Attack_Tag":
            continue
        preview = str(value).replace("\n", " ")[:160]
        print(f"  {key}: {type(value).__name__} | {preview}")

del records

In [ ]:
import json
from pathlib import Path
from collections import defaultdict, Counter

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_1_train/dataset_1_train.json"
)

with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

groups = defaultdict(list)

for i, record in enumerate(records):
    req = record.get("request") or {}
    tag = req.get("Attack_Tag")
    label = "Benign" if tag is None or str(tag).strip() == "" else "Malware"

    signature = (
        str(req.get("method", "")).upper(),
        str(req.get("url", "")),
        str(req.get("body", ""))
    )
    groups[signature].append((i, label, str(tag) if tag is not None else "NO_TAG"))

duplicate_groups = {sig: rows for sig, rows in groups.items() if len(rows) > 1}
conflicting_groups = {
    sig: rows for sig, rows in duplicate_groups.items()
    if len({row[1] for row in rows}) > 1
}

print("Total records:", len(records))
print("Unique signatures:", len(groups))
print("Duplicate signature groups:", len(duplicate_groups))
print("Records in duplicate groups:", sum(len(rows) for rows in duplicate_groups.values()))
print("Duplicate groups with conflicting binary labels:", len(conflicting_groups))

print("\nBinary labels across all records:")
print(dict(Counter(
    "Benign" if (r.get("request") or {}).get("Attack_Tag") is None
    or str((r.get("request") or {}).get("Attack_Tag")).strip() == ""
    else "Malware"
    for r in records
)))

if conflicting_groups:
    print("\nExamples of conflicting groups:")
    for signature, rows in list(conflicting_groups.items())[:5]:
        print({
            "method": signature[0],
            "url": signature[1][:180],
            "body": signature[2][:100],
            "labels_and_tags": [(label, tag) for _, label, tag in rows]
        })

print("\nRepeated groups with consistent binary labels:",
      len(duplicate_groups) - len(conflicting_groups))

del records, groups, duplicate_groups, conflicting_groups

In [ ]:
import json
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_1_train/dataset_1_train.json"
)

with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

def make_text(record):
    req = record.get("request") or {}
    resp = record.get("response") or {}

    req_headers = req.get("headers") or {}
    resp_headers = resp.get("headers") or {}

    parts = [
        "METHOD " + str(req.get("method", "")),
        "URL " + str(req.get("url", "")),
        "REQUEST_BODY " + str(req.get("body", "")),
        "REQUEST_HEADERS " + json.dumps(req_headers, sort_keys=True),
        "STATUS " + str(resp.get("status_code", "")),
        "RESPONSE_BODY " + str(resp.get("body", "")),
        "RESPONSE_HEADERS " + json.dumps(resp_headers, sort_keys=True),
    ]
    return " ".join(parts)

def make_group(record):
    req = dict(record.get("request") or {})
    req.pop("Attack_Tag", None)
    return json.dumps(req, sort_keys=True, ensure_ascii=False)

df = pd.DataFrame({
    "text": [make_text(r) for r in records],
    "group": [make_group(r) for r in records],
    "label": [
        "Benign" if not str((r.get("request") or {}).get("Attack_Tag") or "").strip()
        else "Malware"
        for r in records
    ]
})

X = df["text"]
y = df["label"]
groups = df["group"]

def build_model():
    return Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

def evaluate(name, train_idx, test_idx):
    model = build_model()
    model.fit(X.iloc[train_idx], y.iloc[train_idx])
    pred = model.predict(X.iloc[test_idx])

    print(f"\n{'=' * 20} {name} {'=' * 20}")
    print("Train records:", len(train_idx))
    print("Test records:", len(test_idx))
    print("Train labels:", y.iloc[train_idx].value_counts().to_dict())
    print("Test labels:", y.iloc[test_idx].value_counts().to_dict())
    print("Accuracy:", round(accuracy_score(y.iloc[test_idx], pred), 4))
    print("Macro F1:", round(f1_score(y.iloc[test_idx], pred, average="macro"), 4))
    print("Confusion matrix [Benign, Malware]:")
    print(confusion_matrix(
        y.iloc[test_idx], pred, labels=["Benign", "Malware"]
    ))
    print(classification_report(
        y.iloc[test_idx], pred, labels=["Benign", "Malware"], zero_division=0
    ))

    return model

indices = np.arange(len(df))

random_train, random_test = train_test_split(
    indices,
    test_size=0.1765,
    random_state=42,
    stratify=y
)

group_split = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
group_train, group_test = next(group_split.split(X, y, groups=groups))

print("Dataset 1 records:", len(df))
print("Unique full-request groups:", groups.nunique())
print("Groups crossing the random split:",
      len(set(groups.iloc[random_train]) & set(groups.iloc[random_test])))
print("Groups crossing the group split:",
      len(set(groups.iloc[group_train]) & set(groups.iloc[group_test])))

random_model = evaluate("Random stratified split", random_train, random_test)
group_model = evaluate("Full-request group split", group_train, group_test)

del records, df

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlsplit

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_1_train/dataset_1_train.json"
)

with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

def request_text(record):
    req = record.get("request") or {}
    return " ".join([
        "METHOD " + str(req.get("method", "")),
        "URL " + str(req.get("url", "")),
        "BODY " + str(req.get("body", "")),
        "HEADERS " + json.dumps(req.get("headers") or {}, sort_keys=True)
    ])

def response_text(record):
    resp = record.get("response") or {}
    return " ".join([
        "STATUS " + str(resp.get("status_code", "")),
        "RESPONSE_BODY " + str(resp.get("body", "")),
        "RESPONSE_HEADERS " + json.dumps(resp.get("headers") or {}, sort_keys=True)
    ])

def endpoint_group(record):
    req = record.get("request") or {}
    url = str(req.get("url", ""))
    parsed = urlsplit(url)
    return str(req.get("method", "")).upper() + " " + (parsed.path or "/")

request_features = [request_text(r) for r in records]
combined_features = [
    request_text(r) + " RESPONSE " + response_text(r)
    for r in records
]
labels = [
    "Benign" if not str((r.get("request") or {}).get("Attack_Tag") or "").strip()
    else "Malware"
    for r in records
]
groups = [endpoint_group(r) for r in records]

X_idx = list(range(len(records)))
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(X_idx, labels, groups=groups)
)

print("Records:", len(records))
print("Unique endpoint groups:", len(set(groups)))
print("Train/test records:", len(train_idx), "/", len(test_idx))
print("Endpoint groups crossing split:",
      len(set(groups[i] for i in train_idx) &
          set(groups[i] for i in test_idx)))
print("Test label counts:",
      pd.Series([labels[i] for i in test_idx]).value_counts().to_dict())

def run_experiment(name, features):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit([features[i] for i in train_idx],
              [labels[i] for i in train_idx])

    predictions = model.predict([features[i] for i in test_idx])
    truth = [labels[i] for i in test_idx]

    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(truth, predictions), 4))
    print("Macro F1:", round(f1_score(truth, predictions, average="macro"), 4))
    print(classification_report(truth, predictions, zero_division=0))

run_experiment("Request-only classifier", request_features)
run_experiment("Request + response classifier", combined_features)

del records, request_features, combined_features, labels, groups

In [ ]:
import json
import pandas as pd
from pathlib import Path
from urllib.parse import urlsplit

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_1_train/dataset_1_train.json"
)

with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

labels = []
request_texts = []
endpoint_texts = []
groups = []

for r in records:
    req = r.get("request") or {}
    tag = str(req.get("Attack_Tag") or "").strip()

    labels.append("Malware" if tag else "Benign")

    url = str(req.get("url", ""))
    parsed = urlsplit(url)
    path = parsed.path or "/"
    method = str(req.get("method", "")).upper()

    endpoint_texts.append(f"METHOD {method} PATH {path}")

    request_texts.append(" ".join([
        f"METHOD {method}",
        f"URL {url}",
        f"BODY {req.get('body', '')}",
        "HEADERS " + json.dumps(req.get("headers") or {}, sort_keys=True)
    ]))

    groups.append(f"{method} {path}")

indices = list(range(len(records)))
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(indices, labels, groups=groups)
)

def evaluate(name, texts):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(
        [texts[i] for i in train_idx],
        [labels[i] for i in train_idx]
    )
    pred = model.predict([texts[i] for i in test_idx])
    truth = [labels[i] for i in test_idx]

    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(truth, pred), 4))
    print("Macro F1:", round(f1_score(truth, pred, average="macro"), 4))
    print(classification_report(truth, pred, zero_division=0))

print("Train/test:", len(train_idx), "/", len(test_idx))
print("Endpoint groups crossing split:",
      len(set(groups[i] for i in train_idx) &
          set(groups[i] for i in test_idx)))

evaluate("Endpoint-only baseline", endpoint_texts)
evaluate("Full request baseline", request_texts)

del records, labels, request_texts, endpoint_texts, groups

In [ ]:
import json
import pandas as pd
from pathlib import Path
from urllib.parse import urlsplit

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_2_train/dataset_2_train.json"
)

print("Loading Dataset 2...")
with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

labels = []
request_texts = []
endpoint_texts = []
groups = []

for r in records:
    req = r.get("request") or {}
    tag = str(req.get("Attack_Tag") or "").strip()
    labels.append("Malware" if tag else "Benign")

    method = str(req.get("method", "")).upper()
    url = str(req.get("url", ""))
    path = urlsplit(url).path or "/"
    group = f"{method} {path}"
    groups.append(group)

    endpoint_texts.append(f"METHOD {method} PATH {path}")
    request_texts.append(" ".join([
        f"METHOD {method}",
        f"URL {url}",
        f"BODY {req.get('body', '')}",
        "HEADERS " + json.dumps(req.get("headers") or {}, sort_keys=True)
    ]))

print("Records:", len(records))
print("Label counts:", pd.Series(labels).value_counts().to_dict())
print("Unique endpoint groups:", len(set(groups)))

indices = list(range(len(records)))
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(indices, labels, groups=groups)
)

print("Train/test:", len(train_idx), "/", len(test_idx))
print("Endpoint groups crossing split:",
      len(set(groups[i] for i in train_idx) &
          set(groups[i] for i in test_idx)))

def evaluate(name, texts):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(
        [texts[i] for i in train_idx],
        [labels[i] for i in train_idx]
    )
    pred = model.predict([texts[i] for i in test_idx])
    truth = [labels[i] for i in test_idx]

    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(truth, pred), 4))
    print("Macro F1:", round(f1_score(truth, pred, average="macro"), 4))
    print(classification_report(truth, pred, zero_division=0))

evaluate("Dataset 2 endpoint-only", endpoint_texts)
evaluate("Dataset 2 full-request", request_texts)

del records, labels, request_texts, endpoint_texts, groups

In [ ]:
import json
import pandas as pd
from pathlib import Path
from urllib.parse import urlsplit

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_3_train/dataset_3_train.json"
)

print("Loading Dataset 3...")
with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

labels = []
request_texts = []
endpoint_texts = []
groups = []

for r in records:
    req = r.get("request") or {}
    tag = str(req.get("Attack_Tag") or "").strip()
    labels.append("Malware" if tag else "Benign")

    method = str(req.get("method", "")).upper()
    url = str(req.get("url", ""))
    path = urlsplit(url).path or "/"
    groups.append(f"{method} {path}")

    endpoint_texts.append(f"METHOD {method} PATH {path}")
    request_texts.append(" ".join([
        f"METHOD {method}",
        f"URL {url}",
        f"BODY {req.get('body', '')}",
        "HEADERS " + json.dumps(
            req.get("headers") or {}, sort_keys=True
        )
    ]))

print("Records:", len(records))
print("Label counts:", pd.Series(labels).value_counts().to_dict())
print("Unique endpoint groups:", len(set(groups)))

indices = list(range(len(records)))
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
train_idx, test_idx = next(
    splitter.split(indices, labels, groups=groups)
)

print("Train/test:", len(train_idx), "/", len(test_idx))
print(
    "Endpoint groups crossing split:",
    len(set(groups[i] for i in train_idx) &
        set(groups[i] for i in test_idx))
)

def evaluate(name, texts):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(
        [texts[i] for i in train_idx],
        [labels[i] for i in train_idx]
    )
    pred = model.predict([texts[i] for i in test_idx])
    truth = [labels[i] for i in test_idx]

    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(truth, pred), 4))
    print("Macro F1:", round(
        f1_score(truth, pred, average="macro"), 4
    ))
    print(classification_report(
        truth, pred, zero_division=0
    ))

evaluate("Dataset 3 endpoint-only", endpoint_texts)
evaluate("Dataset 3 full-request", request_texts)

del records, labels, request_texts, endpoint_texts, groups

In [ ]:
import json
import pandas as pd
from pathlib import Path
from urllib.parse import urlsplit

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

FILE = Path(
    "/kaggle/working/api_pentest_research/ATRDF/extracted/"
    "dataset_4_train/dataset_4_train.json"
)

print("Loading Dataset 4...")
with FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

labels = []
request_texts = []
endpoint_texts = []
groups = []

for r in records:
    req = r.get("request") or {}
    tag = str(req.get("Attack_Tag") or "").strip()
    labels.append("Malware" if tag else "Benign")

    method = str(req.get("method", "")).upper()
    url = str(req.get("url", ""))
    path = urlsplit(url).path or "/"
    groups.append(f"{method} {path}")

    endpoint_texts.append(f"METHOD {method} PATH {path}")
    request_texts.append(" ".join([
        f"METHOD {method}",
        f"URL {url}",
        f"BODY {req.get('body', '')}",
        "HEADERS " + json.dumps(
            req.get("headers") or {}, sort_keys=True
        )
    ]))

print("Records:", len(records))
print("Label counts:", pd.Series(labels).value_counts().to_dict())
print("Unique endpoint groups:", len(set(groups)))

indices = list(range(len(records)))
splitter = GroupShuffleSplit(
    n_splits=1, test_size=0.20, random_state=42
)
train_idx, test_idx = next(
    splitter.split(indices, labels, groups=groups)
)

print("Train/test:", len(train_idx), "/", len(test_idx))
print(
    "Endpoint groups crossing split:",
    len(set(groups[i] for i in train_idx) &
        set(groups[i] for i in test_idx))
)

def evaluate(name, texts):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=2,
            max_features=100000,
            sublinear_tf=True
        )),
        ("clf", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(
        [texts[i] for i in train_idx],
        [labels[i] for i in train_idx]
    )
    pred = model.predict([texts[i] for i in test_idx])
    truth = [labels[i] for i in test_idx]

    print(f"\n=== {name} ===")
    print("Accuracy:", round(accuracy_score(truth, pred), 4))
    print("Macro F1:", round(
        f1_score(truth, pred, average="macro"), 4
    ))
    print(classification_report(
        truth, pred, zero_division=0
    ))

evaluate("Dataset 4 endpoint-only", endpoint_texts)
evaluate("Dataset 4 full-request", request_texts)

del records, labels, request_texts, endpoint_texts, groups

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlsplit
from collections import Counter

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

def inspect_dataset(dataset_name):
    file_path = BASE / dataset_name / f"{dataset_name}.json"
    endpoints = Counter()
    labels = Counter()
    endpoint_labels = {}

    with file_path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        req = record.get("request") or {}
        method = str(req.get("method", "")).upper()
        path = urlsplit(str(req.get("url", ""))).path or "/"
        endpoint = f"{method} {path}"

        tag = str(req.get("Attack_Tag") or "").strip()
        label = "Malware" if tag else "Benign"

        endpoints[endpoint] += 1
        labels[label] += 1
        endpoint_labels.setdefault(endpoint, Counter())[label] += 1

    print(
        f"{dataset_name}: {len(records):,} records, "
        f"{len(endpoints):,} unique endpoints, labels={dict(labels)}"
    )
    del records
    return set(endpoints), endpoint_labels

results = {}
for name in ["dataset_1_train", "dataset_2_train",
             "dataset_3_train", "dataset_4_train"]:
    results[name] = inspect_dataset(name)

print("\n=== Endpoint overlap with Dataset 4 ===")
target_name = "dataset_4_train"
target_endpoints = results[target_name][0]

for name in ["dataset_1_train", "dataset_2_train", "dataset_3_train"]:
    source_endpoints = results[name][0]
    shared = source_endpoints & target_endpoints
    print(
        f"{name} -> Dataset 4: "
        f"{len(shared):,} shared endpoints; "
        f"{len(shared) / max(1, len(target_endpoints)):.1%} "
        "of Dataset 4 endpoints covered"
    )

print("\n=== Endpoint label consistency across datasets ===")
for name in ["dataset_1_train", "dataset_2_train", "dataset_3_train"]:
    source_labels = results[name][1]
    shared = set(source_labels) & target_endpoints

    both_classes = sum(
        len(source_labels[e]) > 1
        for e in shared
    )

    print(
        f"{name}: {len(shared):,} shared endpoints; "
        f"{both_classes:,} have both Benign and Malware labels "
        "within that dataset"
    )

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlsplit

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

def load_dataset(name):
    path = BASE / name / f"{name}.json"
    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    endpoint_texts = []
    request_texts = []
    labels = []

    for record in records:
        req = record.get("request") or {}

        method = str(req.get("method", "")).upper()
        url = str(req.get("url", ""))
        path = urlsplit(url).path or "/"

        tag = str(req.get("Attack_Tag") or "").strip()
        labels.append("Malware" if tag else "Benign")

        endpoint_texts.append(f"METHOD {method} PATH {path}")

        request_texts.append(" ".join([
            f"METHOD {method}",
            f"URL {url}",
            f"BODY {req.get('body', '')}",
            "HEADERS " + json.dumps(
                req.get("headers") or {}, sort_keys=True
            )
        ]))

    del records
    return endpoint_texts, request_texts, labels

print("Loading source Dataset 3...")
source_endpoint, source_request, y_train = load_dataset("dataset_3_train")

print("Loading separate target Dataset 4...")
target_endpoint, target_request, y_test = load_dataset("dataset_4_train")

print("Source records:", len(y_train))
print("Source labels:", {
    label: y_train.count(label) for label in set(y_train)
})
print("Target records:", len(y_test))
print("Target labels:", {
    label: y_test.count(label) for label in set(y_test)
})

def cross_dataset_test(name, train_texts, test_texts):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=100000,
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print(f"\n=== Dataset 3 -> Dataset 4: {name} ===")
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print("Macro F1:", round(
        f1_score(y_test, predictions, average="macro"), 4
    ))
    print(classification_report(
        y_test,
        predictions,
        labels=["Benign", "Malware"],
        zero_division=0
    ))

    del vectorizer, X_train, X_test, model, predictions

cross_dataset_test(
    "Endpoint-only",
    source_endpoint,
    target_endpoint
)

cross_dataset_test(
    "Full-request",
    source_request,
    target_request
)

del source_endpoint, source_request
del target_endpoint, target_request, y_train, y_test

print("\nCross-dataset evaluation complete.")

In [ ]:
import json
from pathlib import Path
from urllib.parse import urlsplit

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

def load_dataset(name):
    file_path = BASE / name / f"{name}.json"

    with file_path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    endpoints, requests, labels = [], [], []

    for record in records:
        req = record.get("request") or {}
        method = str(req.get("method", "")).upper()
        url = str(req.get("url", ""))
        path = urlsplit(url).path or "/"

        tag = str(req.get("Attack_Tag") or "").strip()
        labels.append("Malware" if tag else "Benign")

        endpoints.append(f"METHOD {method} PATH {path}")
        requests.append(" ".join([
            f"METHOD {method}",
            f"URL {url}",
            f"BODY {req.get('body', '')}",
            "HEADERS " + json.dumps(
                req.get("headers") or {}, sort_keys=True
            )
        ]))

    del records
    return endpoints, requests, labels

print("Loading source Dataset 4...")
src_endpoint, src_request, y_train = load_dataset("dataset_4_train")

print("Loading target Dataset 3...")
tgt_endpoint, tgt_request, y_test = load_dataset("dataset_3_train")

print("Source records:", len(y_train))
print("Target records:", len(y_test))

def evaluate(name, train_texts, test_texts):
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_features=100000,
        sublinear_tf=True
    )

    X_train = vectorizer.fit_transform(train_texts)
    X_test = vectorizer.transform(test_texts)

    model = LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    )
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print(f"\n=== Dataset 4 -> Dataset 3: {name} ===")
    print("Accuracy:", round(accuracy_score(y_test, predictions), 4))
    print("Macro F1:", round(
        f1_score(y_test, predictions, average="macro"), 4
    ))
    print(classification_report(
        y_test,
        predictions,
        labels=["Benign", "Malware"],
        zero_division=0
    ))

    del vectorizer, X_train, X_test, model, predictions

evaluate("Endpoint-only", src_endpoint, tgt_endpoint)
evaluate("Full-request", src_request, tgt_request)

del src_endpoint, src_request, tgt_endpoint, tgt_request
del y_train, y_test

print("\nReverse cross-dataset evaluation complete.")

In [ ]:
import shutil
import subprocess
from pathlib import Path

BASE = Path("/kaggle/working/api_pentest_research")

print("=== Project directories ===")
for path in sorted(BASE.iterdir()):
    if path.is_dir():
        print(path.name)

print("\n=== Docker availability ===")
docker = shutil.which("docker")
print("Docker executable:", docker)

if docker:
    result = subprocess.run(
        ["docker", "version"],
        capture_output=True,
        text=True,
        timeout=20
    )
    print("Docker version check exit code:", result.returncode)
    print((result.stdout or result.stderr)[:3000])
else:
    print("Docker is not available on PATH.")

print("\n=== API benchmark Compose files ===")
compose_files = []

for root in BASE.iterdir():
    if root.is_dir():
        for pattern in ("docker-compose.yml", "docker-compose.yaml",
                        "compose.yml", "compose.yaml"):
            compose_files.extend(root.rglob(pattern))

for file in compose_files[:60]:
    print(file.relative_to(BASE))

print("Compose files found:", len(compose_files))

In [ ]:
import importlib.util
import socket
import sys
import urllib.request
import json

print("=== Python environment ===")
print(sys.version)

for package in ["flask", "fastapi", "uvicorn", "requests", "httpx"]:
    print(f"{package}:",
          "available" if importlib.util.find_spec(package) else "missing")

print("\n=== Localhost port availability ===")
for port in [8000, 5000, 8080]:
    sock = socket.socket()
    try:
        sock.bind(("127.0.0.1", port))
        print(f"127.0.0.1:{port}: available")
    except OSError as e:
        print(f"127.0.0.1:{port}: unavailable ({e})")
    finally:
        sock.close()

print("\n=== Kaggle workspace ===")
from pathlib import Path
print("Working directory:", Path.cwd())
print("Writable:", __import__("os").access("/kaggle/working", __import__("os").W_OK))

In [ ]:
import threading
import time
from flask import Flask, jsonify, request

app = Flask("controlled_api_lab")

LAB_STATE = {
    "users": {
        "alice": {"id": "alice", "role": "user"},
        "bob": {"id": "bob", "role": "user"},
        "admin": {"id": "admin", "role": "admin"}
    },
    "documents": {
        "101": {"id": "101", "owner": "alice", "content": "ALICE_PRIVATE_CANARY"},
        "102": {"id": "102", "owner": "bob", "content": "BOB_PRIVATE_CANARY"}
    },
    "audit": []
}

@app.get("/health")
def health():
    return jsonify({"status": "ok", "lab": "controlled-api-lab"})

@app.get("/api")
def api_catalogue():
    return jsonify({
        "endpoints": [
            {"method": "GET", "path": "/health"},
            {"method": "GET", "path": "/api"},
            {"method": "GET", "path": "/api/profile"},
            {"method": "GET", "path": "/api/documents/<document_id>"},
            {"method": "POST", "path": "/api/search"}
        ]
    })

@app.get("/api/profile")
def profile():
    user_id = request.headers.get("X-Lab-User")
    if user_id not in LAB_STATE["users"]:
        return jsonify({"error": "authentication required"}), 401
    return jsonify(LAB_STATE["users"][user_id])

@app.get("/api/documents/<document_id>")
def get_document(document_id):
    user_id = request.headers.get("X-Lab-User")
    if user_id not in LAB_STATE["users"]:
        return jsonify({"error": "authentication required"}), 401

    document = LAB_STATE["documents"].get(document_id)
    if document is None:
        return jsonify({"error": "not found"}), 404

    # This lab intentionally omits the ownership check for testing.
    LAB_STATE["audit"].append({
        "event": "document_read",
        "user": user_id,
        "document_id": document_id
    })
    return jsonify(document)

@app.post("/api/search")
def search():
    data = request.get_json(silent=True) or {}
    term = str(data.get("term", ""))

    # Safe simulated behavior: no SQL engine or command execution.
    if any(marker in term.lower() for marker in ["' or '1'='1", "union select"]):
        return jsonify({
            "results": ["SIMULATED_INJECTION_SIGNAL"],
            "simulation": True
        })

    return jsonify({"results": [], "simulation": True})

def start_lab():
    app.run(
        host="127.0.0.1",
        port=8000,
        debug=False,
        use_reloader=False,
        threaded=True
    )

if "api_lab_thread" not in globals() or not api_lab_thread.is_alive():
    api_lab_thread = threading.Thread(target=start_lab, daemon=True)
    api_lab_thread.start()
    time.sleep(1)

import requests

response = requests.get("http://127.0.0.1:8000/health", timeout=5)
print("HTTP status:", response.status_code)
print("Response:", response.json())
print("Local API lab health check passed:", response.status_code == 200)

In [ ]:
import requests
import json

BASE_URL = "http://127.0.0.1:8000"
results = []

def record_test(name, response, verified, explanation):
    results.append({
        "test": name,
        "http_status": response.status_code,
        "verified": bool(verified),
        "explanation": explanation,
        "response": response.json()
    })

# Test 1: API availability
r = requests.get(f"{BASE_URL}/health", timeout=5)
record_test(
    "health_check",
    r,
    r.status_code == 200 and r.json().get("status") == "ok",
    "Checks that the local API responds with its expected health marker."
)

# Test 2: Protected profile without credentials
r = requests.get(f"{BASE_URL}/api/profile", timeout=5)
record_test(
    "unauthenticated_profile",
    r,
    r.status_code == 401,
    "Passes if the protected profile rejects a request without a lab user."
)

# Test 3: Bob attempts to read Alice's seeded private document
r = requests.get(
    f"{BASE_URL}/api/documents/101",
    headers={"X-Lab-User": "bob"},
    timeout=5
)
body = r.json()
cross_user_access = (
    r.status_code == 200
    and body.get("owner") == "alice"
    and body.get("content") == "ALICE_PRIVATE_CANARY"
)
record_test(
    "cross_user_document_access",
    r,
    cross_user_access,
    "Confirmed lab access-control failure only if Bob receives Alice's exact private canary."
)

# Test 4: Simulated injection signal (not real SQL execution)
r = requests.post(
    f"{BASE_URL}/api/search",
    json={"term": "' OR '1'='1"},
    timeout=5
)
body = r.json()
simulation_signal = (
    r.status_code == 200
    and body.get("simulation") is True
    and "SIMULATED_INJECTION_SIGNAL" in body.get("results", [])
)
record_test(
    "simulated_injection_signal",
    r,
    simulation_signal,
    "Confirms the predefined simulation marker; does not prove SQL injection."
)

print(json.dumps(results, indent=2))
print("\n=== Summary ===")
print("Tests executed:", len(results))
print("Verified expected outcomes:", sum(x["verified"] for x in results))
print("Cross-user access confirmed:", cross_user_access)
print("Actual SQL injection tested: No")

In [ ]:
import requests
import pandas as pd

BASE_URL = "http://127.0.0.1:8000"

TEST_ACTIONS = {
    "health_check": {
        "method": "GET",
        "path": "/health",
        "headers": {},
        "expected": "control"
    },
    "unauthenticated_profile": {
        "method": "GET",
        "path": "/api/profile",
        "headers": {},
        "expected": "control"
    },
    "alice_reads_own_document": {
        "method": "GET",
        "path": "/api/documents/101",
        "headers": {"X-Lab-User": "alice"},
        "expected": "control"
    },
    "bob_reads_alice_document": {
        "method": "GET",
        "path": "/api/documents/101",
        "headers": {"X-Lab-User": "bob"},
        "expected": "access_control"
    },
    "alice_reads_bob_document": {
        "method": "GET",
        "path": "/api/documents/102",
        "headers": {"X-Lab-User": "alice"},
        "expected": "access_control"
    },
    "ordinary_search": {
        "method": "POST",
        "path": "/api/search",
        "headers": {},
        "json": {"term": "ordinary search term"},
        "expected": "control"
    },
    "injection_simulation": {
        "method": "POST",
        "path": "/api/search",
        "headers": {},
        "json": {"term": "' OR '1'='1"},
        "expected": "simulation"
    }
}

def execute_action(name, spec):
    response = requests.request(
        method=spec["method"],
        url=BASE_URL + spec["path"],
        headers=spec.get("headers", {}),
        json=spec.get("json"),
        timeout=5
    )

    body = response.json()
    expected = spec["expected"]
    outcome = "unexpected"

    if expected == "control":
        if name == "health_check":
            passed = (
                response.status_code == 200
                and body.get("status") == "ok"
            )
        elif name == "unauthenticated_profile":
            passed = response.status_code == 401
        elif name == "ordinary_search":
            passed = (
                response.status_code == 200
                and body.get("results") == []
            )
        else:
            user = spec.get("headers", {}).get("X-Lab-User")
            passed = (
                response.status_code == 200
                and body.get("owner") == user
            )
        outcome = "control_passed" if passed else "control_failed"

    elif expected == "access_control":
        requested_user = spec["headers"]["X-Lab-User"]
        confirmed = (
            response.status_code == 200
            and body.get("owner") != requested_user
            and body.get("content") in {
                "ALICE_PRIVATE_CANARY",
                "BOB_PRIVATE_CANARY"
            }
        )
        outcome = (
            "confirmed_lab_access_control"
            if confirmed else "not_confirmed"
        )

    elif expected == "simulation":
        confirmed = (
            response.status_code == 200
            and body.get("simulation") is True
            and "SIMULATED_INJECTION_SIGNAL" in body.get("results", [])
        )
        outcome = (
            "simulation_signal_only" if confirmed
            else "simulation_not_observed"
        )

    return {
        "action": name,
        "http_status": response.status_code,
        "outcome": outcome,
        "confirmed_lab_vulnerability": (
            outcome == "confirmed_lab_access_control"
        ),
        "reward": (
            10 if outcome == "confirmed_lab_access_control"
            else 1 if outcome == "simulation_signal_only"
            else 0
        )
    }

rows = [
    execute_action(name, spec)
    for name, spec in TEST_ACTIONS.items()
]

results_df = pd.DataFrame(rows)
display(results_df)

print("\nOutcome counts:")
print(results_df["outcome"].value_counts().to_dict())

print(
    "\nConfirmed access-control test cases:",
    int(results_df["confirmed_lab_vulnerability"].sum())
)
print(
    "Total test reward:",
    int(results_df["reward"].sum())
)

In [ ]:
import json
import torch
from pathlib import Path
from collections import Counter

print("=== GPU Environment ===")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"\nGPU {i}:", torch.cuda.get_device_name(i))
        props = torch.cuda.get_device_properties(i)
        print("VRAM:", round(props.total_memory / 1024**3, 2), "GB")

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

print("\n=== Dataset Files ===")
for name in [
    "dataset_1_train",
    "dataset_2_train",
    "dataset_3_train",
    "dataset_4_train"
]:
    path = BASE / name / f"{name}.json"
    print(name, "| exists:", path.exists(),
          "| size MB:", round(path.stat().st_size / 1024**2, 2)
          if path.exists() else "N/A")

print("\n=== Dataset 1 Label and Field Inspection ===")
file_path = BASE / "dataset_1_train" / "dataset_1_train.json"

with file_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

tags = Counter()
request_fields = Counter()
response_fields = Counter()

for record in records:
    req = record.get("request") or {}
    res = record.get("response") or {}

    tag = str(req.get("Attack_Tag") or "").strip()
    tags[tag if tag else "Benign (missing Attack_Tag)"] += 1

    request_fields.update(req.keys())
    response_fields.update(res.keys())

print("Records:", len(records))
print("Tag distribution:", dict(tags))
print("Request fields:", dict(request_fields))
print("Response fields:", dict(response_fields))

print("\nSample request:")
print(json.dumps(records[0].get("request", {}), indent=2)[:2000])

del records
print("\nInspection complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")
INPUT_FILE = BASE / "dataset_1_train" / "dataset_1_train.json"
OUTPUT_DIR = Path("/kaggle/working/api_pentest_research/finetuning")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / "dataset1_sft.jsonl"

ATTACK_LABELS = {
    "Directory Traversal",
    "Cookie Injection",
    "LOG4J",
    "RCE",
    "Log Forging",
    "SQL Injection",
    "XSS"
}

DROP_HEADERS = {
    "cookie",
    "set-cookie",
    "date",
    "authorization",
    "proxy-authorization"
}

def clean_headers(headers):
    cleaned = {}
    for key, value in (headers or {}).items():
        if key.lower() not in DROP_HEADERS:
            cleaned[key] = value
    return cleaned

with INPUT_FILE.open("r", encoding="utf-8") as f:
    records = json.load(f)

counts = Counter()
examples = []

with OUTPUT_FILE.open("w", encoding="utf-8") as out:
    for record in records:
        req = record.get("request") or {}
        raw_tag = str(req.get("Attack_Tag") or "").strip()

        if not raw_tag:
            category = "Benign"
        elif raw_tag in ATTACK_LABELS:
            category = raw_tag
        else:
            category = "Other"

        binary_label = "Benign" if category == "Benign" else "Malware"

        request_data = {
            "method": str(req.get("method", "")).upper(),
            "url": str(req.get("url", "")),
            "headers": clean_headers(req.get("headers")),
            "body": str(req.get("body", ""))
        }

        example = {
            "input": (
                "Classify this HTTP request using the supplied dataset "
                "categories. Do not claim that an attack succeeded.\n"
                + json.dumps(request_data, sort_keys=True, ensure_ascii=False)
            ),
            "category": category,
            "binary_label": binary_label
        }

        out.write(json.dumps(example, ensure_ascii=False) + "\n")
        counts[category] += 1

        if len(examples) < 3:
            examples.append(example)

print("Output file:", OUTPUT_FILE)
print("Total examples:", sum(counts.values()))
print("Category counts:", dict(counts))
print("Binary counts:", {
    "Benign": counts["Benign"],
    "Malware": sum(v for k, v in counts.items() if k != "Benign")
})
print("\nSample formatted examples:")
for example in examples:
    print(json.dumps(example, indent=2, ensure_ascii=False)[:1800])

print("\nFile size MB:", round(OUTPUT_FILE.stat().st_size / 1024**2, 3))

In [ ]:
import json
from pathlib import Path
from collections import Counter

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for name in [
    "dataset_1_train",
    "dataset_2_train",
    "dataset_3_train",
    "dataset_4_train"
]:
    file_path = BASE / name / f"{name}.json"

    with file_path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    categories = Counter()
    malformed_urls = []
    missing_method = 0
    missing_url = 0
    missing_body = 0

    for record in records:
        req = record.get("request") or {}
        tag = str(req.get("Attack_Tag") or "").strip()
        categories[tag if tag else "Benign"] += 1

        url = str(req.get("url") or "")
        if url.startswith("[http") or "](" in url:
            if len(malformed_urls) < 5:
                malformed_urls.append(url[:250])

        if not req.get("method"):
            missing_method += 1
        if not url:
            missing_url += 1
        if "body" not in req:
            missing_body += 1

    print(f"\n=== {name} ===")
    print("Records:", len(records))
    print("Categories:", dict(categories))
    print("URL formatting anomalies:", sum(
        1 for r in records
        if str((r.get("request") or {}).get("url") or "").startswith("[http")
        or "](" in str((r.get("request") or {}).get("url") or "")
    ))
    print("Missing method:", missing_method)
    print("Missing URL:", missing_url)
    print("Missing body field:", missing_body)

    if malformed_urls:
        print("Example URL anomalies:")
        for url in malformed_urls:
            print(repr(url))

    del records

print("\nAudit complete. No files were modified.")

In [ ]:
import json
import hashlib
import random
from pathlib import Path
from collections import Counter, defaultdict

BASE = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")
OUT = Path("/kaggle/working/api_pentest_research/finetuning")
OUT.mkdir(parents=True, exist_ok=True)

DATASETS = [
    "dataset_1_train",
    "dataset_2_train",
    "dataset_3_train",
    "dataset_4_train"
]

SEED = 42
VALIDATION_FRACTION = 0.15
DROP_HEADERS = {
    "cookie", "set-cookie", "date",
    "authorization", "proxy-authorization"
}

def clean_headers(headers):
    return {
        k: v for k, v in (headers or {}).items()
        if k.lower() not in DROP_HEADERS
    }

def request_signature(req):
    normalized = {
        "method": str(req.get("method", "")).upper(),
        "url": str(req.get("url", "")),
        "body": str(req.get("body", "")),
        "headers": clean_headers(req.get("headers"))
    }
    raw = json.dumps(
        normalized, sort_keys=True, ensure_ascii=False
    )
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

rows = []
group_to_rows = defaultdict(list)
source_counts = Counter()
category_counts = Counter()

for source in DATASETS:
    path = BASE / source / f"{source}.json"

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        req = record.get("request") or {}
        tag = str(req.get("Attack_Tag") or "").strip()
        category = tag if tag else "Benign"

        request_data = {
            "method": str(req.get("method", "")).upper(),
            "url": str(req.get("url", "")),
            "headers": clean_headers(req.get("headers")),
            "body": str(req.get("body", ""))
        }

        prompt = (
            "Classify the HTTP request into one of these dataset-derived "
            "categories: Benign, Cookie Injection, Directory Traversal, "
            "LOG4J, Log Forging, RCE, SQL Injection, XSS. "
            "Return only the category. A category label does not prove "
            "that an attack succeeded.\n"
            + json.dumps(request_data, sort_keys=True, ensure_ascii=False)
        )

        row = {
            "input": prompt,
            "category": category,
            "binary_label": "Benign" if category == "Benign" else "Malware",
            "source_dataset": source,
            "group": request_signature(req)
        }

        rows.append(row)
        group_to_rows[row["group"]].append(row)
        source_counts[source] += 1
        category_counts[category] += 1

    del records

rng = random.Random(SEED)
group_ids = list(group_to_rows.keys())
rng.shuffle(group_ids)

target_val_rows = int(len(rows) * VALIDATION_FRACTION)
val_groups = set()
val_count = 0

for group_id in group_ids:
    if val_count >= target_val_rows:
        break
    val_groups.add(group_id)
    val_count += len(group_to_rows[group_id])

train_rows = [
    row for row in rows if row["group"] not in val_groups
]
val_rows = [
    row for row in rows if row["group"] in val_groups
]

train_groups = {row["group"] for row in train_rows}
validation_groups = {row["group"] for row in val_rows}
overlap = train_groups & validation_groups

def write_jsonl(path, examples):
    with path.open("w", encoding="utf-8") as f:
        for example in examples:
            cleaned = {k: v for k, v in example.items() if k != "group"}
            f.write(json.dumps(cleaned, ensure_ascii=False) + "\n")

combined_path = OUT / "combined_sft.jsonl"
train_path = OUT / "train_sft.jsonl"
val_path = OUT / "validation_sft.jsonl"

write_jsonl(combined_path, rows)
write_jsonl(train_path, train_rows)
write_jsonl(val_path, val_rows)

print("Total examples:", len(rows))
print("Unique request-signature groups:", len(group_ids))
print("Source counts:", dict(source_counts))
print("Category counts:", dict(category_counts))
print("Train examples:", len(train_rows))
print("Validation examples:", len(val_rows))
print("Validation fraction:", round(len(val_rows) / len(rows), 4))
print("Train/validation group overlap:", len(overlap))
print("Train categories:", dict(Counter(r["category"] for r in train_rows)))
print("Validation categories:", dict(Counter(r["category"] for r in val_rows)))
print("\nFiles created:")
print(combined_path)
print(train_path)
print(val_path)

In [ ]:
import importlib
import torch

print("=== GPU ===")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

print("\n=== Fine-tuning packages ===")
packages = [
    "transformers",
    "datasets",
    "peft",
    "trl",
    "accelerate",
    "bitsandbytes",
    "sklearn"
]

for package in packages:
    try:
        module = importlib.import_module(package)
        version = getattr(module, "__version__", "version unavailable")
        print(f"{package}: installed ({version})")
    except Exception as e:
        print(f"{package}: unavailable ({type(e).__name__}: {e})")

print("\n=== Prepared dataset files ===")
from pathlib import Path

data_dir = Path(
    "/kaggle/working/api_pentest_research/finetuning"
)

for name in [
    "train_sft.jsonl",
    "validation_sft.jsonl",
    "combined_sft.jsonl"
]:
    p = data_dir / name
    print(name, "| exists:", p.exists(),
          "| MB:", round(p.stat().st_size / 1024**2, 2)
          if p.exists() else "N/A")

In [ ]:
import sys
import torch
import transformers
import datasets
import peft
import accelerate
import trl
import bitsandbytes as bnb

print("=== Package versions ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("TRL:", trl.__version__)
print("Bitsandbytes:", bnb.__version__)

print("\n=== GPU status ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

print("\n=== QLoRA support check ===")
try:
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig
    from trl import SFTTrainer, SFTConfig

    print("BitsAndBytesConfig: OK")
    print("LoraConfig: OK")
    print("SFTTrainer: OK")
    print("SFTConfig: OK")
except Exception as e:
    print(type(e).__name__ + ":", str(e))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Vocabulary size:", len(tokenizer))
print("EOS token:", tokenizer.eos_token)
print("PAD token:", tokenizer.pad_token)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("\nLoading 4-bit model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map={"": 0},
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("\nModel loaded successfully.")
print("Model:", MODEL_NAME)
print("Device:", model.get_input_embeddings().weight.device)
print("Parameters (billions):", round(
    sum(p.numel() for p in model.parameters()) / 1e9, 3
))

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i} memory allocated:",
        round(torch.cuda.memory_allocated(i) / 1024**3, 2),
        "GB"
    )

In [ ]:
import torch
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

print("Preparing 4-bit model for LoRA training...")

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total = sum(p.numel() for p in model.parameters())

print("\nTrainable parameters:", f"{trainable:,}")
print("Total parameters:", f"{total:,}")
print("Trainable percentage:", round(100 * trainable / total, 4), "%")

print("\nGPU memory:")
for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}:",
        round(torch.cuda.memory_allocated(i) / 1024**3, 2),
        "GB allocated"
    )

In [ ]:
import json
import random
from pathlib import Path
from collections import Counter, defaultdict

DATA_DIR = Path(
    "/kaggle/working/api_pentest_research/finetuning"
)
TRAIN_FILE = DATA_DIR / "train_sft.jsonl"
VAL_FILE = DATA_DIR / "validation_sft.jsonl"

SEED = 42
random.seed(SEED)

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_FILE)
val_rows = read_jsonl(VAL_FILE)

def balanced_sample(rows, per_category, seed):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row["category"]].append(row)

    rng = random.Random(seed)
    sampled = []

    for category, examples in sorted(grouped.items()):
        rng.shuffle(examples)
        sampled.extend(examples[:per_category])

    rng.shuffle(sampled)
    return sampled

pilot_train = balanced_sample(train_rows, 100, SEED)
pilot_val = balanced_sample(val_rows, 30, SEED + 1)

PILOT_TRAIN_FILE = DATA_DIR / "pilot_train.jsonl"
PILOT_VAL_FILE = DATA_DIR / "pilot_validation.jsonl"

for path, rows in [
    (PILOT_TRAIN_FILE, pilot_train),
    (PILOT_VAL_FILE, pilot_val)
]:
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Pilot training examples:", len(pilot_train))
print("Training category counts:", dict(Counter(
    row["category"] for row in pilot_train
)))
print("\nPilot validation examples:", len(pilot_val))
print("Validation category counts:", dict(Counter(
    row["category"] for row in pilot_val
)))

print("\n=== Token length inspection ===")
for row in pilot_train[:5]:
    messages = [
        {
            "role": "user",
            "content": row["input"]
        },
        {
            "role": "assistant",
            "content": row["category"]
        }
    ]
    token_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )
    print(
        "Category:", row["category"],
        "| tokens:", len(token_ids)
    )

print("\nPilot files:")
print(PILOT_TRAIN_FILE)
print(PILOT_VAL_FILE)

In [ ]:
import json
from pathlib import Path
from collections import Counter

from datasets import load_dataset

DATA_DIR = Path(
    "/kaggle/working/api_pentest_research/finetuning"
)

TRAIN_FILE = str(DATA_DIR / "train_sft.jsonl")
VAL_FILE = str(DATA_DIR / "validation_sft.jsonl")

print("Loading full training dataset...")
full_dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE
    }
)

train_ds = full_dataset["train"]
val_ds = full_dataset["validation"]

print("Training examples:", len(train_ds))
print("Validation examples:", len(val_ds))
print("Training columns:", train_ds.column_names)

def format_example(example):
    messages = [
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["category"]}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

print("\nMeasuring full formatted sequence lengths...")
sample_indices = list(range(min(1000, len(train_ds))))
lengths = []

for i in sample_indices:
    text = format_example(train_ds[i])
    lengths.append(len(tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]))

lengths_sorted = sorted(lengths)

def percentile(values, p):
    return values[min(len(values) - 1, int((len(values) - 1) * p))]

print("Measured examples:", len(lengths))
print("Median tokens:", percentile(lengths_sorted, 0.50))
print("90th percentile:", percentile(lengths_sorted, 0.90))
print("95th percentile:", percentile(lengths_sorted, 0.95))
print("99th percentile:", percentile(lengths_sorted, 0.99))
print("Maximum in sample:", max(lengths))

print("\nCategory distribution in full training data:")
print(dict(Counter(train_ds["category"])))

print("\nFull datasets are ready. No training has started.")

In [ ]:
import json
import hashlib
import random
from pathlib import Path
from collections import defaultdict, Counter

DATA_DIR = Path(
    "/kaggle/working/api_pentest_research/finetuning"
)

TRAIN_FILE = DATA_DIR / "train_sft.jsonl"
VAL_FILE = DATA_DIR / "validation_sft.jsonl"

SEED = 2026
TEST_FRACTION = 0.10
DROP_HEADERS = {
    "cookie", "set-cookie", "date",
    "authorization", "proxy-authorization"
}

def signature_from_example(row):
    text = row["input"].split("\n", 1)[-1]
    request = json.loads(text)

    headers = {
        k: v for k, v in (request.get("headers") or {}).items()
        if k.lower() not in DROP_HEADERS
    }

    normalized = {
        "method": str(request.get("method", "")).upper(),
        "url": str(request.get("url", "")),
        "body": str(request.get("body", "")),
        "headers": headers
    }

    raw = json.dumps(normalized, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

def read_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train_rows = read_jsonl(TRAIN_FILE)
val_rows = read_jsonl(VAL_FILE)

grouped = defaultdict(list)
for row in train_rows:
    grouped[signature_from_example(row)].append(row)

group_ids = list(grouped)
random.Random(SEED).shuffle(group_ids)

target_test_count = int(len(train_rows) * TEST_FRACTION)
test_groups = set()
test_count = 0

for group_id in group_ids:
    if test_count >= target_test_count:
        break
    test_groups.add(group_id)
    test_count += len(grouped[group_id])

new_train = [
    row for row in train_rows
    if signature_from_example(row) not in test_groups
]
test_rows = [
    row for row in train_rows
    if signature_from_example(row) in test_groups
]

def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

NEW_TRAIN_FILE = DATA_DIR / "train_final.jsonl"
TEST_FILE = DATA_DIR / "test_holdout.jsonl"

write_jsonl(NEW_TRAIN_FILE, new_train)
write_jsonl(TEST_FILE, test_rows)

new_train_groups = {signature_from_example(r) for r in new_train}
test_signature_groups = {signature_from_example(r) for r in test_rows}
val_signature_groups = {signature_from_example(r) for r in val_rows}

print("New training examples:", len(new_train))
print("Validation examples (unchanged):", len(val_rows))
print("Held-out test examples:", len(test_rows))
print("Train/test exact-signature overlap:",
      len(new_train_groups & test_signature_groups))
print("Train/validation exact-signature overlap:",
      len(new_train_groups & val_signature_groups))
print("Test/validation exact-signature overlap:",
      len(test_signature_groups & val_signature_groups))
print("Training categories:", dict(Counter(r["category"] for r in new_train)))
print("Test categories:", dict(Counter(r["category"] for r in test_rows)))
print("\nFiles created:")
print(NEW_TRAIN_FILE)
print(VAL_FILE)
print(TEST_FILE)

In [ ]:
import inspect
import torch
from pathlib import Path
from trl import SFTTrainer, SFTConfig

DATA_DIR = Path(
    "/kaggle/working/api_pentest_research/finetuning"
)

TRAIN_FILE = DATA_DIR / "train_final.jsonl"
VAL_FILE = DATA_DIR / "validation_sft.jsonl"
OUTPUT_DIR = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora"
)

assert TRAIN_FILE.exists(), f"Missing training file: {TRAIN_FILE}"
assert VAL_FILE.exists(), f"Missing validation file: {VAL_FILE}"
assert torch.cuda.is_available(), "CUDA is unavailable"
assert "model" in globals(), "Qwen model is not loaded"
assert "tokenizer" in globals(), "Tokenizer is not loaded"

print("=== Trainer API compatibility ===")
print("TRL SFTTrainer:", inspect.signature(SFTTrainer))
print("TRL SFTConfig:", inspect.signature(SFTConfig))

print("\n=== Training inputs ===")
print("Model:", "Qwen/Qwen2.5-1.5B-Instruct")
print("Training file:", TRAIN_FILE)
print("Validation file:", VAL_FILE)
print("Output directory:", OUTPUT_DIR)
print("GPU:", torch.cuda.get_device_name(0))

training_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    gradient_checkpointing=True,
    max_length=512,
    packing=False,
    report_to="none",
    seed=42,
    data_seed=42
)

print("\n=== Proposed run ===")
print("Epochs:", training_config.num_train_epochs)
print("Per-device batch size:", training_config.per_device_train_batch_size)
print("Gradient accumulation:", training_config.gradient_accumulation_steps)
print("Maximum sequence length:", training_config.max_length)
print("Evaluation strategy:", training_config.eval_strategy)
print("Save strategy:", training_config.save_strategy)
print("\nConfiguration created. Training has not started.")

In [ ]:
from trl import SFTConfig
from pathlib import Path

OUTPUT_DIR = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora"
)

training_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    fp16=True,
    gradient_checkpointing=True,
    max_length=512,
    packing=False,
    completion_only_loss=True,
    optim="adamw_torch",
    report_to="none",
    seed=42,
    data_seed=42
)

print("Configuration updated.")
print("Epochs:", training_config.num_train_epochs)
print("Completion-only loss:", training_config.completion_only_loss)
print("Batch size per device:", training_config.per_device_train_batch_size)
print("Gradient accumulation:", training_config.gradient_accumulation_steps)
print("Maximum sequence length:", training_config.max_length)
print("Optimizer:", training_config.optim)
print("Training has not started.")

In [ ]:
import torch

names = [
    "tokenizer",
    "model",
    "training_config",
    "train_formatted",
    "val_formatted",
    "trainer",
    "train_ds",
    "val_ds"
]

for name in names:
    exists = name in globals()
    print(f"{name}: {'Available' if exists else 'Missing'}")

print("\nVisible GPUs:", torch.cuda.device_count())
print("GPU 0:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA unavailable")

if "model" in globals():
    print("Model device:", next(model.parameters()).device)

In [ ]:
from trl import SFTTrainer
import torch

training_config._n_gpu = 1

assert training_config.n_gpu == 1
assert next(model.parameters()).device.type == "cuda"

trainer = SFTTrainer(
    model=model,
    args=training_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    processing_class=tokenizer,
    peft_config=None
)

print("Trainer initialized successfully.")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Trainer GPU count:", training_config.n_gpu)
print("Training has not started.")

In [ ]:
import torch

batch = next(iter(trainer.get_train_dataloader()))

for i in range(min(2, batch["input_ids"].shape[0])):
    labels = batch["labels"][i]
    supervised = labels[labels != -100]

    print(f"\nExample {i + 1}")
    print("Total tokens:", len(labels))
    print("Tokens contributing to loss:", len(supervised))
    print(
        "Supervised text:",
        repr(tokenizer.decode(supervised, skip_special_tokens=False)[:500])
    )

print("\nTraining has not started.")

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class ExplicitCompletionCollator:
    tokenizer: object

    def __call__(self, features):
        rows = []

        for feature in features:
            text = feature["text"]
            marker = "<|im_start|>assistant\n"

            if marker not in text:
                raise ValueError("Assistant response boundary not found.")

            prompt_text, completion_text = text.rsplit(marker, 1)
            prompt_text += marker

            prompt_ids = self.tokenizer(
                prompt_text,
                add_special_tokens=False
            )["input_ids"]

            completion_ids = self.tokenizer(
                completion_text,
                add_special_tokens=False
            )["input_ids"]

            input_ids = prompt_ids + completion_ids
            labels = [-100] * len(prompt_ids) + completion_ids.copy()

            max_len = training_config.max_length
            if len(input_ids) > max_len:
                excess = len(input_ids) - max_len
                if excess >= len(prompt_ids):
                    prompt_ids = []
                    input_ids = input_ids[-max_len:]
                    labels = labels[-max_len:]
                else:
                    prompt_ids = prompt_ids[excess:]
                    input_ids = input_ids[excess:]
                    labels = labels[excess:]

            rows.append((input_ids, labels))

        max_len = max(len(ids) for ids, _ in rows)
        pad_id = self.tokenizer.pad_token_id

        batch_input_ids, batch_attention, batch_labels = [], [], []

        for ids, labels in rows:
            pad = max_len - len(ids)
            batch_input_ids.append(ids + [pad_id] * pad)
            batch_attention.append([1] * len(ids) + [0] * pad)
            batch_labels.append(labels + [-100] * pad)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }

correct_collator = ExplicitCompletionCollator(tokenizer)

test_batch = correct_collator([
    train_formatted[0],
    train_formatted[1]
])

for i in range(2):
    labels = test_batch["labels"][i]
    supervised = labels[labels != -100]
    print(f"\nExample {i + 1}")
    print("Total tokens:", len(labels))
    print("Supervised tokens:", len(supervised))
    print("Supervised text:",
          repr(tokenizer.decode(supervised, skip_special_tokens=False)))

    assert len(supervised) > 0
    assert len(supervised) < len(labels)

print("\nCollator test passed. Training has not started.")

In [ ]:
print("Trainer dataset columns:", trainer.train_dataset.column_names)
print("First example keys:", trainer.train_dataset[0].keys())

print("\nFirst example:")
for key, value in trainer.train_dataset[0].items():
    if isinstance(value, list):
        print(key, "length:", len(value), "sample:", value[:8])
    else:
        print(key, repr(value)[:200])

print("\nTraining has not started.")

In [ ]:
from torch.utils.data import DataLoader

test_examples = [train_formatted[0], train_formatted[1]]

print("Original formatted keys:", test_examples[0].keys())

test_loader = DataLoader(
    test_examples,
    batch_size=2,
    collate_fn=correct_collator
)

batch = next(iter(test_loader))

for i in range(batch["input_ids"].shape[0]):
    active = int(batch["attention_mask"][i].sum())
    labels = batch["labels"][i]
    supervised = labels[labels != -100]

    print(f"\nExample {i + 1}")
    print("Active tokens:", active)
    print("Supervised tokens:", len(supervised))
    print(
        "Supervised text:",
        repr(tokenizer.decode(supervised, skip_special_tokens=False))
    )

    assert 0 < len(supervised) < active

print("\nCustom collator independently verified.")
print("Training has not started.")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    data_collator=correct_collator,
    processing_class=tokenizer
)

print("Trainer rebuilt with explicit completion masking.")
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Training has not started.")

In [ ]:
from transformers import Trainer

training_config.remove_unused_columns = False

trainer = Trainer(
    model=model,
    args=training_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    data_collator=correct_collator,
    processing_class=tokenizer
)

print("Trainer rebuilt successfully.")
print("Automatic column removal:", training_config.remove_unused_columns)
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Training has not started.")

In [ ]:
import torch

batch = next(iter(trainer.get_train_dataloader()))

for i in range(batch["input_ids"].shape[0]):
    active = int(batch["attention_mask"][i].sum())
    labels = batch["labels"][i]
    supervised = labels[labels != -100]

    print(f"\nExample {i + 1}")
    print("Active tokens:", active)
    print("Supervised tokens:", len(supervised))
    print(
        "Supervised text:",
        repr(tokenizer.decode(supervised, skip_special_tokens=False))
    )

    assert 0 < len(supervised) < active

print("\nGPU:", torch.cuda.get_device_name(0))
print("Training examples:", len(trainer.train_dataset))
print("Validation examples:", len(trainer.eval_dataset))
print("Real batch verified. Training has not started.")

In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("FP16 enabled:", training_config.fp16)
print("BF16 enabled:", training_config.bf16)

print("\nTrainable parameter dtypes:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

print("\nExisting gradients:")
found = False
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, param.grad.dtype)
        found = True
        break

if not found:
    print("No gradients currently retained.")

print("\nTraining status: failed before completing an optimizer step.")

In [ ]:
import torch

print("Before:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float16)
        param.grad = None

training_config.fp16 = True
training_config.bf16 = False

print("\nAfter:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.dtype)
        break

print("FP16:", training_config.fp16)
print("BF16:", training_config.bf16)

assert all(
    p.dtype == torch.float16
    for p in model.parameters()
    if p.requires_grad
)

print("\nPrecision check passed. Training has not restarted.")

In [ ]:
import torch
from transformers import Trainer

for param in model.parameters():
    param.grad = None

training_config.fp16 = True
training_config.bf16 = False
training_config.remove_unused_columns = False

trainer = Trainer(
    model=model,
    args=training_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    data_collator=correct_collator,
    processing_class=tokenizer
)

print("Trainer rebuilt.")
print("FP16:", training_config.fp16)
print("BF16:", training_config.bf16)
print("Trainable dtype:", next(
    p.dtype for p in model.parameters() if p.requires_grad
))
print("Training examples:", len(trainer.train_dataset))
print("Training has not started.")

In [ ]:
import torch

model.train()
batch = next(iter(trainer.get_train_dataloader()))

device = next(model.parameters()).device
batch = {k: v.to(device) for k, v in batch.items()}

with torch.autocast("cuda", dtype=torch.float16):
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )
    loss = outputs.loss

print("Loss:", loss.item())
print("Loss dtype:", loss.dtype)

loss.backward()

grad_dtypes = {}
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is not None:
        grad_dtypes[str(param.grad.dtype)] = (
            grad_dtypes.get(str(param.grad.dtype), 0) + 1
        )

print("Gradient dtypes:", grad_dtypes)

for param in model.parameters():
    if param.grad is not None:
        param.grad = None

assert torch.isfinite(loss).item(), "Loss is not finite."
print("Forward/backward test passed.")
print("No optimizer step performed; training has not started.")

In [ ]:
import torch
from transformers import Trainer

for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.float()
        param.grad = None

training_config.fp16 = False
training_config.bf16 = False
training_config.remove_unused_columns = False

trainer = Trainer(
    model=model,
    args=training_config,
    train_dataset=train_formatted,
    eval_dataset=val_formatted,
    data_collator=correct_collator,
    processing_class=tokenizer
)

trainable_dtypes = {
    str(param.dtype)
    for param in model.parameters()
    if param.requires_grad
}

print("Trainable parameter dtypes:", trainable_dtypes)
print("FP16 enabled:", training_config.fp16)
print("BF16 enabled:", training_config.bf16)
print("Trainer rebuilt successfully.")

In [ ]:
import torch

model.train()
trainer.create_optimizer()

batch = next(iter(trainer.get_train_dataloader()))
device = next(model.parameters()).device
batch = {k: v.to(device) for k, v in batch.items()}

trainer.optimizer.zero_grad(set_to_none=True)

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"]
)

loss = outputs.loss
print("Loss:", loss.item())

loss.backward()

trainable_params = [
    p for p in model.parameters()
    if p.requires_grad
]

grad_norm = torch.nn.utils.clip_grad_norm_(
    trainable_params,
    training_config.max_grad_norm
)

before = trainable_params[0].detach().clone()
trainer.optimizer.step()
after = trainable_params[0].detach()

changed = not torch.equal(before, after)

trainer.optimizer.zero_grad(set_to_none=True)

print("Gradient norm:", float(grad_norm))
print("Parameter updated:", changed)
print("Optimizer step test passed:", torch.isfinite(loss).item())

In [ ]:
import torch

trainable_params = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

name, param = trainable_params[0]
print("Parameter:", name)
print("Parameter dtype:", param.dtype)
print("Gradient is None:", param.grad is None)

if param.grad is not None:
    print("Gradient absolute maximum:", param.grad.abs().max().item())
    print("Nonzero gradient elements:", torch.count_nonzero(param.grad).item())

optimizer_state = trainer.optimizer.state.get(param, {})
print("Optimizer state keys:", list(optimizer_state.keys()))

for key, value in optimizer_state.items():
    if torch.is_tensor(value):
        print(key, "shape:", tuple(value.shape), "dtype:", value.dtype)

print("Optimizer type:", type(trainer.optimizer).__name__)
print("Learning rate:", trainer.optimizer.param_groups[0]["lr"])

print("\nChecking gradients across all trainable parameters:")
total_with_grad = 0
total_nonzero = 0

for _, p in trainable_params:
    if p.grad is not None:
        total_with_grad += 1
        if torch.count_nonzero(p.grad).item() > 0:
            total_nonzero += 1

print("Parameters with gradients:", total_with_grad)
print("Parameters with nonzero gradients:", total_nonzero)

In [ ]:
import torch

model.train()
trainer.optimizer.zero_grad(set_to_none=True)

batch = next(iter(trainer.get_train_dataloader()))
device = next(model.parameters()).device
batch = {k: v.to(device) for k, v in batch.items()}

outputs = model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    labels=batch["labels"]
)

loss = outputs.loss
print("Loss:", loss.item())

loss.backward()

trainable = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

with_grad = [(name, p) for name, p in trainable if p.grad is not None]
finite_grads = [
    (name, p) for name, p in with_grad
    if torch.isfinite(p.grad).all().item()
]
nonzero_grads = [
    (name, p) for name, p in finite_grads
    if torch.count_nonzero(p.grad).item() > 0
]

print("Trainable parameters:", len(trainable))
print("Parameters with gradients:", len(with_grad))
print("Parameters with finite gradients:", len(finite_grads))
print("Parameters with nonzero gradients:", len(nonzero_grads))

assert torch.isfinite(loss).item(), "Loss is not finite."
assert nonzero_grads, "No nonzero gradients found."

before = {
    name: p.detach().clone()
    for name, p in trainable
}

torch.nn.utils.clip_grad_norm_(
    [p for _, p in trainable],
    training_config.max_grad_norm
)

trainer.optimizer.step()

changed = [
    name for name, p in trainable
    if not torch.equal(before[name], p.detach())
]

print("Parameters changed:", len(changed), "/", len(trainable))
print("Example changed parameter:", changed[0] if changed else "None")

trainer.optimizer.zero_grad(set_to_none=True)

assert changed, "No trainable parameter changed after optimizer.step()."
print("Gradient and optimizer update verification passed.")

In [ ]:
import os
from pathlib import Path

output_dir = Path("/kaggle/working/api_pentest_research/qwen_api_security_lora")

if not output_dir.exists():
    print("Training output directory not found.")
else:
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1])
    )

    print("Checkpoint directory:", output_dir)
    print("Checkpoints found:", len(checkpoints))

    for checkpoint in checkpoints:
        print("\n", checkpoint.name)
        for item in sorted(checkpoint.iterdir()):
            print(" ", item.name, f"({item.stat().st_size / (1024**2):.1f} MB)")
            

In [ ]:
from pathlib import Path
import json

checkpoint = Path("/kaggle/working/api_pentest_research/qwen_api_security_lora/checkpoint-1000")
state_file = checkpoint / "trainer_state.json"

if state_file.exists():
    with open(state_file, "r") as f:
        state = json.load(f)
    print("Checkpoint state loaded:", state.get("global_step"))
else:
    state = None
    print("No previous checkpoint found. Starting fresh.")

In [ ]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

subprocess.run(["nvidia-smi"])

In [ ]:
import torch
import torch.distributed as dist
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

print("Distributed available:", dist.is_available())
print("NCCL available:", dist.is_nccl_available())
print("CUDA device count:", torch.cuda.device_count())

if dist.is_available() and dist.is_nccl_available():
    print("NCCL backend is available.")
else:
    print("Distributed GPU backend needs further checking.")

print("Current checkpoint:", trainer.state.global_step)
print("Training examples:", len(train_formatted))
print("Validation examples:", len(val_formatted))

In [ ]:
from pathlib import Path
import json

output_dir = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora"
)

checkpoints = sorted(
    [
        p for p in output_dir.glob("checkpoint-*")
        if p.is_dir() and p.name.split("-")[-1].isdigit()
    ],
    key=lambda p: int(p.name.split("-")[-1])
)

print("Available checkpoints:")

for cp in checkpoints:
    state_file = cp / "trainer_state.json"
    adapter_file = cp / "adapter_model.safetensors"
    optimizer_file = cp / "optimizer.pt"
    scheduler_file = cp / "scheduler.pt"

    step = None
    if state_file.exists():
        with open(state_file, "r") as f:
            step = json.load(f).get("global_step")

    complete = all(
        p.exists() and p.stat().st_size > 0
        for p in [
            state_file,
            adapter_file,
            optimizer_file,
            scheduler_file
        ]
    )

    print(
        cp.name,
        "| saved step:", step,
        "| required files present:", complete
    )

if checkpoints:
    latest = checkpoints[-1]
    print("\nLatest checkpoint candidate:", latest)
else:
    print("\nNo checkpoint directories found.")

In [ ]:
import accelerate
from accelerate.commands.launch import launch_command_parser

print("Accelerate version:", accelerate.__version__)

parser = launch_command_parser()
print("\nRelevant distributed launch options:")

for action in parser._actions:
    option_names = [
        option for option in action.option_strings
        if any(key in option for key in [
            "multi_gpu", "num_processes", "mixed_precision",
            "main_process_port", "use_cpu"
        ])
    ]

    if option_names:
        print(", ".join(option_names), "| default:", action.default)

In [ ]:
print("\n=== CURRENT LORA CONFIG ===")

peft_config = getattr(model, "peft_config", {})

if peft_config:
    for adapter_name, lora_config in peft_config.items():
        print("Adapter:", adapter_name)

        for key in [
            "base_model_name_or_path",
            "r",
            "lora_alpha",
            "lora_dropout",
            "target_modules",
            "task_type",
        ]:
            print(f"{key}: {getattr(lora_config, key, None)}")
else:
    print("No PEFT adapter configuration found.")

print("\nCheckpoint:", checkpoint)
print("Checkpoint exists:", checkpoint.is_dir())

In [ ]:
import torch

batch = correct_collator([
    train_formatted[0],
    train_formatted[1]
])

print("Batch keys:", list(batch.keys()))
print("Input shape:", tuple(batch["input_ids"].shape))
print("Label shape:", tuple(batch["labels"].shape))
print("Attention-mask shape:", tuple(batch["attention_mask"].shape))

for i in range(len(batch["input_ids"])):
    labels = batch["labels"][i]
    valid_labels = labels[labels != -100]

    print(f"\nExample {i + 1}")
    print("Total tokens:", int(batch["attention_mask"][i].sum()))
    print("Supervised tokens:", len(valid_labels))
    print(
        "Supervised text:",
        tokenizer.decode(valid_labels.tolist())
    )

assert batch["input_ids"].shape == batch["labels"].shape
assert batch["input_ids"].shape == batch["attention_mask"].shape
assert torch.all(batch["labels"][batch["attention_mask"] == 0] == -100)

print("\nCollator checks passed.")
print("Tokenizer padding token ID:", tokenizer.pad_token_id)
print("Tokenizer EOS token ID:", tokenizer.eos_token_id)
print("Maximum sequence length:", training_config.max_length)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
train_file = root / "finetuning" / "train_final.jsonl"
validation_file = root / "finetuning" / "validation_sft.jsonl"
checkpoint = root / "qwen_api_security_lora" / "checkpoint-1000"

checks = {
    "Training JSONL": train_file,
    "Validation JSONL": validation_file,
    "Resume checkpoint": checkpoint,
    "LoRA adapter": checkpoint / "adapter_model.safetensors",
    "LoRA config": checkpoint / "adapter_config.json",
}

for label, path in checks.items():
    if path.is_file():
        size = f"{path.stat().st_size / (1024**3):.2f} GiB"
    elif path.is_dir():
        size = "directory"
    else:
        size = "MISSING"

    print(f"{label}: {path} | {size}")

print("\nTraining rows currently loaded:", len(train_formatted))
print("Validation rows currently loaded:", len(val_formatted))

In [ ]:
from pathlib import Path

script_path = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed.py"
)

script = r'''
import os
import json
import torch
import torch.distributed as dist
from pathlib import Path
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ROOT = Path("/kaggle/working/api_pentest_research")
DATA_DIR = ROOT / "finetuning"
CHECKPOINT = ROOT / "qwen_api_security_lora" / "checkpoint-1000"
OUTPUT_DIR = ROOT / "qwen_api_security_lora_distributed"
MAX_LENGTH = 512

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

train_path = DATA_DIR / "train_final.jsonl"
val_path = DATA_DIR / "validation_sft.jsonl"

def read_text_dataset(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            rows.append({"text": item["input"] + item["output"]})
    return Dataset.from_list(rows)

class ExplicitCompletionCollator:
    def __init__(self, tokenizer, max_length=512):
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __call__(self, features):
        rows = []
        marker = "<|im_start|>assistant\n"

        for feature in features:
            text = feature["text"]
            if marker not in text:
                raise ValueError("Assistant completion marker missing.")

            prompt_text, completion_text = text.rsplit(marker, 1)
            prompt_text += marker

            prompt_ids = self.tokenizer(
                prompt_text, add_special_tokens=False
            )["input_ids"]
            completion_ids = self.tokenizer(
                completion_text, add_special_tokens=False
            )["input_ids"]

            input_ids = prompt_ids + completion_ids
            labels = [-100] * len(prompt_ids) + completion_ids.copy()

            if len(input_ids) > self.max_length:
                excess = len(input_ids) - self.max_length
                if excess >= len(prompt_ids):
                    input_ids = input_ids[-self.max_length:]
                    labels = labels[-self.max_length:]
                else:
                    input_ids = input_ids[excess:]
                    labels = labels[excess:]

            rows.append((input_ids, labels))

        max_len = max(len(ids) for ids, _ in rows)
        pad_id = self.tokenizer.pad_token_id
        input_batch, label_batch, mask_batch = [], [], []

        for ids, labels in rows:
            padding = max_len - len(ids)
            input_batch.append(ids + [pad_id] * padding)
            label_batch.append(labels + [-100] * padding)
            mask_batch.append([1] * len(ids) + [0] * padding)

        return {
            "input_ids": torch.tensor(input_batch, dtype=torch.long),
            "labels": torch.tensor(label_batch, dtype=torch.long),
            "attention_mask": torch.tensor(mask_batch, dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": local_rank},
)
model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT,
    is_trainable=True,
)

train_dataset = read_text_dataset(train_path)
val_dataset = read_text_dataset(val_path)
collator = ExplicitCompletionCollator(tokenizer, MAX_LENGTH)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_grad_norm=1.0,
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    eval_strategy="no",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    optim="adamw_torch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
    processing_class=tokenizer,
)

if dist.is_initialized() and dist.get_rank() == 0:
    print("Distributed processes:", dist.get_world_size())
    print("Training examples:", len(train_dataset))
    print("Resume checkpoint:", CHECKPOINT)
    print("Output directory:", OUTPUT_DIR)

trainer.train()
'''

script_path.write_text(script, encoding="utf-8")
print("Script created:", script_path)
print("Bytes:", script_path.stat().st_size)
print("Training has NOT started.")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/finetuning")

for filename in ["train_final.jsonl", "validation_sft.jsonl"]:
    path = root / filename
    print(f"\n=== {filename} ===")

    with open(path, "r", encoding="utf-8") as f:
        record = json.loads(f.readline())

    print("Keys:", list(record.keys()))

    for key, value in record.items():
        if isinstance(value, str):
            print(f"{key}: {value[:300]!r}")
        else:
            print(f"{key}: {type(value).__name__} {value}")

print("\n=== Existing formatted training example ===")
print(repr(train_formatted[0]["text"][-180:]))

In [ ]:
from pathlib import Path

script_path = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed.py"
)

script = script_path.read_text(encoding="utf-8")

old = '''def read_text_dataset(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            rows.append({"text": item["input"] + item["output"]})
    return Dataset.from_list(rows)
'''

new = '''def read_text_dataset(path, tokenizer):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            messages = [
                {
                    "role": "system",
                    "content": (
                        "You are Qwen, created by Alibaba Cloud. "
                        "You are a helpful assistant."
                    ),
                },
                {"role": "user", "content": item["input"]},
                {"role": "assistant", "content": item["category"]},
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            rows.append({"text": text})
    return Dataset.from_list(rows)
'''

if old not in script:
    raise ValueError("Expected original dataset loader was not found; script unchanged.")

script = script.replace(old, new)
script = script.replace(
    "train_dataset = read_text_dataset(train_path)",
    "train_dataset = read_text_dataset(train_path, tokenizer)"
)
script = script.replace(
    "val_dataset = read_text_dataset(val_path)",
    "val_dataset = read_text_dataset(val_path, tokenizer)"
)

script_path.write_text(script, encoding="utf-8")

print("Dataset loader corrected.")
print("Script path:", script_path)
print("Script contains item['category']:", 'item["category"]' in script)
print("Script contains old item['output']:", 'item["output"]' in script)
print("Training has NOT started.")


In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/finetuning")
path = root / "train_final.jsonl"

with open(path, "r", encoding="utf-8") as f:
    record = json.loads(f.readline())

messages = [
    {
        "role": "system",
        "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
    },
    {"role": "user", "content": record["input"]},
    {"role": "assistant", "content": record["category"]}
]

reconstructed = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

existing = train_formatted[0]["text"]

print("Reconstructed text ending:")
print(repr(reconstructed[-180:]))

print("\nExisting text ending:")
print(repr(existing[-180:]))

print("\nReconstructed completion marker:", "<|im_start|>assistant\n" in reconstructed)
print("Reconstructed category present:", record["category"] in reconstructed)
print("Reconstructed length:", len(reconstructed))
print("Existing length:", len(existing))
print("Exact match:", reconstructed == existing)

In [ ]:
import torch
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora/checkpoint-1000"
)

optimizer_path = checkpoint / "optimizer.pt"
scheduler_path = checkpoint / "scheduler.pt"

print("Checkpoint exists:", checkpoint.is_dir())

if checkpoint.is_dir():
    for path in [optimizer_path, scheduler_path]:
        print(f"{path.name}: {path.exists()}")

    if optimizer_path.exists() and scheduler_path.exists():
        optimizer_state = torch.load(
            optimizer_path,
            map_location="cpu",
            weights_only=False
        )

        scheduler_state = torch.load(
            scheduler_path,
            map_location="cpu",
            weights_only=False
        )

        print("Optimizer state keys:", list(optimizer_state.keys()))
        print("Scheduler state:", scheduler_state)
    else:
        print("Optimizer or scheduler state is missing.")
        print("Cannot resume from this checkpoint.")
else:
    print("Previous checkpoint was deleted.")
    print("Fresh training requires no previous optimizer or scheduler state.")

In [ ]:
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora/checkpoint-1000"
)

print("Old checkpoint exists:", checkpoint.is_dir())

current_trainable = [
    (name, param)
    for name, param in model.named_parameters()
    if param.requires_grad
]

print("Trainable parameter tensors:", len(current_trainable))

for name, param in current_trainable[:5]:
    print(name, tuple(param.shape), param.dtype)

print("Total trainable parameters:",
      sum(param.numel() for _, param in current_trainable))

In [ ]:
import torch
import peft
from pathlib import Path

print("=== CUDA ===")
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

print("\n=== QUANTIZATION ===")
quant_config = model.get_base_model().config
print("Model type:", quant_config.model_type)
print("Model name:", quant_config._name_or_path)

hf_quantizer = getattr(model, "hf_quantizer", None)
print("Quantizer class:", type(hf_quantizer).__name__ if hf_quantizer else None)

print("\n=== PEFT ===")
print("PEFT model:", type(model).__name__)
print("Adapter names:", list(model.peft_config.keys()))

for adapter_name, config in model.peft_config.items():
    print("Adapter:", adapter_name)
    print("Rank:", config.r)
    print("Alpha:", config.lora_alpha)
    print("Task type:", config.task_type)
    print("Target modules:", sorted(config.target_modules))

print("\n=== CHECKPOINT ===")
checkpoint = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora/checkpoint-1000"
)
print("Checkpoint exists:", checkpoint.is_dir())
print("Original checkpoint remains untouched.")

In [ ]:
import os
import subprocess
import sys

print("Python executable:", sys.executable)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "not set"))

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "accelerate.commands.launch",
        "--help",
    ],
    capture_output=True,
    text=True,
    timeout=60,
)

print("Launcher help exit code:", result.returncode)
print("Multi-GPU option available:", "--multi_gpu" in result.stdout)
print("Process-count option available:", "--num_processes" in result.stdout)

if result.returncode != 0:
    print("Launcher error:")
    print(result.stderr[-3000:])
else:
    print("Launcher initialized successfully.")

In [ ]:
from pathlib import Path

smoke_path = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_smoke_test.py"
)

smoke_script = r'''
import os
import torch
import torch.distributed as dist
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
CHECKPOINT = (
    "/kaggle/working/api_pentest_research/"
    "qwen_api_security_lora/checkpoint-1000"
)

local_rank = int(os.environ["LOCAL_RANK"])
rank = int(os.environ["RANK"])
world_size = int(os.environ["WORLD_SIZE"])

torch.cuda.set_device(local_rank)

dist.init_process_group(backend="nccl")

print(
    f"Rank {rank}/{world_size}: using GPU {local_rank}: "
    f"{torch.cuda.get_device_name(local_rank)}",
    flush=True,
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": local_rank},
)

model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT,
    is_trainable=True,
)

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(
    f"Rank {rank}: model and adapter loaded; "
    f"trainable parameters={trainable}",
    flush=True,
)

assert trainable > 0

dist.barrier()

if rank == 0:
    print("DISTRIBUTED SMOKE TEST PASSED", flush=True)

dist.destroy_process_group()
'''

smoke_path.write_text(smoke_script, encoding="utf-8")

print("Smoke-test script created:", smoke_path)
print("Training steps: 0")
print("Original checkpoint modified: No")

In [ ]:
import torch
import subprocess

print("=== GPU MEMORY ===")

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.used,memory.free",
        "--format=csv"
    ],
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else result.stderr)

print("\n=== CURRENT NOTEBOOK MODEL ===")

if "model" in globals():
    print("Model exists in notebook:", type(model).__name__)
    print("Model device map:", getattr(model, "hf_device_map", None))

    for i in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(i)
        print(
            f"GPU {i}: "
            f"{free_bytes / (1024**3):.2f} GiB free / "
            f"{total_bytes / (1024**3):.2f} GiB total"
        )
else:
    print("No model variable found in notebook.")

In [ ]:
import gc
import torch
from pathlib import Path

checkpoint = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora/checkpoint-1000"
)

required_files = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "optimizer.pt",
    "scheduler.pt",
    "trainer_state.json",
]

missing = [name for name in required_files if not (checkpoint / name).is_file()]

if missing:
    print("STOP: checkpoint files missing:", missing)
else:
    print("Checkpoint verified. Releasing notebook model and trainer.")

    del trainer
    del model

    gc.collect()
    torch.cuda.empty_cache()

    print("\nGPU memory after cleanup:")
    for i in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(i)
        print(
            f"GPU {i}: {free_bytes / (1024**3):.2f} GiB free / "
            f"{total_bytes / (1024**3):.2f} GiB total"
        )

    print("\nCheckpoint preserved:", checkpoint.exists())

In [ ]:
import subprocess
import sys
from pathlib import Path

script_path = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_smoke_test.py"
)

command = [
    sys.executable,
    "-m",
    "accelerate.commands.launch",
    "--multi_gpu",
    "--num_processes", "2",
    "--mixed_precision", "no",
    "--main_process_port", "29519",
    str(script_path),
]

print("Retrying distributed smoke test...")
print("GPU processes: 2")
print("Training steps: 0")

result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    timeout=900,
)

print("\n=== STDOUT ===")
print(result.stdout[-6000:])

print("\n=== STDERR (last 6000 characters) ===")
print(result.stderr[-6000:])

print("\nExit code:", result.returncode)
print(
    "Smoke test passed:",
    result.returncode == 0
    and "DISTRIBUTED SMOKE TEST PASSED" in result.stdout
)

In [ ]:
import inspect
import peft.utils.save_and_load as save_load
import transformers.integrations.tensor_parallel as tp

print("PEFT save/load module:", save_load.__file__)
print("Transformers tensor-parallel module:", tp.__file__)

print("\nEmbeddingParallel available:", hasattr(tp, "EmbeddingParallel"))
print("Available tensor-parallel symbols:")
print([
    name for name in dir(tp)
    if "Parallel" in name or "parallel" in name.lower()
])

print("\nPEFT tensor-parallel adapter-loading function:")
print(inspect.getsource(save_load._maybe_shard_state_dict_for_tp))

In [ ]:
import inspect
import peft.utils.save_and_load as save_load

print("Function return statement:")
source = inspect.getsource(save_load._maybe_shard_state_dict_for_tp)
print(source[-700:])

print("\nCurrent checkpoint:")
print("/kaggle/working/api_pentest_research/qwen_api_security_lora/checkpoint-1000")

print("\nDistributed training script exists:",
      __import__("os").path.exists(
          "/kaggle/working/api_pentest_research/finetuning/train_distributed.py"
      ))

print("\nSmoke-test script exists:",
      __import__("os").path.exists(
          "/kaggle/working/api_pentest_research/finetuning/distributed_smoke_test.py"
      ))

In [ ]:

from pathlib import Path
import json

project = Path("/kaggle/working/api_pentest_research")
finetuning = project / "finetuning"
checkpoint = project / "qwen_api_security_lora" / "checkpoint-1000"

patch = '''
import peft.utils.save_and_load as _peft_save_load

def _ddp_skip_tensor_parallel_sharding(model, state_dict, adapter_name):
    for module in model.modules():
        base_getter = getattr(module, "get_base_layer", None)
        if base_getter is None:
            continue
        base_layer = base_getter()
        if (getattr(base_layer, "_hf_tp_plan", None) is not None
                and getattr(base_layer, "_hf_device_mesh", None) is not None):
            raise RuntimeError(
                "This workaround is only valid for DDP models without tensor parallelism."
            )
    return None

_peft_save_load._maybe_shard_state_dict_for_tp = _ddp_skip_tensor_parallel_sharding
'''

for filename in ["train_distributed.py", "distributed_smoke_test.py"]:
    path = finetuning / filename

    if not path.is_file():
        print(f"Missing script: {path}")
        continue

    source = path.read_text()

    if "_ddp_skip_tensor_parallel_sharding" not in source:
        lines = source.splitlines(keepends=True)
        insert_at = 0

        while insert_at < len(lines) and (
            lines[insert_at].startswith("#!")
            or lines[insert_at].startswith("#")
            or not lines[insert_at].strip()
            or lines[insert_at].startswith("from __future__ import")
        ):
            insert_at += 1

        lines.insert(insert_at, patch + "\n")
        path.write_text("".join(lines))
        print(f"Patched: {path}")
    else:
        print(f"Already patched: {path}")

print("\n=== CHECKPOINT STATUS ===")
print("Checkpoint:", checkpoint)
print("Exists:", checkpoint.is_dir())

if checkpoint.is_dir():
    for name in [
        "adapter_config.json",
        "adapter_model.safetensors",
        "optimizer.pt",
        "scheduler.pt",
        "trainer_state.json"
    ]:
        print(f"{name}: {'present' if (checkpoint / name).is_file() else 'MISSING'}")

    config_path = checkpoint / "adapter_config.json"

    if config_path.is_file():
        config = json.loads(config_path.read_text())
        print("Adapter type:", config.get("peft_type"))
        print("LoRA rank:", config.get("r"))
else:
    print("Old checkpoint is absent; this cell will not attempt to resume it.")

print("\nScript inspection complete.")


In [ ]:
import subprocess
import sys
from pathlib import Path

smoke_path = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_smoke_test.py"
)

command = [
    sys.executable,
    "-m", "accelerate.commands.launch",
    "--multi_gpu",
    "--num_processes", "2",
    "--mixed_precision", "no",
    "--main_process_port", "29519",
    str(smoke_path),
]

print("Starting two-GPU smoke test...")
print("No training will be started.\n")

try:
    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
        timeout=900,
    )

    print("Return code:", result.returncode)
    print("\n--- STDOUT ---")
    print(result.stdout[-12000:] or "(empty)")
    print("\n--- STDERR ---")
    print(result.stderr[-12000:] or "(empty)")

except subprocess.TimeoutExpired as exc:
    print("Smoke test timed out after 15 minutes.")
    print("STDOUT:", (exc.stdout or b"")[-6000:])
    print("STDERR:", (exc.stderr or b"")[-6000:])

In [ ]:
from pathlib import Path

path = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed.py"
)
source = path.read_text()
lines = source.splitlines()

keywords = [
    "CHECKPOINT",
    "OUTPUT_DIR",
    "TrainingArguments",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "save_steps",
    "save_total_limit",
    "resume_from_checkpoint",
    "trainer.train",
    "SFTTrainer",
    "Trainer(",
    "ExplicitCompletionCollator",
    "max_steps",
    "num_train_epochs",
]

for i, line in enumerate(lines, start=1):
    if any(word in line for word in keywords):
        start = max(1, i - 2)
        end = min(len(lines), i + 3)
        print(f"\n--- Lines {start}-{end} ---")
        for j in range(start, end + 1):
            print(f"{j:4}: {lines[j-1]}")

In [ ]:

from pathlib import Path
import json

root = Path("/kaggle/working/api_pentest_research")
script = root / "finetuning" / "train_distributed.py"
output_dir = root / "qwen_api_security_lora_distributed"

assert script.is_file(), f"Training script missing: {script}"

if output_dir.exists() and any(output_dir.iterdir()):
    raise RuntimeError(
        f"Output directory is not empty: {output_dir}. "
        "Inspect it before proceeding to avoid overwriting existing work."
    )

output_dir.mkdir(parents=True, exist_ok=True)

source = script.read_text()

old_batch = "per_device_train_batch_size=2,"
new_batch = "per_device_train_batch_size=1,"

if source.count(old_batch) == 1:
    source = source.replace(old_batch, new_batch, 1)
elif source.count(new_batch) == 1:
    print("Per-device batch size is already 1.")
else:
    raise RuntimeError("Could not safely identify the batch-size setting.")

old_resume = 'trainer.train(resume_from_checkpoint=str(OUTPUT_DIR / "checkpoint-1000"))'

if old_resume in source:
    source = source.replace(old_resume, "trainer.train()", 1)
elif "trainer.train()" in source:
    print("Training already starts without resuming a checkpoint.")
else:
    raise RuntimeError(
        "Inspect the training call manually; refusing to change it blindly."
    )

patched_script = root / "finetuning" / "train_distributed_fresh.py"
patched_script.write_text(source)

print("Fresh training script prepared:", patched_script)
print("Output directory:", output_dir)
print("Per-device batch size:", 1)
print("Gradient accumulation: Check the training script")
print("Checkpoint resume: Disabled")
print("Training launched: NO")


In [ ]:

from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
script = root / "finetuning" / "train_distributed_fresh.py"
original_script = root / "finetuning" / "train_distributed.py"
train_data = root / "finetuning" / "train_final.jsonl"
val_data = root / "finetuning" / "validation_sft.jsonl"
output_dir = root / "qwen_api_security_lora_distributed"

checks = {
    "Fresh training script": script.is_file(),
    "Original training script preserved": original_script.is_file(),
    "Training data": train_data.is_file(),
    "Validation data": val_data.is_file(),
    "Fresh output directory": output_dir.is_dir(),
}

if script.is_file():
    source = script.read_text()

    checks["Batch size per GPU is 1"] = (
        "per_device_train_batch_size=1," in source
    )
    checks["Gradient accumulation is 8"] = (
        "gradient_accumulation_steps=8," in source
    )
    checks["No explicit checkpoint-1000 resume"] = (
        'trainer.train(resume_from_checkpoint=str(OUTPUT_DIR / "checkpoint-1000"))'
        not in source
    )
    checks["Compatibility patch present"] = (
        "_ddp_skip_tensor_parallel_sharding" in source
    )

for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {label}")

print("\nCheckpoint resume: Disabled")
print("Training launched: NO")

if not all(checks.values()):
    print("\nSetup needs attention before launching.")
else:
    print("\nBasic fresh-run checks passed.")
    print("Distributed launch configuration still needs verification.")


In [ ]:

from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
finetuning = root / "finetuning"

for filename in [
    "train_distributed.py",
    "train_distributed_fresh.py",
    "distributed_smoke_test.py",
]:
    path = finetuning / filename
    print(f"\n=== {filename} ===")

    if not path.is_file():
        print("NOT FOUND")
        continue

    lines = path.read_text().splitlines()

    for i, line in enumerate(lines):
        if any(term in line for term in [
            "trainer.train(",
            "OUTPUT_DIR",
            "resume_from_checkpoint",
            "per_device_train_batch_size",
            "gradient_accumulation_steps",
            "TrainingArguments",
            "DistributedDataParallel",
        ]):
            start = max(0, i - 1)
            end = min(len(lines), i + 2)
            for j in range(start, end):
                print(f"{j + 1}: {lines[j]}")
            print()


In [ ]:

from pathlib import Path
import subprocess
import json

root = Path("/kaggle/working/api_pentest_research")
finetuning = root / "finetuning"
out_dir = root / "qwen_api_security_lora_distributed"

print("=== TRAINING SCRIPTS ===")
for name in [
    "train_distributed.py",
    "train_distributed_fresh.py",
    "distributed_smoke_test.py",
]:
    path = finetuning / name
    print(f"{name}: {'EXISTS' if path.is_file() else 'MISSING'}")

print("\n=== LOG AND PID FILES ===")
for name in ["distributed_training.log", "distributed_training.pid"]:
    path = finetuning / name
    print(f"{name}: {'EXISTS' if path.is_file() else 'MISSING'}")
    if path.is_file() and name.endswith(".log"):
        print("\n".join(path.read_text(errors="replace").splitlines()[-30:]))

print("\n=== OUTPUT DIRECTORY ===")
print("Exists:", out_dir.is_dir())

if out_dir.is_dir():
    for item in sorted(out_dir.glob("checkpoint-*")):
        state_file = item / "trainer_state.json"
        if state_file.is_file():
            state = json.loads(state_file.read_text())
            print(item.name, "step =", state.get("global_step"))
        else:
            print(item.name, "(trainer_state.json missing)")

print("\n=== GPU STATUS ===")
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout or result.stderr)

print("\n=== TRAINING PROCESSES ===")
result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,ppid,args | grep -E 'accelerate|train_distributed' | grep -v grep"],
    capture_output=True,
    text=True
)
print(result.stdout or "No matching training processes found.")


In [ ]:
from pathlib import Path
import torch
import json

root = Path("/kaggle/working/api_pentest_research")
original = root / "qwen_api_security_lora" / "checkpoint-1000"
copied = root / "qwen_api_security_lora_distributed" / "checkpoint-1000"

for label, checkpoint in [("ORIGINAL", original), ("DISTRIBUTED COPY", copied)]:
    print(f"\n=== {label} ===")
    for name in ["scaler.pt", "optimizer.pt", "scheduler.pt", "trainer_state.json"]:
        path = checkpoint / name
        print(f"{name}: {'present' if path.exists() else 'absent'}")

    args_path = checkpoint / "training_args.bin"
    if args_path.exists():
        args = torch.load(args_path, map_location="cpu", weights_only=False)
        print("Saved fp16:", getattr(args, "fp16", None))
        print("Saved bf16:", getattr(args, "bf16", None))

print("\nOriginal checkpoint remains untouched:", original.is_dir())

In [ ]:

from pathlib import Path
import torch
import inspect
from transformers import Trainer

checkpoint = Path(
    "/kaggle/working/api_pentest_research/qwen_api_security_lora_distributed/checkpoint-1000"
)

scaler_path = checkpoint / "scaler.pt"

if scaler_path.is_file():
    print("Scaler file size:", scaler_path.stat().st_size, "bytes")

    try:
        scaler_state = torch.load(
            scaler_path,
            map_location="cpu",
            weights_only=False,
        )
        print("Scaler state type:", type(scaler_state).__name__)
        print("Scaler state:", scaler_state)
    except Exception as exc:
        print("Scaler read error:", repr(exc))
else:
    print("Scaler file not found.")
    print("Checkpoint exists:", checkpoint.is_dir())
    print("No scaler state can be inspected for this checkpoint.")

print("\nInstalled Trainer scaler restoration method:")
if hasattr(Trainer, "_load_scaler"):
    print(inspect.getsource(Trainer._load_scaler))
else:
    print("This Transformers version has no Trainer._load_scaler method.")


In [ ]:

from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
finetuning = root / "finetuning"

for name in ["train_distributed.py", "train_distributed_fresh.py"]:
    path = finetuning / name
    print(f"\n=== {name} ===")

    if not path.is_file():
        print("File not found")
        continue

    lines = path.read_text().splitlines()

    keywords = [
        "OUTPUT_DIR",
        "TrainingArguments",
        "per_device_train_batch_size",
        "gradient_accumulation_steps",
        "trainer.train(",
        "resume_from_checkpoint",
        "local_rank",
        "torch.distributed",
        "WORLD_SIZE",
    ]

    for i, line in enumerate(lines):
        if any(word in line for word in keywords):
            print(f"{i + 1}: {line.strip()}")


In [ ]:

from pathlib import Path
import subprocess

root = Path("/kaggle/working/api_pentest_research")
script = root / "finetuning" / "train_distributed_fresh.py"
output_dir = root / "qwen_api_security_lora_distributed"
finetuning = root / "finetuning"

print("Fresh script exists:", script.is_file())
print("Output directory exists:", output_dir.is_dir())

if output_dir.is_dir():
    items = list(output_dir.iterdir())
    print("Output directory entries:", len(items))
    for item in items[:20]:
        print(" ", item.name)

print("\nTraining processes:")
result = subprocess.run(
    ["bash", "-lc",
     "ps -eo pid,ppid,args | grep -E 'accelerate|train_distributed' | grep -v grep"],
    capture_output=True,
    text=True
)
print(result.stdout or "No matching training process found.")

pid_path = finetuning / "distributed_training.pid"
print("\nPID file exists:", pid_path.is_file())
if pid_path.is_file():
    print("Recorded PID:", pid_path.read_text().strip())


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)

lines = script.read_text().splitlines()

for start, end in [(85, 155), (150, 200)]:
    print(f"\n=== SCRIPT LINES {start}-{end} ===")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1}: {lines[i]}")


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)

lines = script.read_text().splitlines()

print("=== SCRIPT CONFIGURATION ===")
for i, line in enumerate(lines[:85], start=1):
    if any(key in line for key in [
        "ROOT =",
        "BASE_MODEL",
        "CHECKPOINT",
        "OUTPUT_DIR",
        "MAX_LENGTH",
        "train_path",
        "val_path",
        "def read_text_dataset",
        "def __init__",
    ]):
        print(f"{i}: {line}")

print("\n=== CHECKPOINT PATHS ===")
root = Path("/kaggle/working/api_pentest_research")

for path in [
    root / "qwen_api_security_lora",
    root / "qwen_api_security_lora" / "checkpoint-1000",
    root / "qwen_api_security_lora_distributed",
    root / "qwen_api_security_lora_distributed" / "checkpoint-1000",
]:
    print(path, "EXISTS" if path.exists() else "MISSING")


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)

source = script.read_text()

old_model_block = '''base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": local_rank},
)
model = PeftModel.from_pretrained(
    base_model,
    CHECKPOINT,
    is_trainable=True,
)'''

new_model_block = '''base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": local_rank},
)

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=False,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.config.use_cache = False'''

if old_model_block not in source:
    raise RuntimeError("Model block did not match. No changes made.")

if "from pathlib import Path" not in source:
    raise RuntimeError("Could not find the expected import location.")

source = source.replace(
    "from pathlib import Path",
    "from pathlib import Path\nfrom peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training",
    1,
)

source = source.replace(old_model_block, new_model_block, 1)
source = source.replace("    fp16=False,", "    fp16=True,", 1)
source = source.replace("    gradient_checkpointing=True,", "    gradient_checkpointing=False,", 1)
source = source.replace(
    'print("Resume checkpoint:", CHECKPOINT)',
    'print("Training mode: fresh LoRA adapter")',
    1,
)

script.write_text(source)

print("Script patched for fresh LoRA training.")
print("FP16 enabled; gradient checkpointing disabled for speed.")
print("Deleted checkpoint will not be loaded.")


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)
source = script.read_text()

checks = {
    "Fresh LoRA adapter": "model = get_peft_model(base_model, lora_config)" in source,
    "FP16 enabled": "fp16=True" in source,
    "Gradient checkpointing disabled": "gradient_checkpointing=False" in source,
    "Deleted checkpoint not loaded": "PeftModel.from_pretrained(" not in source,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

if not all(checks.values()):
    raise RuntimeError("Do not launch until all checks pass.")


In [ ]:

import subprocess
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
script = root / "finetuning" / "train_distributed_fresh.py"
log_path = root / "finetuning" / "distributed_training.log"
pid_path = root / "finetuning" / "distributed_training.pid"
output_dir = root / "qwen_api_security_lora_distributed"

if not script.exists():
    raise FileNotFoundError(script)

if any(output_dir.iterdir()):
    raise RuntimeError(
        f"Output directory is not empty: {output_dir}. "
        "Inspect it before starting to avoid overwriting existing results."
    )

log_file = open(log_path, "w")
process = subprocess.Popen(
    [
        "accelerate", "launch",
        "--multi_gpu",
        "--num_processes", "2",
        "--mixed_precision", "fp16",
        "--main_process_port", "29521",
        str(script),
    ],
    cwd=str(root),
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

pid_path.write_text(str(process.pid))
print("Training launcher started.")
print("PID:", process.pid)
print("Log:", log_path)


In [ ]:

import subprocess
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")

log = root / "finetuning" / "distributed_training.log"
if log.exists():
    lines = log.read_text(errors="replace").splitlines()
    print("\n".join(lines[-35:]))

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi"])


In [ ]:

from pathlib import Path

log = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_training.log"
)

lines = log.read_text(errors="replace").splitlines()

print("=== ERROR DETAILS ===")
for i, line in enumerate(lines):
    if any(term in line.lower() for term in [
        "traceback", "error:", "exception:", "failed", "nameerror",
        "importerror", "runtimeerror", "valueerror"
    ]):
        start = max(0, i - 3)
        end = min(len(lines), i + 12)
        print("\n".join(f"{j+1}: {lines[j]}" for j in range(start, end)))
        print("-" * 60)

print("\n=== LAST 100 LOG LINES ===")
print("\n".join(
    f"{i+1}: {line}"
    for i, line in enumerate(lines[-100:], start=max(0, len(lines)-100))
))


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)

source = script.read_text()

old = "tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)"
new = "tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)"

if old not in source:
    raise RuntimeError(
        "Expected tokenizer line not found. No changes made."
    )

source = source.replace(old, new, 1)
script.write_text(source)

print("Tokenizer now loads from:", "Qwen/Qwen2.5-1.5B-Instruct")
print("Deleted checkpoint is no longer used for tokenizer loading.")


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)
source = script.read_text()

assert "AutoTokenizer.from_pretrained(BASE_MODEL)" in source
assert "model = get_peft_model(base_model, lora_config)" in source
assert "PeftModel.from_pretrained(" not in source
assert "fp16=True" in source

print("All fresh-training checks passed.")


In [ ]:

from pathlib import Path

log = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_training.log"
)

if log.exists():
    print("\n".join(log.read_text(errors="replace").splitlines()[-40:]))
else:
    print("Training log not found.")


In [ ]:

import subprocess
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
script_name = "train_distributed_fresh.py"

result = subprocess.run(
    ["ps", "-eo", "pid=,args="],
    capture_output=True,
    text=True,
)

matches = []
for line in result.stdout.splitlines():
    if script_name in line and "ps -eo" not in line:
        matches.append(line.strip())

print("=== MATCHING PROCESSES ===")
print("\n".join(matches) if matches else "No training script process found.")

log_path = root / "finetuning" / "distributed_training.log"
if log_path.exists():
    print("\n=== LATEST LOG ===")
    print("\n".join(log_path.read_text(errors="replace").splitlines()[-30:]))


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_distributed_fresh.py"
)

lines = script.read_text().splitlines()

print("=== CHECKPOINT REFERENCES ===")
matches = [
    (i, line.strip())
    for i, line in enumerate(lines, 1)
    if "CHECKPOINT" in line or "checkpoint-1000" in line
]

if matches:
    for i, line in matches:
        print(f"{i}: {line}")
else:
    print("No checkpoint references found.")

print("\n=== TOKENIZER AND MODEL LOADING ===")
for i, line in enumerate(lines, 1):
    if any(term in line for term in [
        "AutoTokenizer.from_pretrained",
        "AutoModelForCausalLM.from_pretrained",
        "PeftModel.from_pretrained",
        "get_peft_model",
    ]):
        print(f"{i}: {line.strip()}")


In [ ]:

from pathlib import Path
import subprocess
import time

root = Path("/kaggle/working/api_pentest_research")
script = root / "finetuning" / "train_distributed_fresh.py"
log_path = root / "finetuning" / "distributed_training_fresh.log"
pid_path = root / "finetuning" / "distributed_training_fresh.pid"
output_dir = root / "qwen_api_security_lora_distributed"

source = script.read_text()
source = source.replace(
    'CHECKPOINT = ROOT / "qwen_api_security_lora" / "checkpoint-1000"\n',
    "",
)
script.write_text(source)

assert "AutoTokenizer.from_pretrained(BASE_MODEL)" in source
assert "model = get_peft_model(base_model, lora_config)" in source
assert "PeftModel.from_pretrained(" not in source
assert "fp16=True" in source

output_dir.mkdir(parents=True, exist_ok=True)
if any(output_dir.iterdir()):
    raise RuntimeError(
        "Training output directory is not empty. Inspect it before launching."
    )

processes = subprocess.run(
    ["ps", "-eo", "pid=,args="],
    capture_output=True,
    text=True,
).stdout.splitlines()

active = [
    line.strip() for line in processes
    if "train_distributed_fresh.py" in line
    and "ps -eo" not in line
    and "accelerate launch" in line
]

if active:
    raise RuntimeError("A matching training launcher already exists:\n" + "\n".join(active))

with open(log_path, "w") as log:
    proc = subprocess.Popen(
        [
            "accelerate", "launch",
            "--multi_gpu",
            "--num_processes", "2",
            "--mixed_precision", "fp16",
            "--main_process_port", "29521",
            str(script),
        ],
        cwd=str(root),
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

pid_path.write_text(str(proc.pid))
time.sleep(30)

print("Launcher PID:", proc.pid)
print("Log:", log_path)
print("\n=== LATEST LOG ===")
print("\n".join(log_path.read_text(errors="replace").splitlines()[-50:]))


In [ ]:

from pathlib import Path
import subprocess
import time

log = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_training_fresh.log"
)

time.sleep(30)

lines = log.read_text(errors="replace").splitlines()
print("=== LATEST TRAINING LOG ===")
print("\n".join(lines[-35:]))

print("\n=== GPU UTILIZATION ===")
subprocess.run(["nvidia-smi"])


In [ ]:

from pathlib import Path
import subprocess

root = Path("/kaggle/working/api_pentest_research")
log = root / "finetuning" / "distributed_training_fresh.log"

lines = log.read_text(errors="replace").splitlines()

print("=== RECENT TRAINING OUTPUT ===")
print("\n".join(lines[-20:]))

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi"])


In [ ]:

from pathlib import Path
import subprocess
import time

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_distributed_fresh.py"
LOG = ROOT / "finetuning" / "distributed_training_restart.log"
PID_FILE = ROOT / "finetuning" / "distributed_training.pid"
OUTPUT = ROOT / "qwen_api_security_lora_distributed"

assert SCRIPT.exists(), f"Training script missing: {SCRIPT}"

source = SCRIPT.read_text()
assert "AutoTokenizer.from_pretrained(BASE_MODEL)" in source
assert "model = get_peft_model(base_model, lora_config)" in source
assert "PeftModel.from_pretrained(" not in source
assert "fp16=True" in source

# Check for an existing training process.
result = subprocess.run(
    ["ps", "-eo", "pid=,args="],
    capture_output=True,
    text=True,
)

active = [
    line.strip()
    for line in result.stdout.splitlines()
    if "train_distributed_fresh.py" in line
    and "ps -eo" not in line
]

if active:
    print("A training process already exists; not launching another.")
    print("\n".join(active))
    print("\nRecent log:")
    old_log = ROOT / "finetuning" / "distributed_training_fresh.log"
    if old_log.exists():
        print("\n".join(
            old_log.read_text(errors="replace").splitlines()[-25:]
        ))
else:
    checkpoints = list(OUTPUT.glob("checkpoint-*")) if OUTPUT.exists() else []

    if checkpoints:
        raise RuntimeError(
            "Existing checkpoints found. Do not start a fresh run over them. "
            f"Found: {checkpoints}"
        )

    OUTPUT.mkdir(parents=True, exist_ok=True)

    with open(LOG, "w") as log_file:
        process = subprocess.Popen(
            [
                "accelerate", "launch",
                "--multi_gpu",
                "--num_processes", "2",
                "--mixed_precision", "fp16",
                "--main_process_port", "29521",
                str(SCRIPT),
            ],
            cwd=str(ROOT),
            stdout=log_file,
            stderr=subprocess.STDOUT,
            start_new_session=True,
        )

    PID_FILE.write_text(str(process.pid))
    time.sleep(30)

    print("Launcher PID:", process.pid)
    print("Log file:", LOG)
    print("\n=== TRAINING LOG ===")
    print("\n".join(LOG.read_text(errors="replace").splitlines()[-40:]))

    print("\n=== GPU STATUS ===")
    subprocess.run(["nvidia-smi"])


In [ ]:

from pathlib import Path
import subprocess
import time

log = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_training_fresh.log"
)

def get_step():
    lines = log.read_text(errors="replace").splitlines()
    for line in reversed(lines):
        import re
        match = re.search(r"(\d+)/21949", line)
        if match:
            return int(match.group(1))
    return None

before = get_step()

print("Step before:", before)
time.sleep(20)
after = get_step()

print("Step after 20 seconds:", after)
print("Training advancing:", after is not None and before is not None and after > before)

print("\n=== TRAINING PROCESSES ===")
subprocess.run(
    ["bash", "-lc", "ps -p 546,565,566 -o pid,stat,etime,args"]
)


In [ ]:
from pathlib import Path
import json
from datetime import datetime

root = Path("/kaggle/working/api_pentest_research")

for folder in [
    root / "qwen_api_security_lora",
    root / "qwen_api_security_lora_distributed",
]:
    print(f"\n=== {folder} ===")

    if not folder.exists():
        print("Directory not found")
        continue

    checkpoints = sorted(
        folder.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1])
        if p.name.split("-")[-1].isdigit() else -1
    )

    for ckpt in checkpoints:
        state_file = ckpt / "trainer_state.json"
        adapter_file = ckpt / "adapter_model.safetensors"

        step = "unknown"
        if state_file.exists():
            try:
                state = json.loads(state_file.read_text())
                step = state.get("global_step", "unknown")
            except Exception:
                pass

        print(
            ckpt.name,
            "| step:", step,
            "| adapter:", adapter_file.exists(),
            "| modified:", datetime.fromtimestamp(
                ckpt.stat().st_mtime
            ).strftime("%Y-%m-%d %H:%M:%S")
        )

In [ ]:
from pathlib import Path
import json
import subprocess
import socket

ROOT = Path("/kaggle/working/api_pentest_research")

print("=== PROJECT STATUS ===")
print("Project directory exists:", ROOT.exists())

if ROOT.exists():
    for name in [
        "finetuning",
        "qwen_api_security_lora",
        "qwen_api_security_lora_distributed",
    ]:
        folder = ROOT / name
        print(f"\n{name}: {'FOUND' if folder.exists() else 'NOT FOUND'}")

        if folder.exists():
            checkpoints = sorted(
                folder.glob("checkpoint-*"),
                key=lambda p: int(p.name.split("-")[-1])
                if p.name.split("-")[-1].isdigit() else -1
            )

            for ckpt in checkpoints[-5:]:
                state_file = ckpt / "trainer_state.json"
                step = "unknown"

                if state_file.exists():
                    try:
                        step = json.loads(
                            state_file.read_text()
                        ).get("global_step", "unknown")
                    except Exception:
                        pass

                print(
                    ckpt.name,
                    "| step:", step,
                    "| adapter:", (ckpt / "adapter_model.safetensors").exists()
                )

print("\n=== TRAINING FILES ===")
training_dir = ROOT / "finetuning"

for name in [
    "train_distributed_fresh.py",
    "train_distributed.py",
    "distributed_training_fresh.log",
    "distributed_training_restart.log",
    "train_final.jsonl",
    "test_holdout.jsonl",
]:
    path = training_dir / name
    print(name, "|", "FOUND" if path.exists() else "NOT FOUND")

    if path.is_file() and path.suffix == ".log":
        lines = path.read_text(errors="replace").splitlines()
        print("\n".join(lines[-5:]))

print("\n=== API LAB STATUS ===")
try:
    with socket.create_connection(("127.0.0.1", 8000), timeout=2):
        print("Port 8000 is accepting connections")
except OSError:
    print("Port 8000 is not accepting connections")

print("\n=== TRAINING PROCESS STATUS ===")
result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,stat,etime,args"],
    capture_output=True,
    text=True
)

matches = [
    line for line in result.stdout.splitlines()
    if "train_distributed" in line
    and "grep" not in line
    and "ps -eo" not in line
]

print("\n".join(matches) if matches else "No matching training process found")

print("\n=== GPU STATUS ===")
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
from pathlib import Path
import json
import re

ROOT = Path("/kaggle/working/api_pentest_research")
TRAIN = ROOT / "finetuning"
CKPT_DIR = ROOT / "qwen_api_security_lora_distributed"

print("=== CHECKPOINTS ===")
for ckpt in [CKPT_DIR / "checkpoint-2500", CKPT_DIR / "checkpoint-3000"]:
    print(f"\n{ckpt}: {'FOUND' if ckpt.exists() else 'MISSING'}")

    if ckpt.exists():
        for name in [
            "adapter_model.safetensors",
            "adapter_config.json",
            "trainer_state.json",
            "optimizer.pt",
            "scheduler.pt",
            "rng_state.pth",
        ]:
            p = ckpt / name
            print(f"  {name}: {'YES' if p.exists() else 'NO'}")

        state = ckpt / "trainer_state.json"
        if state.exists():
            data = json.loads(state.read_text())
            print("  global_step:", data.get("global_step"))
            print("  epoch:", data.get("epoch"))
            print("  max_steps:", data.get("max_steps"))

print("\n=== TRAINING CONFIGURATION ===")
for filename in ["train_distributed_fresh.py", "train_distributed.py"]:
    path = TRAIN / filename
    print(f"\n--- {filename}: {'FOUND' if path.exists() else 'MISSING'} ---")

    if path.exists():
        source = path.read_text(errors="replace")
        for i, line in enumerate(source.splitlines(), 1):
            if re.search(
                r"resume_from_checkpoint|output_dir|num_train_epochs|max_steps|"
                r"per_device_train_batch_size|gradient_accumulation_steps|"
                r"save_steps|logging_steps|learning_rate|train_final|"
                r"load_dataset|SFTTrainer|TrainingArguments",
                line,
                re.IGNORECASE,
            ):
                print(f"{i}: {line[:220]}")

print("\n=== DATASET SIZE ===")
for filename in ["train_final.jsonl", "test_holdout.jsonl", "validation_sft.jsonl"]:
    path = TRAIN / filename
    if path.exists():
        with path.open("r", errors="replace") as f:
            count = sum(1 for _ in f)
        print(f"{filename}: {count:,} rows")
    else:
        print(f"{filename}: NOT FOUND")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
script = ROOT / "finetuning" / "train_distributed_fresh.py"

lines = script.read_text(errors="replace").splitlines()

print("=== SCRIPT: LINES 1–100 ===")
for i, line in enumerate(lines[:100], 1):
    print(f"{i:03}: {line}")

print("\n=== SCRIPT: LINES 100–220 ===")
for i, line in enumerate(lines[100:220], 101):
    print(f"{i:03}: {line}")

print("\n=== CHECKPOINT DIRECTORY CONTENTS ===")
for name in ["checkpoint-2500", "checkpoint-3000"]:
    folder = ROOT / "qwen_api_security_lora_distributed" / name
    print(f"\n{name}")
    if folder.exists():
        for p in sorted(folder.iterdir()):
            print(f"{p.name} | {p.stat().st_size / (1024**2):.2f} MB")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
SOURCE = ROOT / "finetuning" / "train_distributed_fresh.py"
NEW_SCRIPT = ROOT / "finetuning" / "train_resume_3000.py"
CHECKPOINT = ROOT / "qwen_api_security_lora_distributed" / "checkpoint-3000"
NEW_OUTPUT = ROOT / "qwen_api_security_lora_resume_3000"

assert SOURCE.exists(), "Original training script is missing"
assert CHECKPOINT.exists(), "Checkpoint-3000 is missing"
assert (CHECKPOINT / "adapter_model.safetensors").exists(), "Checkpoint adapter is missing"
assert (CHECKPOINT / "optimizer.pt").exists(), "Checkpoint optimizer state is missing"
assert (CHECKPOINT / "scheduler.pt").exists(), "Checkpoint scheduler state is missing"
assert (CHECKPOINT / "rng_state_0.pth").exists(), "Rank 0 RNG state is missing"
assert (CHECKPOINT / "rng_state_1.pth").exists(), "Rank 1 RNG state is missing"
assert not NEW_SCRIPT.exists(), "Resume script already exists; refusing to overwrite it"
assert not NEW_OUTPUT.exists(), "Resume output already exists; refusing to reuse it"

source = SOURCE.read_text()

old_output = 'OUTPUT_DIR = ROOT / "qwen_api_security_lora_distributed"'
assert old_output in source
source = source.replace(
    old_output,
    'OUTPUT_DIR = ROOT / "qwen_api_security_lora_resume_3000"',
    1,
)

old_epochs = "    num_train_epochs=1,"
assert old_epochs in source
source = source.replace(
    old_epochs,
    "    num_train_epochs=1,\n    max_steps=4500,",
    1,
)

old_train = "trainer.train()"
assert source.count(old_train) == 1
source = source.replace(
    old_train,
    'trainer.train(resume_from_checkpoint=str(ROOT / "qwen_api_security_lora_distributed" / "checkpoint-3000"))',
    1,
)

NEW_SCRIPT.write_text(source)

print("Resume script created:", NEW_SCRIPT)
print("Checkpoint to resume:", CHECKPOINT)
print("New output directory:", NEW_OUTPUT)
print("Target total steps: 4500")
print("Additional steps planned: 1500")
print("Original checkpoints modified: NO")
print("Training started: NO")

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_resume_3000.py"
CKPT = ROOT / "qwen_api_security_lora_distributed" / "checkpoint-3000"
OUT = ROOT / "qwen_api_security_lora_resume_3000"

source = SCRIPT.read_text()
tree = ast.parse(source)

settings = {}
for node in tree.body:
    if isinstance(node, ast.Assign):
        for target in node.targets:
            if isinstance(target, ast.Name) and target.id in {"ROOT", "OUTPUT_DIR", "MAX_LENGTH"}:
                try:
                    settings[target.id] = ast.literal_eval(node.value)
                except (ValueError, TypeError):
                    pass

print("=== CONFIGURATION ===")
print("Output directory variable configured:", 'OUTPUT_DIR = ROOT / "qwen_api_security_lora_resume_3000"' in source)
print("Resume checkpoint configured:", "resume_from_checkpoint=str(ROOT / " in source and '"checkpoint-3000"' in source)
print("Max steps configured:", "max_steps=4500" in source)
print("New output directory already exists:", OUT.exists())
print("Checkpoint exists:", CKPT.exists())
print("Checkpoint adapter exists:", (CKPT / "adapter_model.safetensors").exists())

assert 'OUTPUT_DIR = ROOT / "qwen_api_security_lora_resume_3000"' in source
assert "resume_from_checkpoint=str(ROOT / " in source
assert '"checkpoint-3000"' in source
assert "max_steps=4500" in source
assert not OUT.exists(), "Output directory already exists; inspect it before proceeding."
assert (CKPT / "adapter_model.safetensors").exists()

print("\nConfiguration checks passed. Training has NOT started.")

In [ ]:
from pathlib import Path
import subprocess
import time

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_resume_3000.py"
LOG = ROOT / "finetuning" / "resume_3000.log"
OUTPUT = ROOT / "qwen_api_security_lora_resume_3000"

assert SCRIPT.exists()
assert not OUTPUT.exists(), "Output directory exists; inspect before launching."
assert not LOG.exists(), "Resume log already exists; inspect before launching."

with open(LOG, "w", buffering=1) as log_file:
    process = subprocess.Popen(
        [
            "accelerate", "launch",
            "--multi_gpu",
            "--num_processes", "2",
            "--mixed_precision", "fp16",
            "--main_process_port", "29523",
            str(SCRIPT),
        ],
        cwd=str(ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Launched process PID:", process.pid)
print("Log file:", LOG)
print("Waiting 30 seconds for startup...")
time.sleep(30)

print("\n=== LATEST TRAINING LOG ===")
if LOG.exists():
    lines = LOG.read_text(errors="replace").splitlines()
    print("\n".join(lines[-35:]) if lines else "Log is currently empty.")

print("\n=== GPU STATUS ===")
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
)
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/api_pentest_research")
LOG = ROOT / "finetuning" / "resume_3000.log"

print("=== PROCESS STATUS ===")
result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,ppid,stat,etime,args"],
    capture_output=True,
    text=True,
)
print("\n".join(
    line for line in result.stdout.splitlines()
    if "accelerate" in line
    or "train_resume_3000.py" in line
    or "torchrun" in line
) or "No matching processes found.")

print("\n=== LOG SIZE AND LAST 100 LINES ===")
if LOG.exists():
    print("Log size:", LOG.stat().st_size, "bytes")
    lines = LOG.read_text(errors="replace").splitlines()
    print("\n".join(lines[-100:]) if lines else "Log is empty.")
else:
    print("Log file missing.")

print("\n=== GPU STATUS ===")
result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True,
)
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
import sys
import importlib.metadata as metadata
import torch

print("=== PYTHON / PYTORCH ===")
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("\n=== PACKAGE VERSIONS ===")
for package in [
    "bitsandbytes",
    "transformers",
    "peft",
    "accelerate",
    "trl",
    "datasets",
    "triton",
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\n=== BITSANDBYTES IMPORT TEST ===")
try:
    import bitsandbytes as bnb
    print("Import succeeded")
    print("Module version:", getattr(bnb, "__version__", "unknown"))
except Exception as exc:
    print(type(exc).__name__ + ":", str(exc))

print("\n=== GPU DEVICES ===")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))

In [ ]:
%pip install -q "bitsandbytes>=0.46.1"

In [ ]:
import bitsandbytes as bnb
import torch
from transformers import BitsAndBytesConfig

print("bitsandbytes:", bnb.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("4-bit configuration created successfully")
print("Training has NOT started.")

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = None

try:
    print("Loading base model on GPU 0 in 4-bit mode...")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=quant_config,
        device_map={"": 0},
    )

    print("\nSUCCESS: Base model loaded.")
    print("Model type:", type(model).__name__)
    print("Model device:", next(model.parameters()).device)
    print("4-bit quantization:", getattr(model, "is_loaded_in_4bit", False))
    print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

except Exception as exc:
    print("\nMODEL LOAD FAILED")
    print(type(exc).__name__ + ":", str(exc))
    raise

finally:
    if model is not None:
        del model
    gc.collect()
    torch.cuda.empty_cache()
    print("\nTest complete; model memory released.")

In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

ROOT = "/kaggle/working/api_pentest_research"
CHECKPOINT = f"{ROOT}/qwen_api_security_lora_distributed/checkpoint-3000"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = None
model = None

try:
    print("Loading quantized base model...")
    base_model = AutoModelForCausalLM.from_pretrained(
        "Qwen/Qwen2.5-1.5B-Instruct",
        quantization_config=quant_config,
        device_map={"": 0},
    )

    print("Loading saved LoRA adapter as trainable...")
    model = PeftModel.from_pretrained(
        base_model,
        CHECKPOINT,
        is_trainable=True,
    )

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())

    print("\n=== ADAPTER CHECK ===")
    print("Adapter loaded: YES")
    print("Trainable parameters:", f"{trainable:,}")
    print("Total parameters:", f"{total:,}")
    print("Trainable adapter:", trainable > 0)
    print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

    assert trainable > 0, "No trainable parameters found."

except Exception as exc:
    print("\nADAPTER TEST FAILED")
    print(type(exc).__name__ + ":", str(exc))
    raise

finally:
    if model is not None:
        del model
    if base_model is not None:
        del base_model
    gc.collect()
    torch.cuda.empty_cache()
    print("\nTest complete; GPU memory released.")

In [ ]:
from pathlib import Path
import subprocess
import time

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_resume_3000.py"
LOG = ROOT / "finetuning" / "resume_3000_attempt2.log"
CHECKPOINT = ROOT / "qwen_api_security_lora_distributed" / "checkpoint-3000"
OUTPUT = ROOT / "qwen_api_security_lora_resume_3000"

assert SCRIPT.exists()
assert CHECKPOINT.exists()
assert not LOG.exists(), "This attempt's log already exists; inspect it first."
assert not OUTPUT.exists(), "Resume output directory already exists; inspect before launching."

with open(LOG, "w", buffering=1) as log_file:
    process = subprocess.Popen(
        [
            "accelerate", "launch",
            "--num_machines", "1",
            "--num_processes", "2",
            "--mixed_precision", "fp16",
            "--main_process_port", "29525",
            str(SCRIPT),
        ],
        cwd=str(ROOT),
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Launched PID:", process.pid)
print("Log:", LOG)
print("Waiting 60 seconds...")
time.sleep(60)

print("\n=== LATEST LOG ===")
lines = LOG.read_text(errors="replace").splitlines()
print("\n".join(lines[-50:]) if lines else "Log is empty.")

print("\n=== GPU STATUS ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:
from pathlib import Path
import subprocess
import time
import json

ROOT = Path("/kaggle/working/api_pentest_research")
LOG = ROOT / "finetuning" / "resume_3000_attempt2.log"
OUT = ROOT / "qwen_api_security_lora_resume_3000"

print("Waiting 60 seconds to check progress...")
time.sleep(60)

print("\n=== LATEST TRAINING LOG ===")
lines = LOG.read_text(errors="replace").splitlines()
print("\n".join(lines[-35:]))

print("\n=== RECENT CHECKPOINTS ===")
if OUT.exists():
    checkpoints = sorted(
        OUT.glob("checkpoint-*"),
        key=lambda p: int(p.name.split("-")[-1])
    )
    if checkpoints:
        for ckpt in checkpoints[-5:]:
            state_file = ckpt / "trainer_state.json"
            step = "unknown"
            if state_file.exists():
                try:
                    step = json.loads(state_file.read_text()).get("global_step", "unknown")
                except Exception:
                    pass
            print(ckpt.name, "| step:", step)
    else:
        print("No checkpoint saved yet.")
else:
    print("Output directory not found.")

print("\n=== TRAINING PROCESS ===")
result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,stat,etime,args"],
    capture_output=True,
    text=True,
)
print("\n".join(
    line for line in result.stdout.splitlines()
    if "train_resume_3000.py" in line or "accelerate launch" in line
) or "No matching process found.")

print("\n=== GPU STATUS ===")
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

In [ ]:

from pathlib import Path
import subprocess

log = Path(
    "/kaggle/working/api_pentest_research/finetuning/distributed_training_fresh.log"
)

lines = log.read_text(errors="replace").splitlines()

print("=== LATEST TRAINING PROGRESS ===")
for line in lines[-30:]:
    print(line)

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi"])


In [ ]:
from pathlib import Path
import re
import subprocess
import time

ROOT = Path("/kaggle/working/api_pentest_research")
LOG = ROOT / "finetuning" / "resume_3000_attempt2.log"

def get_progress():
    if not LOG.exists():
        return None

    text = LOG.read_text(errors="replace")
    matches = re.findall(r"(\d+)/4500", text)
    return int(matches[-1]) if matches else None

def show_status(label):
    step = get_progress()
    print(f"{label}: step {step}/4500" if step is not None else f"{label}: step not found")

    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used", "--format=csv,noheader"],
        capture_output=True,
        text=True,
    )
    print("GPU utilization, memory:")
    print(result.stdout.strip() if result.returncode == 0 else result.stderr)

print("=== INITIAL STATUS ===")
show_status("Before waiting")

print("\nWaiting 60 seconds...")
time.sleep(60)

print("\n=== STATUS AFTER 60 SECONDS ===")
show_status("After waiting")

print("\n=== LATEST LOG LINES ===")
lines = LOG.read_text(errors="replace").splitlines()
print("\n".join(lines[-15:]))

In [ ]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/api_pentest_research")
OUT = ROOT / "qwen_api_security_lora_resume_3000"
LOG = ROOT / "finetuning" / "resume_3000_attempt2.log"

print("=== FINAL MODEL FILES ===")
for name in [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]:
    path = OUT / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")
    if path.exists():
        print(f"  Size: {path.stat().st_size / (1024**2):.2f} MB")

print("\n=== SAVED CHECKPOINTS ===")
checkpoints = sorted(
    OUT.glob("checkpoint-*"),
    key=lambda p: int(p.name.split("-")[-1])
    if p.name.split("-")[-1].isdigit() else -1
)

if not checkpoints:
    print("No checkpoint directories found.")
else:
    for ckpt in checkpoints:
        state_file = ckpt / "trainer_state.json"
        step = "unknown"
        if state_file.exists():
            try:
                step = json.loads(state_file.read_text()).get("global_step", "unknown")
            except Exception:
                pass
        print(ckpt.name, "| global_step:", step,
              "| adapter:", (ckpt / "adapter_model.safetensors").exists())

print("\n=== FINAL TRAINING SUMMARY ===")
if LOG.exists():
    lines = LOG.read_text(errors="replace").splitlines()
    for line in lines[-15:]:
        print(line)
else:
    print("Training log not found.")

In [ ]:
from pathlib import Path
import json

ROOT = Path("/kaggle/working/api_pentest_research")
CKPT = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"

print("=== FINAL CHECKPOINT ===")
print("Path:", CKPT)
print("Exists:", CKPT.exists())

assert CKPT.exists(), "Final checkpoint directory is missing."

for name in [
    "adapter_model.safetensors",
    "adapter_config.json",
    "trainer_state.json",
    "optimizer.pt",
    "scheduler.pt",
    "rng_state_0.pth",
    "rng_state_1.pth",
    "tokenizer.json",
    "tokenizer_config.json",
]:
    p = CKPT / name
    print(f"{name}: {'FOUND' if p.exists() else 'MISSING'}")
    if p.exists():
        print(f"  Size: {p.stat().st_size / (1024**2):.2f} MB")

state_file = CKPT / "trainer_state.json"
if state_file.exists():
    state = json.loads(state_file.read_text())
    print("\n=== TRAINING STATE ===")
    print("Global step:", state.get("global_step"))
    print("Epoch:", state.get("epoch"))
    print("Best metric:", state.get("best_metric"))

config_file = CKPT / "adapter_config.json"
if config_file.exists():
    config = json.loads(config_file.read_text())
    print("\n=== ADAPTER CONFIGURATION ===")
    print("Base model:", config.get("base_model_name_or_path"))
    print("LoRA rank:", config.get("r"))
    print("LoRA alpha:", config.get("lora_alpha"))

print("\nOriginal checkpoints have not been modified.")

In [ ]:
from pathlib import Path
import shutil
import zipfile

ROOT = Path("/kaggle/working/api_pentest_research")
CKPT = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"

EXPORT_DIR = ROOT / "MSK_API_Security_Model"
ZIP_PATH = Path("/kaggle/working/MSK_API_Security_Model_step4500.zip")

required = [
    "adapter_model.safetensors",
    "adapter_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

for name in required:
    assert (CKPT / name).exists(), f"Required file missing: {name}"

if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)

EXPORT_DIR.mkdir(parents=True)

optional = [
    "chat_template.jinja",
    "special_tokens_map.json",
    "generation_config.json",
]

for name in required + optional:
    src = CKPT / name
    if src.exists():
        shutil.copy2(src, EXPORT_DIR / name)

readme = """# MSK API Security Model — Step 4500

Type: LoRA adapter checkpoint
Base model: Qwen/Qwen2.5-1.5B-Instruct
LoRA rank: 16
LoRA alpha: 32
Training global step: 4500

This package contains the adapter, not the full base model.
The base model must be available separately to load this adapter.

Example loading code:

from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map="auto",
)

model = PeftModel.from_pretrained(base, "./MSK_API_Security_Model")
"""

(EXPORT_DIR / "README.txt").write_text(readme)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in EXPORT_DIR.iterdir():
        zf.write(path, arcname=f"MSK_API_Security_Model/{path.name}")

print("Export directory:", EXPORT_DIR)
print("ZIP created:", ZIP_PATH)
print("ZIP size: %.2f MB" % (ZIP_PATH.stat().st_size / (1024**2)))
print("\nFiles included:")
for path in sorted(EXPORT_DIR.iterdir()):
    print(path.name, "| %.2f MB" % (path.stat().st_size / (1024**2)))

from IPython.display import display, FileLink
display(FileLink(str(ZIP_PATH)))

In [ ]:
from pathlib import Path
from collections import Counter
import json

ROOT = Path("/kaggle/working/api_pentest_research")
DATA = ROOT / "finetuning"
MODEL = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
TEST = DATA / "test_holdout.jsonl"

print("=== MODEL CHECKPOINT ===")
print("Checkpoint exists:", MODEL.exists())

for name in ["adapter_model.safetensors", "adapter_config.json"]:
    print(name, ":", (MODEL / name).exists())

print("\n=== HOLDOUT DATASET ===")
print("Test file exists:", TEST.exists())

assert MODEL.exists(), "Final model checkpoint not found."
assert (MODEL / "adapter_model.safetensors").exists(), "Adapter weights missing."
assert TEST.exists(), "Holdout dataset not found."

counts = Counter()
examples = []
total = 0
missing_fields = Counter()

with TEST.open("r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        item = json.loads(line)
        total += 1

        for field in ["input", "category"]:
            if field not in item:
                missing_fields[field] += 1

        if "category" in item:
            counts[item["category"]] += 1

        if len(examples) < 3:
            examples.append(item)

print("Total rows:", total)
print("Missing fields:", dict(missing_fields))
print("\nCategory distribution:")
for category, count in counts.most_common():
    print(f"{category}: {count:,} ({count / total:.2%})")

print("\nExample record keys:", list(examples[0].keys()) if examples else [])
print("\nExample category:", examples[0].get("category") if examples else None)
print("Example input preview:")
print(str(examples[0].get("input", ""))[:500] if examples else "No examples")

print("\nHoldout inspection complete. No files modified.")

In [ ]:
import json
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

ROOT = Path("/kaggle/working/api_pentest_research")
CKPT = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
TEST = ROOT / "finetuning" / "test_holdout.jsonl"

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

with TEST.open("r", encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()][:3]

tokenizer = AutoTokenizer.from_pretrained(CKPT)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": 0},
)

print("Loading trained adapter...")
model = PeftModel.from_pretrained(base, str(CKPT))
model.eval()

print("\n=== INFERENCE SMOKE TEST ===")

for i, item in enumerate(examples, 1):
    messages = [
        {
            "role": "system",
            "content": (
                "You are Qwen, created by Alibaba Cloud. "
                "You are a helpful assistant."
            ),
        },
        {"role": "user", "content": item["input"]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=12,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(generated, skip_special_tokens=True).strip()

    print(f"\nExample {i}")
    print("Expected:", item["category"])
    print("Predicted:", repr(prediction))
    print("Exact match:", prediction == item["category"])

print("\nSmoke test complete. No training performed.")

In [ ]:
import json
import time
from pathlib import Path
from collections import Counter

import torch
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

ROOT = Path("/kaggle/working/api_pentest_research")
CKPT = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
TEST = ROOT / "finetuning" / "test_holdout.jsonl"
OUT = ROOT / "finetuning" / "holdout_predictions_step4500.jsonl"

LABELS = [
    "Benign",
    "Cookie Injection",
    "Directory Traversal",
    "LOG4J",
    "Log Forging",
    "RCE",
    "SQL Injection",
    "XSS",
]

BATCH_SIZE = 16
MAX_INPUT_LENGTH = 512
SAVE_EVERY_BATCHES = 25

assert CKPT.exists() and TEST.exists()

# Load examples and ground-truth labels.
examples = []
with TEST.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            examples.append(json.loads(line))

y_true = [item["category"] for item in examples]
assert all(label in LABELS for label in y_true), "Unexpected ground-truth label."

# Keep predictions separate from the test labels.
tokenizer = AutoTokenizer.from_pretrained(CKPT)
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=quant_config,
    device_map={"": 0},
)

model = PeftModel.from_pretrained(base, str(CKPT))
model.eval()

# Resume from a previous partial evaluation if available.
predictions = []
start_index = 0

if OUT.exists():
    with OUT.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                predictions.append(json.loads(line)["prediction"])

    if len(predictions) <= len(examples):
        start_index = len(predictions)
        print(f"Resuming evaluation at example {start_index:,}.")
    else:
        raise RuntimeError("Prediction file has more rows than the holdout set.")

def make_prompt(item):
    messages = [
        {
            "role": "system",
            "content": (
                "You are Qwen, created by Alibaba Cloud. "
                "You are a helpful assistant."
            ),
        },
        {"role": "user", "content": item["input"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def extract_label(text):
    text = text.strip()
    for label in LABELS:
        if text == label or text.startswith(label + "\n"):
            return label
    return "INVALID"

started = time.time()
batch_count = 0

with OUT.open("a", encoding="utf-8") as fout:
    for start in range(start_index, len(examples), BATCH_SIZE):
        batch = examples[start:start + BATCH_SIZE]
        prompts = [make_prompt(item) for item in batch]

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_INPUT_LENGTH,
        ).to("cuda:0")

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=12,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        new_tokens = output[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(
            new_tokens,
            skip_special_tokens=True,
        )

        for item, raw in zip(batch, decoded):
            pred = extract_label(raw)
            predictions.append(pred)
            fout.write(json.dumps({
                "prediction": pred,
                "source_dataset": item.get("source_dataset"),
            }) + "\n")

        batch_count += 1

        if batch_count % SAVE_EVERY_BATCHES == 0:
            fout.flush()
            done = len(predictions)
            elapsed = time.time() - started
            print(
                f"Progress: {done:,}/{len(examples):,} "
                f"({done / len(examples):.1%}) | "
                f"elapsed: {elapsed / 60:.1f} min"
            )

print("\n=== HOLDOUT RESULTS ===")
print("Examples:", len(y_true))
print("Predictions:", len(predictions))
print("Accuracy:", round(accuracy_score(y_true, predictions), 4))
print("Macro F1:", round(f1_score(
    y_true, predictions, labels=LABELS, average="macro", zero_division=0
), 4))
print("Weighted F1:", round(f1_score(
    y_true, predictions, labels=LABELS, average="weighted", zero_division=0
), 4))
print("\n=== PER-CATEGORY METRICS ===")
print(classification_report(
    y_true, predictions, labels=LABELS, zero_division=0, digits=4
))
print("=== CONFUSION MATRIX ===")
print("Label order:", LABELS)
print(confusion_matrix(y_true, predictions, labels=LABELS))
print("\nPredictions saved to:", OUT)
print("Total evaluation time: %.1f minutes" % ((time.time() - started) / 60))

In [ ]:
import json
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working/api_pentest_research")
TEST = ROOT / "finetuning" / "test_holdout.jsonl"
PRED = ROOT / "finetuning" / "holdout_predictions_step4500.jsonl"

with TEST.open(encoding="utf-8") as f:
    examples = [json.loads(line) for line in f if line.strip()]

with PRED.open(encoding="utf-8") as f:
    predictions = [json.loads(line) for line in f if line.strip()]

assert len(examples) == len(predictions), "Prediction count mismatch"

print("=== PREDICTION COUNT ===")
print("Holdout:", len(examples))
print("Predictions:", len(predictions))

print("\n=== SOURCE DATASET DISTRIBUTION ===")
print(Counter(item.get("source_dataset", "UNKNOWN") for item in examples))

labels = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

print("\n=== LABEL TEXT APPEARING IN INPUT ===")
for label in labels:
    count = sum(
        label.lower() in item["input"].lower()
        for item in examples
    )
    print(f"{label}: {count:,}/{len(examples):,}")

print("\n=== MISCLASSIFIED EXAMPLES ===")
errors = [
    (item, pred)
    for item, pred in zip(examples, predictions)
    if item["category"] != pred["prediction"]
]

print("Total errors:", len(errors))
for i, (item, pred) in enumerate(errors[:10], 1):
    print(f"\nError {i}")
    print("Expected:", item["category"])
    print("Predicted:", pred["prediction"])
    print("Source:", item.get("source_dataset"))
    print("Input preview:", item["input"][:700].replace("\n", " "))

print("\n=== DONE ===")

In [ ]:
import json
from pathlib import Path
from collections import Counter, defaultdict

ROOT = Path("/kaggle/working/api_pentest_research/finetuning")

TRAIN = ROOT / "train_final.jsonl"
TEST = ROOT / "test_holdout.jsonl"

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

train = load_jsonl(TRAIN)
test = load_jsonl(TEST)

def normalize(text):
    return " ".join(text.strip().split()).lower()

train_inputs = {normalize(x["input"]) for x in train}
test_inputs = [normalize(x["input"]) for x in test]

overlap = sum(x in train_inputs for x in test_inputs)

print("=== EXACT NORMALIZED INPUT OVERLAP ===")
print("Training rows:", len(train))
print("Holdout rows:", len(test))
print("Holdout inputs also present in training:", overlap)
print("Holdout overlap percentage:", round(100 * overlap / len(test), 4), "%")

print("\n=== CATEGORY DISTRIBUTION BY SOURCE ===")
counts = defaultdict(Counter)

for item in test:
    source = item.get("source_dataset", "UNKNOWN")
    category = item.get("category", "UNKNOWN")
    counts[source][category] += 1

for source, distribution in sorted(counts.items()):
    print(f"\n{source} (total={sum(distribution.values())})")
    for category, count in distribution.most_common():
        print(f"  {category}: {count}")

print("\n=== TRAINING CATEGORY COUNTS ===")
for category, count in Counter(x["category"] for x in train).most_common():
    print(f"{category}: {count}")

print("\n=== SAMPLE LOG4J TRAINING EXAMPLES ===")
shown = 0
for item in train:
    if item.get("category") == "LOG4J" and item.get("source_dataset") == "dataset_1_train":
        print("\nInput:", item["input"][:600].replace("\n", " "))
        shown += 1
        if shown == 3:
            break

print("\nAudit cell complete.")

In [ ]:
import json
from pathlib import Path
from collections import defaultdict
from sklearn.metrics import accuracy_score, f1_score, recall_score

ROOT = Path("/kaggle/working/api_pentest_research/finetuning")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test = load_jsonl(ROOT / "test_holdout.jsonl")
predictions = load_jsonl(ROOT / "holdout_predictions_step4500.jsonl")

assert len(test) == len(predictions)

labels = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

groups = defaultdict(list)

for actual, predicted in zip(test, predictions):
    source = actual.get("source_dataset", "UNKNOWN")
    groups[source].append((actual["category"], predicted["prediction"]))

print("=== PERFORMANCE BY SOURCE DATASET ===")

for source, rows in sorted(groups.items()):
    y_true = [x[0] for x in rows]
    y_pred = [x[1] for x in rows]

    print(f"\n{source} | samples={len(rows)}")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Macro F1:  {f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0):.4f}")

    log4j_indices = [i for i, label in enumerate(y_true) if label == "LOG4J"]

    if log4j_indices:
        log4j_recall = recall_score(
            [y_true[i] for i in log4j_indices],
            [y_pred[i] for i in log4j_indices],
            labels=["LOG4J"],
            average="macro",
            zero_division=0
        )
        print(f"LOG4J recall: {log4j_recall:.4f} ({sum(y_pred[i] == 'LOG4J' for i in log4j_indices)}/{len(log4j_indices)})")

    errors = sum(a != p for a, p in rows)
    print(f"Misclassified: {errors}")

print("\n=== SOURCE EVALUATION COMPLETE ===")

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working/api_pentest_research/finetuning")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test = load_jsonl(ROOT / "test_holdout.jsonl")
predictions = load_jsonl(ROOT / "holdout_predictions_step4500.jsonl")

def request_part(text):
    marker = '{"body":'
    position = text.find(marker)
    return text[position:] if position >= 0 else text

patterns = {
    "jndi": r"jndi",
    "ldap": r"ldap",
    "rmi": r"rmi",
    "dns": r"dns",
    "lowercase ${": r"\$\{",
    "upper/lower lookup": r"lookup",
    "java class reference": r"java(?:x)?\.",
    "URL encoded marker": r"%24|%7b|%7d",
}

groups = {
    "LOG4J correctly detected": [],
    "LOG4J missed": [],
}

for actual, predicted in zip(test, predictions):
    if actual["category"] != "LOG4J":
        continue

    request = request_part(actual["input"]).lower()

    if predicted["prediction"] == "LOG4J":
        groups["LOG4J correctly detected"].append(request)
    else:
        groups["LOG4J missed"].append(request)

print("=== LOG4J INDICATOR COMPARISON ===")

for group_name, requests in groups.items():
    print(f"\n{group_name}: {len(requests)} examples")

    for name, pattern in patterns.items():
        count = sum(bool(re.search(pattern, request)) for request in requests)
        percentage = 100 * count / len(requests) if requests else 0
        print(f"  {name}: {count}/{len(requests)} ({percentage:.1f}%)")

print("\n=== SAMPLE MISSED LOG4J REQUESTS ===")
for i, request in enumerate(groups["LOG4J missed"][:5], 1):
    print(f"\nExample {i}:")
    print(request[:900])

print("\n=== SAMPLE CORRECTLY DETECTED LOG4J REQUESTS ===")
for i, request in enumerate(groups["LOG4J correctly detected"][:3], 1):
    print(f"\nExample {i}:")
    print(request[:900])

print("\nIndicator audit complete.")

In [ ]:
import json
import re
from pathlib import Path
from collections import defaultdict

ROOT = Path("/kaggle/working/api_pentest_research/finetuning")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

data = load_jsonl(ROOT / "train_final.jsonl")
data += load_jsonl(ROOT / "test_holdout.jsonl")

patterns = {
    "jndi": r"jndi",
    "ldap": r"ldap",
    "${ marker": r"\$\{",
    "rmi": r"rmi",
    "log4j text": r"log4j",
}

stats = defaultdict(lambda: {"total": 0, **{k: 0 for k in patterns}})

for item in data:
    source = item.get("source_dataset", "UNKNOWN")
    category = item.get("category", "UNKNOWN")
    key = (source, category)
    stats[key]["total"] += 1

    text = item["input"].lower()
    marker = text.find('{"body":')
    request = text[marker:] if marker >= 0 else text

    for name, pattern in patterns.items():
        if re.search(pattern, request):
            stats[key][name] += 1

print("=== LOG4J INDICATORS BY SOURCE ===")

for (source, category), values in sorted(stats.items()):
    if category != "LOG4J":
        continue

    total = values["total"]
    print(f"\n{source} | LOG4J examples: {total}")

    for name in patterns:
        count = values[name]
        print(f"  {name}: {count}/{total} ({100 * count / total:.1f}%)")

print("\n=== COMPARISON: OTHER CATEGORIES ===")

for (source, category), values in sorted(stats.items()):
    if category in ("LOG4J", "Benign"):
        continue

    total = values["total"]
    marker_count = values["jndi"]
    if total:
        print(
            f"{source} | {category}: "
            f"JNDI indicator {marker_count}/{total} "
            f"({100 * marker_count / total:.1f}%)"
        )

print("\nLabel audit complete.")

In [ ]:
import json
import re
from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix

ROOT = Path("/kaggle/working/api_pentest_research/finetuning")

def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

test = load_jsonl(ROOT / "test_holdout.jsonl")

y_true = []
y_pred = []

for item in test:
    text = item["input"].lower()
    marker = text.find('{"body":')
    request = text[marker:] if marker >= 0 else text

    has_jndi = bool(re.search(r"jndi|ldap|\$\{", request))

    y_true.append(item["category"])
    y_pred.append("LOG4J" if has_jndi else "Benign")

print("=== SIMPLE INDICATOR BASELINE ===")
print(classification_report(y_true, y_pred, zero_division=0))

print("=== LOG4J VS OTHER LABELS ===")
print("Rows evaluated:", len(test))
print("True LOG4J:", sum(label == "LOG4J" for label in y_true))
print("Predicted LOG4J:", sum(label == "LOG4J" for label in y_pred))
print("Confusion matrix [Benign, LOG4J]:")
print(confusion_matrix(
    ["LOG4J" if x == "LOG4J" else "Benign" for x in y_true],
    ["LOG4J" if x == "LOG4J" else "Benign" for x in y_pred],
    labels=["Benign", "LOG4J"]
))

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")

print("=== PROJECT FILES RELATED TO DATASETS ===")

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue

    name = path.name.lower()

    if any(word in name for word in [
        "readme", "metadata", "manifest", "dataset",
        "label", "log4j", "source", "data_card"
    ]):
        try:
            size_mb = path.stat().st_size / (1024 * 1024)
            print(f"{path.relative_to(ROOT)} | {size_mb:.2f} MB")
        except OSError:
            pass

print("\n=== TOP-LEVEL PROJECT DIRECTORIES ===")
for path in sorted(ROOT.iterdir()):
    if path.is_dir():
        print(path.name)

print("\nFile inventory complete.")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
DATA = ROOT / "ATRDF" / "extracted"

print("=== ORIGINAL VALIDATION DATASETS ===")

for i in range(1, 5):
    folder = DATA / f"dataset_{i}_val"
    files = list(folder.glob("*")) if folder.exists() else []

    print(f"\nDataset {i}")
    print("Folder exists:", folder.exists())

    for file in files[:10]:
        if file.is_file():
            print(f"  {file.name} | {file.stat().st_size / (1024 * 1024):.2f} MB")

print("\n=== CURRENT SPLIT FILES ===")

for name in ["train_final.jsonl", "validation_sft.jsonl", "test_holdout.jsonl"]:
    path = ROOT / "finetuning" / name
    print(f"{name}: exists={path.exists()}", end="")

    if path.exists():
        print(f", size={path.stat().st_size / (1024 * 1024):.2f} MB")
    else:
        print()

print("\nInspection complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working/api_pentest_research")
DATA = ROOT / "ATRDF" / "extracted"

for i in range(1, 5):
    path = DATA / f"dataset_{i}_val" / f"dataset_{i}_val.json"

    print(f"\n{'=' * 20} DATASET {i} {'=' * 20}")

    with path.open("r", encoding="utf-8") as f:
        first = f.read(1)
        f.seek(0)

        if first == "[":
            records = json.load(f)
        elif first == "{":
            obj = json.load(f)
            records = obj if isinstance(obj, list) else obj.get("data", [])
        else:
            records = [json.loads(line) for line in f if line.strip()]

    print("Record count:", len(records))

    if records:
        print("First record keys:", list(records[0].keys()))
        print("First record sample:")
        print(json.dumps(records[0], indent=2, ensure_ascii=False)[:1200])

        for key in ["category", "label", "attack_type", "class", "type"]:
            values = [str(r.get(key)) for r in records if isinstance(r, dict) and key in r]
            if values:
                print(f"Counts for '{key}':", Counter(values).most_common(12))
                break

print("\nValidation format inspection complete.")

In [ ]:
import json
from pathlib import Path

DATA = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for i in range(1, 5):
    path = DATA / f"dataset_{i}_val" / f"dataset_{i}_val.json"

    print(f"\n{'=' * 18} DATASET {i} {'=' * 18}")

    with path.open("r", encoding="utf-8") as f:
        first_record = next(
            item for item in json.JSONDecoder().raw_decode(f.read(2_000_000))[0:1]
        ) if False else None

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f) if i == 1 else None

    if records is not None and records:
        sample = records[0]
    else:
        with path.open("r", encoding="utf-8") as f:
            decoder = json.JSONDecoder()
            chunk = f.read(2_000_000)
            start = chunk.find("[")
            sample, _ = decoder.raw_decode(chunk, start + 1)

    print("Top-level keys:", list(sample.keys()))
    print("Response keys:", list(sample.get("response", {}).keys()))
    print("Response sample:")
    print(json.dumps(sample.get("response", {}), indent=2)[:1500])

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")

print("=== LABEL / METADATA FILES ===")

for p in ROOT.rglob("*"):
    if not p.is_file():
        continue

    name = p.name.lower()

    if any(term in name for term in [
        "label", "ground_truth", "groundtruth", "metadata",
        "annotation", "mapping", "class", "readme", "manifest"
    ]):
        print(f"{p}  ({p.stat().st_size:,} bytes)")

print("\n=== DATASET DIRECTORY CONTENTS ===")

for i in range(1, 5):
    folder = ROOT / "ATRDF" / "extracted" / f"dataset_{i}_val"
    print(f"\n{folder}")

    if folder.exists():
        for p in sorted(folder.iterdir()):
            kind = "DIR " if p.is_dir() else "FILE"
            size = f"{p.stat().st_size:,} bytes" if p.is_file() else ""
            print(f"  {kind} {p.name} {size}")
    else:
        print("  Directory not found")

print("\nInspection complete.")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
ATRDF = ROOT / "ATRDF"

readme = ATRDF / "README.md"

print("=== ATRDF README: LABELING AND DATASET INFORMATION ===")

if readme.exists():
    lines = readme.read_text(encoding="utf-8", errors="replace").splitlines()

    keywords = [
        "label", "class", "dataset", "validation",
        "ground truth", "annotation", "attack", "benign"
    ]

    matches = [
        i for i, line in enumerate(lines)
        if any(word in line.lower() for word in keywords)
    ]

    shown = set()

    for i in matches:
        for j in range(max(0, i - 2), min(len(lines), i + 3)):
            if j not in shown:
                print(f"{j + 1}: {lines[j]}")
                shown.add(j)
        print("---")
else:
    print("ATRDF README not found")

print("\n=== FILES NEAR THE DATASET FOLDERS ===")

for folder in [
    ATRDF,
    ATRDF / "extracted",
    ATRDF / "extracted" / "dataset_1_val",
    ATRDF / "extracted" / "dataset_2_val",
    ATRDF / "extracted" / "dataset_3_val",
    ATRDF / "extracted" / "dataset_4_val",
]:
    print(f"\n{folder}")

    if folder.is_dir():
        for p in sorted(folder.iterdir()):
            if p.is_file():
                print(f"  FILE: {p.name} ({p.stat().st_size:,} bytes)")
            elif p.is_dir():
                print(f"  DIR:  {p.name}")
    else:
        print("  Directory not found")

In [ ]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for i in range(1, 5):
    path = ROOT / f"dataset_{i}_train" / f"dataset_{i}_train.json"

    print(f"\n{'=' * 18} DATASET {i} TRAIN {'=' * 18}")

    if not path.exists():
        print("Training JSON not found:", path)
        folder = path.parent
        if folder.exists():
            print("Available files:", [p.name for p in folder.iterdir()])
        continue

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    print("Record count:", len(records))

    if records:
        sample = records[0]
        print("Top-level keys:", list(sample.keys()))
        print("First record:")
        print(json.dumps(sample, indent=2, ensure_ascii=False)[:1800])

print("\nInspection complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

ROOT = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

def inspect_json_array(path):
    decoder = json.JSONDecoder()

    with path.open("r", encoding="utf-8") as f:
        buffer = ""
        started = False
        eof = False

        while True:
            if not eof and len(buffer) < 65536:
                chunk = f.read(1024 * 1024)
                if chunk:
                    buffer += chunk
                else:
                    eof = True

            if not started:
                pos = buffer.find("[")
                if pos == -1:
                    if eof:
                        return
                    buffer = buffer[-10:]
                    continue
                buffer = buffer[pos + 1:]
                started = True

            buffer = buffer.lstrip()

            if buffer.startswith("]"):
                return

            if buffer.startswith(","):
                buffer = buffer[1:].lstrip()

            try:
                obj, end = decoder.raw_decode(buffer)
            except json.JSONDecodeError:
                if eof:
                    raise
                chunk = f.read(1024 * 1024)
                if chunk:
                    buffer += chunk
                else:
                    eof = True
                continue

            yield obj
            buffer = buffer[end:]

for i in range(1, 5):
    path = ROOT / f"dataset_{i}_train" / f"dataset_{i}_train.json"

    print(f"\n{'=' * 15} DATASET {i} TRAIN {'=' * 15}")

    total = 0
    tagged = 0
    labels = Counter()
    missing_examples = []

    for record in inspect_json_array(path):
        total += 1
        request = record.get("request", {})
        label = request.get("Attack_Tag")

        if label is not None and str(label).strip():
            tagged += 1
            labels[str(label).strip()] += 1
        elif len(missing_examples) < 2:
            missing_examples.append({
                "method": request.get("method"),
                "url": request.get("url"),
                "body": request.get("body")
            })

    print("Total records:", total)
    print("Records with Attack_Tag:", tagged)
    print("Records without Attack_Tag:", total - tagged)
    print("Label counts:", dict(labels))
    print("Examples without labels:", json.dumps(
        missing_examples, indent=2, ensure_ascii=False
    )[:1000])

print("\nLabel audit complete.")

In [ ]:
import json
from pathlib import Path

path = Path(
    "/kaggle/working/api_pentest_research/ATRDF/baseline_model.ipynb"
)

with path.open("r", encoding="utf-8") as f:
    notebook = json.load(f)

keywords = [
    "Attack_Tag",
    "label",
    "Malware",
    "Benign",
    "attack_type",
    "y_train",
    "classification"
]

for i, cell in enumerate(notebook.get("cells", [])):
    source = "".join(cell.get("source", []))

    if any(word.lower() in source.lower() for word in keywords):
        print(f"\n{'=' * 20} CELL {i} ({cell.get('cell_type')}) {'=' * 20}")
        print(source[:3500])

print("\nBaseline labeling inspection complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research")
finetune = root / "finetuning"

holdout_path = finetune / "test_holdout.jsonl"
pred_path = finetune / "holdout_predictions_step4500.jsonl"

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

holdout = load_jsonl(holdout_path)
predictions = load_jsonl(pred_path)

print("Holdout rows:", len(holdout))
print("Prediction rows:", len(predictions))

if len(holdout) != len(predictions):
    raise ValueError("Holdout and prediction row counts do not match.")

missed = []

for i, (sample, pred) in enumerate(zip(holdout, predictions)):
    actual = sample.get("label")
    predicted = pred.get("prediction")

    if actual == "LOG4J" and predicted == "Benign":
        missed.append((i, sample, pred))

print("\nMissed LOG4J samples:", len(missed))

indicators = [
    "jndi", "ldap", "rmi", "dns", "${",
    "javax.naming", "log4j", "lookup"
]

indicator_counts = Counter()
tag_counts = Counter()

for _, sample, pred in missed:
    request = sample.get("request", {})
    request_text = json.dumps(request, ensure_ascii=False).lower()

    for indicator in indicators:
        if indicator in request_text:
            indicator_counts[indicator] += 1

    tag = request.get("Attack_Tag", "<missing>")
    tag_counts[str(tag)] += 1

print("\nIndicator counts among missed samples:")
for indicator, count in indicator_counts.most_common():
    print(f"{indicator}: {count}")

print("\nAttack_Tag counts among missed samples:")
for tag, count in tag_counts.most_common():
    print(f"{tag}: {count}")

print("\nFirst 10 missed examples:")
for idx, sample, pred in missed[:10]:
    request = sample.get("request", {})
    print(f"\nHoldout index: {idx}")
    print("Source:", pred.get("source_dataset"))
    print("Original label:", sample.get("label"))
    print("Prediction:", pred.get("prediction"))
    print("Attack_Tag:", request.get("Attack_Tag", "<missing>"))
    print("Method:", request.get("method"))
    print("URL:", str(request.get("url", ""))[:500])
    print("Body:", str(request.get("body", ""))[:500])
    print("Headers:", json.dumps(
        request.get("headers", {}),
        ensure_ascii=False
    )[:700])

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/finetuning")

for filename in [
    "test_holdout.jsonl",
    "holdout_predictions_step4500.jsonl"
]:
    path = root / filename

    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    with path.open("r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if line.strip():
                record = json.loads(line)
                print("Available keys:", list(record.keys()))
                print(json.dumps(record, indent=2, ensure_ascii=False)[:2500])
                break

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research/finetuning")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

holdout = load_jsonl(root / "test_holdout.jsonl")
predictions = load_jsonl(root / "holdout_predictions_step4500.jsonl")

missed = []

for i, (sample, pred) in enumerate(zip(holdout, predictions)):
    if sample["category"] == "LOG4J" and pred["prediction"] == "Benign":
        missed.append((i, sample, pred))

print("Total holdout samples:", len(holdout))
print("Missed LOG4J samples:", len(missed))

indicators = [
    "jndi", "ldap", "rmi", "dns", "${",
    "javax.naming", "log4j", "lookup"
]

indicator_counts = Counter()
source_counts = Counter()

for _, sample, pred in missed:
    request_text = sample["input"].lower()

    for indicator in indicators:
        if indicator in request_text:
            indicator_counts[indicator] += 1

    source_counts[sample["source_dataset"]] += 1

print("\nMissed samples by source:")
for source, count in source_counts.most_common():
    print(source, count)

print("\nIndicators in missed samples:")
for indicator, count in indicator_counts.most_common():
    print(indicator, count)

print("\nFirst 10 missed examples:")

for idx, sample, pred in missed[:10]:
    print(f"\nHoldout index: {idx}")
    print("True category:", sample["category"])
    print("Predicted category:", pred["prediction"])
    print("Source:", sample["source_dataset"])
    print("Request:", sample["input"][-1200:])

print("\nInspection complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research/finetuning")

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

holdout = load_jsonl(root / "test_holdout.jsonl")
predictions = load_jsonl(root / "holdout_predictions_step4500.jsonl")

indicators = [
    "log4j", "jndi", "ldap", "rmi", "dns",
    "${", "javax.naming", "lookup"
]

groups = {
    "Missed LOG4J": [],
    "Correct LOG4J": []
}

for sample, pred in zip(holdout, predictions):
    if sample["category"] != "LOG4J":
        continue

    if pred["prediction"] == "Benign":
        groups["Missed LOG4J"].append(sample)
    elif pred["prediction"] == "LOG4J":
        groups["Correct LOG4J"].append(sample)

for group_name, samples in groups.items():
    print(f"\n{'=' * 18} {group_name} {'=' * 18}")
    print("Samples:", len(samples))

    counts = Counter()

    for sample in samples:
        text = sample["input"].lower()

        # Extract the serialized HTTP request, excluding the instruction.
        request_start = text.find("\n{")
        request_text = text[request_start:] if request_start != -1 else text

        for indicator in indicators:
            if indicator in request_text:
                counts[indicator] += 1

    for indicator in indicators:
        count = counts[indicator]
        percentage = 100 * count / len(samples) if samples else 0
        print(f"{indicator:15} {count:5} ({percentage:.2f}%)")

print("\nComparison complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

indicators = ["jndi", "ldap", "${", "log4j", "rmi", "dns"]

def load_records(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

for dataset_num in range(1, 5):
    path = root / f"dataset_{dataset_num}_train" / f"dataset_{dataset_num}_train.json"

    if not path.exists():
        print("Missing file:", path)
        continue

    records = load_records(path)
    groups = {"LOG4J": [], "Benign": []}

    for record in records:
        request = record.get("request", {})
        tag = request.get("Attack_Tag")

        if tag == "LOG4J":
            groups["LOG4J"].append(record)
        elif not tag or tag == "Benign":
            groups["Benign"].append(record)

    print(f"\n{'=' * 15} DATASET {dataset_num} {'=' * 15}")

    for group_name, samples in groups.items():
        counts = Counter()

        for record in samples:
            request_text = json.dumps(
                record.get("request", {}),
                ensure_ascii=False
            ).lower()

            for indicator in indicators:
                if indicator in request_text:
                    counts[indicator] += 1

        print(f"\n{group_name}: {len(samples)} records")

        for indicator in indicators:
            count = counts[indicator]
            percentage = 100 * count / len(samples) if samples else 0
            print(f"{indicator:10} {count:6} ({percentage:.2f}%)")

        if group_name == "LOG4J":
            print("Examples:")
            for record in samples[:2]:
                request = record.get("request", {})
                print("Method:", request.get("method"))
                print("URL:", str(request.get("url", ""))[:300])
                print("Body:", str(request.get("body", ""))[:300])

print("\nOriginal training-label audit complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research")
data_root = root / "ATRDF" / "extracted"
finetune = root / "finetuning"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

holdout = load_jsonl(finetune / "test_holdout.jsonl")
predictions = load_jsonl(finetune / "holdout_predictions_step4500.jsonl")

indicators = ["jndi", "ldap", "${", "log4j"]
missed_by_source = {}

for sample, pred in zip(holdout, predictions):
    if sample["category"] == "LOG4J" and pred["prediction"] == "Benign":
        missed_by_source.setdefault(sample["source_dataset"], []).append(sample)

def parse_request(sample):
    text = sample["input"]
    start = text.find("\n{")
    if start == -1:
        return {}
    try:
        return json.loads(text[start + 1:])
    except Exception:
        return {}

for source, samples in missed_by_source.items():
    filename = source + ".json"
    path = data_root / source / filename

    print(f"\n{'=' * 18} {source} {'=' * 18}")

    if not path.exists():
        print("Original file not found:", path)
        continue

    with open(path, "r", encoding="utf-8") as f:
        original_records = json.load(f)

    # Build lookup by method and URL.
    lookup = {}
    for record in original_records:
        req = record.get("request", {})
        key = (req.get("method"), req.get("url"))
        lookup.setdefault(key, []).append(record)

    checked = 0
    matched = 0

    for sample in samples[:5]:
        parsed = parse_request(sample)
        key = (parsed.get("method"), parsed.get("url"))
        matches = lookup.get(key, [])
        checked += 1

        print("\nMethod:", key[0])
        print("URL:", str(key[1])[:250])

        if not matches:
            print("No exact original record match.")
            continue

        matched += 1
        record = matches[0]
        print("Original Attack_Tag:",
              record.get("request", {}).get("Attack_Tag"))

        for section_name, section in record.items():
            serialized = json.dumps(section, ensure_ascii=False).lower()
            found = [x for x in indicators if x in serialized]
            if found:
                print(f"Indicators in {section_name}: {found}")

        original_request = record.get("request", {})
        for field, value in original_request.items():
            value_text = json.dumps(value, ensure_ascii=False).lower()
            found = [x for x in indicators if x in value_text]
            if found:
                print(f"  request.{field}: {found}")

    print(f"\nMatched {matched} of {checked} sampled records.")

print("\nOriginal-record comparison complete.")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
data_root = root / "ATRDF" / "extracted"
finetune = root / "finetuning"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

holdout = load_jsonl(finetune / "test_holdout.jsonl")
predictions = load_jsonl(finetune / "holdout_predictions_step4500.jsonl")

indicators = ["jndi", "ldap", "${", "log4j"]

missed = {}
for sample, pred in zip(holdout, predictions):
    if sample["category"] == "LOG4J" and pred["prediction"] == "Benign":
        missed.setdefault(sample["source_dataset"], (sample, pred))

for source, (sample, pred) in missed.items():
    path = data_root / source / f"{source}.json"

    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    text = sample["input"]
    start = text.find("\n{")
    parsed = json.loads(text[start + 1:]) if start != -1 else {}
    key = (parsed.get("method"), parsed.get("url"))

    matches = [
        record for record in records
        if (
            record.get("request", {}).get("method"),
            record.get("request", {}).get("url")
        ) == key
        and record.get("request", {}).get("Attack_Tag") == "LOG4J"
    ]

    print(f"\n{'=' * 15} {source} {'=' * 15}")
    print("Holdout category:", sample["category"])
    print("Prediction:", pred["prediction"])
    print("Method:", key[0])
    print("URL:", key[1])
    print("Matching original LOG4J records:", len(matches))

    if matches:
        original = matches[0]["request"]
        original_headers = json.dumps(
            original.get("headers", {}),
            ensure_ascii=False
        ).lower()

        model_request = text[start + 1:].lower() if start != -1 else ""

        print("Original header indicators:")
        for indicator in indicators:
            print(indicator, indicator in original_headers)

        print("Model-input indicators:")
        for indicator in indicators:
            print(indicator, indicator in model_request)

        print("Original headers:", json.dumps(
            original.get("headers", {}),
            ensure_ascii=False
        )[:1500])

print("\nComparison complete.")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
finetune = root / "finetuning"

files = [
    finetune / "train_final.jsonl",
    finetune / "test_holdout.jsonl"
]

for path in files:
    print(f"\n{'=' * 18} {path.name} {'=' * 18}")

    with path.open("r", encoding="utf-8") as f:
        found = 0

        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)
            text = row.get("input", "")

            if row.get("category") == "LOG4J":
                start = text.find("\n{")
                if start == -1:
                    continue

                try:
                    request = json.loads(text[start + 1:])
                except Exception:
                    continue

                headers = request.get("headers", {})
                cookie_text = str(headers.get("Set-Cookie", ""))

                print("Category:", row.get("category"))
                print("Source:", row.get("source_dataset"))
                print("Request keys:", list(request.keys()))
                print("Header keys:", list(headers.keys()))
                print("Set-Cookie present:", bool(cookie_text))
                print("JNDI in Set-Cookie:",
                      "jndi" in cookie_text.lower())
                print("LDAP in Set-Cookie:",
                      "ldap" in cookie_text.lower())
                print("Payload sample:", cookie_text[-250:])
                found += 1

                if found >= 3:
                    break

        print("Examples inspected:", found)

print("\nInput representation inspection complete.")

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")

keywords = [
    "train_final.jsonl",
    "test_holdout.jsonl",
    "Set-Cookie",
    "headers",
    "validation_sft.jsonl",
    "jsonl",
    "Attack_Tag",
]

for path in sorted(root.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in {
        ".py", ".ipynb", ".md", ".txt"
    }:
        continue

    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        content = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    matching_lines = [
        (i, line.strip())
        for i, line in enumerate(content.splitlines(), start=1)
        if any(word.lower() in line.lower() for word in keywords)
    ]

    if matching_lines:
        print(f"\n{'=' * 15} {path.relative_to(root)} {'=' * 15}")
        for line_no, line in matching_lines[:20]:
            print(f"{line_no}: {line[:250]}")

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")

for folder in [root / "finetuning", root]:
    print(f"\n=== {folder.relative_to(root.parent)} ===")

    for path in sorted(folder.iterdir()):
        if path.is_file() and path.suffix.lower() in {
            ".py", ".ipynb", ".md", ".txt"
        }:
            print(path.name)

In [ ]:
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")

keywords = [
    "train_final.jsonl",
    "test_holdout.jsonl",
    "validation_sft.jsonl",
    "json.dumps(request",
    "json.dumps(req",
    "jsonl.write",
    "json.dump",
    "request.headers",
    "Set-Cookie",
]

extensions = {".py", ".ipynb", ".sh"}

for path in sorted(root.rglob("*")):
    if not path.is_file() or path.suffix.lower() not in extensions:
        continue

    if any(part.startswith(".") for part in path.parts):
        continue

    try:
        content = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        continue

    lines = content.splitlines()
    matches = [
        (i, line.strip())
        for i, line in enumerate(lines, start=1)
        if any(keyword.lower() in line.lower() for keyword in keywords)
    ]

    if matches:
        print(f"\n=== {path.relative_to(root)} ===")
        for line_no, line in matches[:15]:
            print(f"{line_no}: {line[:220]}")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research/finetuning")

files = {
    "train": root / "train_final.jsonl",
    "holdout": root / "test_holdout.jsonl",
}

for name, path in files.items():
    counts = Counter()
    examples = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue

            row = json.loads(line)
            if row.get("category") != "LOG4J":
                continue

            counts["LOG4J total"] += 1

            text = row.get("input", "")
            start = text.find("\n{")

            if start == -1:
                counts["Unparseable input"] += 1
                continue

            try:
                request = json.loads(text[start + 1:])
            except Exception:
                counts["Unparseable input"] += 1
                continue

            headers = request.get("headers", {})
            all_request_text = json.dumps(request, ensure_ascii=False).lower()

            if "set-cookie" in {key.lower() for key in headers}:
                counts["Set-Cookie present"] += 1

            if any(x in all_request_text for x in ["jndi", "ldap", "${"]):
                counts["Payload indicator present"] += 1
            elif len(examples) < 3:
                examples.append({
                    "source": row.get("source_dataset"),
                    "url": request.get("url"),
                    "headers": list(headers.keys()),
                })

    print(f"\n=== {name.upper()} ===")
    for key, value in counts.items():
        print(f"{key}: {value}")

    print("Examples without payload indicators:")
    for example in examples:
        print(example)

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
data_root = root / "ATRDF" / "extracted"
finetune = root / "finetuning"

holdout_path = finetune / "test_holdout.jsonl"

with holdout_path.open("r", encoding="utf-8") as f:
    sample = next(
        row for row in (json.loads(line) for line in f if line.strip())
        if row.get("category") == "LOG4J"
        and row.get("source_dataset") == "dataset_1_train"
    )

text = sample["input"]
start = text.find("\n{")
request = json.loads(text[start + 1:])

source = sample["source_dataset"]
original_path = (
    data_root / source / f"{source}.json"
)

with original_path.open("r", encoding="utf-8") as f:
    original_records = json.load(f)

matches = [
    record for record in original_records
    if record.get("request", {}).get("method") == request.get("method")
    and record.get("request", {}).get("url") == request.get("url")
    and record.get("request", {}).get("Attack_Tag") == "LOG4J"
]

print("Source:", source)
print("Method:", request.get("method"))
print("URL:", request.get("url"))
print("Prepared header keys:", sorted(request.get("headers", {}).keys()))
print("Matching original records:", len(matches))

if matches:
    original = matches[0].get("request", {})
    original_headers = original.get("headers", {})

    print("\nOriginal header keys:", sorted(original_headers.keys()))
    print("Set-Cookie in original:", "Set-Cookie" in original_headers)
    print("Set-Cookie in prepared input:",
          "Set-Cookie" in request.get("headers", {}))

    print("\nOriginal Set-Cookie payload:")
    print(str(original_headers.get("Set-Cookie", ""))[-500:])

print("\nComparison complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

indicators = ["jndi", "ldap", "${"]
totals = Counter()

for dataset_num in range(1, 5):
    source = f"dataset_{dataset_num}_train"
    path = root / source / f"{source}.json"

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    for record in records:
        request = record.get("request", {})

        if request.get("Attack_Tag") != "LOG4J":
            continue

        headers = request.get("headers", {})
        cookie = str(headers.get("Set-Cookie", "")).lower()

        other_fields = {
            key: value
            for key, value in request.items()
            if key not in {"headers", "Attack_Tag"}
        }
        other_text = json.dumps(other_fields, ensure_ascii=False).lower()

        totals["LOG4J total"] += 1

        if any(x in cookie for x in indicators):
            totals["Payload in Set-Cookie"] += 1
        elif any(
            x in json.dumps(value, ensure_ascii=False).lower()
            for key, value in headers.items()
            if key != "Set-Cookie"
            for x in indicators
        ):
            totals["Payload in other headers"] += 1
        elif any(x in other_text for x in indicators):
            totals["Payload in other request fields"] += 1
        else:
            totals["No payload indicator in request fields"] += 1

print("Original training-data audit:")
for key, value in totals.items():
    print(f"{key}: {value}")

print("\nAudit complete.")

In [ ]:

import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research")
data_dir = root / "finetuning"
source_dir = root / "ATRDF" / "extracted"

files = {
    "train": data_dir / "train_final.jsonl",
    "validation": data_dir / "validation_sft.jsonl",
    "holdout": data_dir / "test_holdout.jsonl",
}

for split, path in files.items():
    print(f"\n{'=' * 18} {split.upper()} {'=' * 18}")

    if not path.exists():
        print("MISSING:", path)
        continue

    with path.open("r", encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]

    print("Rows:", len(rows))
    print("Fields:", list(rows[0].keys()) if rows else [])

    if rows:
        print("Source counts:", dict(Counter(
            row.get("source_dataset", "MISSING")
            for row in rows
        )))

        print("Category counts:", dict(Counter(
            row.get("category", "MISSING")
            for row in rows
        )))

        print("First input preview:")
        print(rows[0].get("input", "")[-500:])

print("\nOriginal source files:")
for i in range(1, 5):
    source = source_dir / f"dataset_{i}_train" / f"dataset_{i}_train.json"
    print(f"Dataset {i}: {'FOUND' if source.exists() else 'MISSING'}")

print("\nPre-rebuild inspection complete.")


In [ ]:

import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research")
data_root = root / "ATRDF" / "extracted"
data_dir = root / "finetuning"

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def parse_input(row):
    text = row["input"]
    start = text.find("\n{")
    if start == -1:
        return None

    try:
        return json.loads(text[start + 1:])
    except Exception:
        return None

cache = {}

def load_source(source):
    if source not in cache:
        path = data_root / source / f"{source}.json"
        with path.open("r", encoding="utf-8") as f:
            cache[source] = json.load(f)
    return cache[source]

for split in ["train", "validation", "holdout"]:
    path = data_dir / {
        "train": "train_final.jsonl",
        "validation": "validation_sft.jsonl",
        "holdout": "test_holdout.jsonl",
    }[split]

    rows = load_jsonl(path)
    counts = Counter()

    print(f"\n=== {split.upper()} ===")

    for row in rows:
        if row.get("category") != "LOG4J":
            continue

        request = parse_input(row)
        if request is None:
            counts["parse_error"] += 1
            continue

        source = row["source_dataset"]
        originals = load_source(source)

        method = request.get("method")
        url = request.get("url")

        candidates = [
            item for item in originals
            if item.get("request", {}).get("method") == method
            and item.get("request", {}).get("url") == url
            and item.get("request", {}).get("Attack_Tag") == "LOG4J"
        ]

        if not candidates:
            counts["no_matching_LOG4J_record"] += 1
            continue

        counts["matched"] += 1

        prepared_headers = {
            k.lower() for k in request.get("headers", {})
        }

        original_headers = {
            k.lower()
            for k in candidates[0].get("request", {}).get("headers", {})
        }

        if "set-cookie" in original_headers and "set-cookie" not in prepared_headers:
            counts["Set-Cookie_omitted"] += 1

        if len(candidates) > 1:
            counts["ambiguous_URL_and_method_match"] += 1

    print("LOG4J records:", sum(
        1 for row in rows if row.get("category") == "LOG4J"
    ))
    for key, value in counts.items():
        print(f"{key}: {value}")

print("\nMatching audit complete. No files were modified.")


In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research")
data_root = root / "ATRDF" / "extracted"
data_dir = root / "finetuning"

def load_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def parse_input(row):
    text = row["input"]
    start = text.find("\n{")
    return json.loads(text[start + 1:]) if start >= 0 else None

cache = {}

def load_source(source):
    if source not in cache:
        path = data_root / source / f"{source}.json"
        with path.open("r", encoding="utf-8") as f:
            cache[source] = json.load(f)
    return cache[source]

for split, filename in [
    ("train", "train_final.jsonl"),
    ("validation", "validation_sft.jsonl"),
    ("holdout", "test_holdout.jsonl"),
]:
    rows = load_jsonl(data_dir / filename)
    stats = Counter()

    for row in rows:
        if row.get("category") != "LOG4J":
            continue

        request = parse_input(row)
        if request is None:
            stats["parse_error"] += 1
            continue

        originals = load_source(row["source_dataset"])
        candidates = [
            item for item in originals
            if item.get("request", {}).get("method") == request.get("method")
            and item.get("request", {}).get("url") == request.get("url")
            and item.get("request", {}).get("Attack_Tag") == "LOG4J"
        ]

        if not candidates:
            stats["no_match"] += 1
            continue

        cookies = {
            json.dumps(
                item.get("request", {}).get("headers", {}).get("Set-Cookie"),
                sort_keys=True
            )
            for item in candidates
        }

        stats["matched"] += 1

        if len(candidates) > 1:
            stats["duplicate_matches"] += 1

        if len(cookies) > 1:
            stats["different_cookie_values"] += 1

    print(f"\n=== {split.upper()} ===")
    for key, value in stats.items():
        print(f"{key}: {value}")

print("\nDuplicate-cookie audit complete. No files were modified.")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for i in range(1, 5):
    source = f"dataset_{i}_train"
    path = root / source / f"{source}.json"

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    print(f"\n=== {source} ===")
    print("Record count:", len(records))

    for index in [0, 1, 2]:
        if index >= len(records):
            continue

        record = records[index]
        request = record.get("request", {})

        print(f"\nRecord index: {index}")
        print("Top-level keys:", list(record.keys()))
        print("Request keys:", list(request.keys()))
        print("Attack_Tag:", request.get("Attack_Tag"))
        print("Method:", request.get("method"))
        print("URL:", request.get("url"))
        print("Headers:", list(request.get("headers", {}).keys()))

print("\nIdentifier inspection complete.")

In [ ]:
import json
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

overall = Counter()
by_dataset = {}

for i in range(1, 5):
    source = f"dataset_{i}_train"
    path = root / source / f"{source}.json"

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    counts = Counter()

    for record in records:
        tag = record.get("request", {}).get("Attack_Tag")
        category = tag if tag else "Benign"
        counts[category] += 1
        overall[category] += 1

    by_dataset[source] = counts

    print(f"\n=== {source} ===")
    print("Records:", len(records))
    print("Classes:", dict(counts))

print("\n=== ALL ORIGINAL TRAINING SOURCES ===")
print("Total records:", sum(overall.values()))
print("Class counts:", dict(overall))

print("\nAudit complete. No files were modified.")

In [ ]:
import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")

for i in range(1, 5):
    source = f"dataset_{i}_train"
    path = root / source / f"{source}.json"

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    print(f"\n{'=' * 25} {source} {'=' * 25}")

    for index, record in enumerate(records):
        request = record.get("request", {})
        tag = request.get("Attack_Tag")

        if tag == "LOG4J":
            print("Record index:", index)
            print("Attack tag:", tag)
            print("Request keys:", list(request.keys()))
            print("Method:", request.get("method"))
            print("URL:", request.get("url"))
            print("Header keys:", list(request.get("headers", {}).keys()))
            print("Body:", str(request.get("body", ""))[:500])
            print("Response keys:", list(record.get("response", {}).keys()))
            print("Set-Cookie:", str(
                request.get("headers", {}).get("Set-Cookie", "")
            )[:1000])
            break
    else:
        print("No LOG4J record found.")

In [ ]:

import json
import hashlib
import random
from pathlib import Path
from collections import Counter, defaultdict

root = Path("/kaggle/working/api_pentest_research/ATRDF/extracted")
out = Path("/kaggle/working/api_pentest_research/finetuning/rebuilt_v2")
out.mkdir(parents=True, exist_ok=True)

random_seed = 42
rng = random.Random(random_seed)
split_ratios = (0.80, 0.10, 0.10)

records_by_class = defaultdict(list)
total = 0

for i in range(1, 5):
    source = f"dataset_{i}_train"
    path = root / source / f"{source}.json"

    with path.open("r", encoding="utf-8") as f:
        records = json.load(f)

    for index, record in enumerate(records):
        request = record.get("request", {})
        tag = request.get("Attack_Tag")
        category = tag if tag else "Benign"

        # Preserve all original request headers, including Set-Cookie and Date.
        complete_request = {
            "method": request.get("method", ""),
            "url": request.get("url", ""),
            "headers": request.get("headers", {}),
            "body": request.get("body", "")
        }

        model_input = (
            "Classify this HTTP request into exactly one category: "
            "Benign, Cookie Injection, Directory Traversal, LOG4J, "
            "Log Forging, RCE, SQL Injection, or XSS.\n"
            "Return only the category name.\n"
            "HTTP request:\n"
            + json.dumps(complete_request, ensure_ascii=False, sort_keys=True)
        )

        row = {
            "input": model_input,
            "category": category,
            "binary_label": "Benign" if category == "Benign" else "Malware",
            "source_dataset": source,
            "source_index": index
        }

        records_by_class[category].append(row)
        total += 1

splits = {"train": [], "validation": [], "test": []}

for category, rows in sorted(records_by_class.items()):
    rng.shuffle(rows)
    n = len(rows)
    n_train = int(n * split_ratios[0])
    n_val = int(n * split_ratios[1])

    splits["train"].extend(rows[:n_train])
    splits["validation"].extend(rows[n_train:n_train + n_val])
    splits["test"].extend(rows[n_train + n_val:])

for name, rows in splits.items():
    rng.shuffle(rows)
    path = out / f"{name}.jsonl"

    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"{name}: {len(rows):,} records -> {path}")

print("\nTotal records:", total)
print("\nClass counts:")
for category, rows in sorted(records_by_class.items()):
    print(f"{category}: {len(rows):,}")

print("\nSplit class counts:")
for name, rows in splits.items():
    print(f"\n{name}:")
    print(dict(Counter(row["category"] for row in rows)))

# Check for exact input overlap across splits.
input_hashes = {
    name: {
        hashlib.sha256(row["input"].encode("utf-8")).hexdigest()
        for row in rows
    }
    for name, rows in splits.items()
}

print("\nExact input overlap:")
print("train vs validation:", len(input_hashes["train"] & input_hashes["validation"]))
print("train vs test:", len(input_hashes["train"] & input_hashes["test"]))
print("validation vs test:", len(input_hashes["validation"] & input_hashes["test"]))

print("\nSet-Cookie presence in LOG4J examples:")
for name, rows in splits.items():
    log4j_rows = [r for r in rows if r["category"] == "LOG4J"]
    with_cookie = sum(
        '"Set-Cookie"' in r["input"] or '"Set-Cookie":' in r["input"]
        for r in log4j_rows
    )
    print(f"{name}: {with_cookie}/{len(log4j_rows)}")

print("\nDataset rebuild complete. Existing model files were not modified.")


In [ ]:

import json
import hashlib
from pathlib import Path
from collections import defaultdict, Counter

root = Path(
    "/kaggle/working/api_pentest_research/finetuning/rebuilt_v2"
)

splits = {}

for name in ["train", "validation", "test"]:
    rows = []
    with (root / f"{name}.jsonl").open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    splits[name] = rows

def input_hash(row):
    return hashlib.sha256(row["input"].encode("utf-8")).hexdigest()

hashes = {
    name: [input_hash(row) for row in rows]
    for name, rows in splits.items()
}

locations = defaultdict(list)

for split_name, rows in splits.items():
    for row, h in zip(rows, hashes[split_name]):
        locations[h].append((split_name, row["category"]))

cross_split = {
    h: entries
    for h, entries in locations.items()
    if len({split for split, _ in entries}) > 1
}

conflicting = {
    h: entries
    for h, entries in cross_split.items()
    if len({label for _, label in entries}) > 1
}

print("Unique input hashes across all splits:", len(locations))
print("Inputs appearing in multiple splits:", len(cross_split))
print("Cross-split inputs with conflicting labels:", len(conflicting))

print("\nCross-split duplicate pairs:")
names = ["train", "validation", "test"]

for i, first in enumerate(names):
    for second in names[i + 1:]:
        first_hashes = Counter(hashes[first])
        second_hashes = Counter(hashes[second])
        shared = set(first_hashes) & set(second_hashes)

        print(
            f"{first} vs {second}:",
            len(shared),
            "shared unique inputs;",
            sum(min(first_hashes[h], second_hashes[h]) for h in shared),
            "matching occurrences"
        )

print("\nExample conflicting duplicates:")
for h, entries in list(conflicting.items())[:5]:
    print(h[:12], entries)

print("\nNo files modified.")


In [ ]:

import json
import hashlib
import random
from pathlib import Path
from collections import defaultdict, Counter

source_dir = Path(
    "/kaggle/working/api_pentest_research/finetuning/rebuilt_v2"
)
output_dir = source_dir.parent / "rebuilt_v3"
output_dir.mkdir(parents=True, exist_ok=True)

seed = 42
rng = random.Random(seed)

# Read all rows and combine them before assigning new splits.
all_rows = []

for split_name in ["train", "validation", "test"]:
    path = source_dir / f"{split_name}.jsonl"
    with path.open("r", encoding="utf-8") as f:
        all_rows.extend(json.loads(line) for line in f)

# Group identical inputs together so copies cannot cross splits.
groups = defaultdict(list)

for row in all_rows:
    key = hashlib.sha256(row["input"].encode("utf-8")).hexdigest()
    groups[key].append(row)

# Stratify groups by category.
groups_by_class = defaultdict(list)

for group in groups.values():
    labels = {row["category"] for row in group}

    if len(labels) != 1:
        raise ValueError(f"Conflicting labels in an identical-input group: {labels}")

    groups_by_class[next(iter(labels))].append(group)

splits = {"train": [], "validation": [], "test": []}

for category, class_groups in sorted(groups_by_class.items()):
    rng.shuffle(class_groups)

    total_records = sum(len(group) for group in class_groups)
    target_train = int(total_records * 0.80)
    target_val = int(total_records * 0.10)

    counts = {"train": 0, "validation": 0, "test": 0}

    for group in class_groups:
        # Assign each whole group to the split furthest below its target.
        targets = {
            "train": target_train,
            "validation": target_val,
            "test": total_records - target_train - target_val
        }
        remaining = {
            name: targets[name] - counts[name]
            for name in targets
        }
        chosen = max(remaining, key=remaining.get)

        splits[chosen].extend(group)
        counts[chosen] += len(group)

for name, rows in splits.items():
    rng.shuffle(rows)

    with (output_dir / f"{name}.jsonl").open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print(f"{name}: {len(rows):,} records")

print("\nClass counts:")
for name, rows in splits.items():
    print(f"\n{name}: {dict(Counter(row['category'] for row in rows))}")

# Verify that no identical inputs cross splits.
split_hashes = {
    name: {
        hashlib.sha256(row["input"].encode("utf-8")).hexdigest()
        for row in rows
    }
    for name, rows in splits.items()
}

print("\nCross-split exact input overlap:")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    print(f"{a} vs {b}: {len(split_hashes[a] & split_hashes[b])}")

print("\nTotal records preserved:", sum(map(len, splits.values())))
print("Output directory:", output_dir)
print("Original rebuilt_v2 and model checkpoints remain unchanged.")


In [ ]:

import json
import hashlib
from pathlib import Path
from collections import Counter

root = Path(
    "/kaggle/working/api_pentest_research/finetuning/rebuilt_v3"
)

expected_classes = {
    "Benign",
    "Cookie Injection",
    "Directory Traversal",
    "LOG4J",
    "Log Forging",
    "RCE",
    "SQL Injection",
    "XSS"
}

splits = {}
total = 0

for name in ["train", "validation", "test"]:
    rows = []

    with (root / f"{name}.jsonl").open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, 1):
            row = json.loads(line)

            required = {
                "input", "category", "binary_label",
                "source_dataset", "source_index"
            }
            if not required.issubset(row):
                raise ValueError(f"{name}:{line_number}: missing required fields")

            if row["category"] not in expected_classes:
                raise ValueError(
                    f"{name}:{line_number}: unexpected category {row['category']}"
                )

            if not isinstance(row["input"], str) or not row["input"]:
                raise ValueError(f"{name}:{line_number}: empty input")

            rows.append(row)

    splits[name] = rows
    total += len(rows)

    print(f"{name}: {len(rows):,} readable records")
    print("  Class counts:", dict(Counter(r["category"] for r in rows)))

print("\nTotal records:", f"{total:,}")

hashes = {
    name: {
        hashlib.sha256(row["input"].encode("utf-8")).hexdigest()
        for row in rows
    }
    for name, rows in splits.items()
}

print("\nExact input overlap:")
for a, b in [
    ("train", "validation"),
    ("train", "test"),
    ("validation", "test")
]:
    print(f"{a} vs {b}: {len(hashes[a] & hashes[b])}")

print("\nLOG4J and Set-Cookie checks:")
for name, rows in splits.items():
    log4j_rows = [r for r in rows if r["category"] == "LOG4J"]
    cookie_rows = [r for r in log4j_rows if '"Set-Cookie"' in r["input"]]

    print(f"{name}: LOG4J={len(log4j_rows):,}, Set-Cookie present={len(cookie_rows):,}")

print("\nDataset integrity checks completed.")


In [ ]:

import json
import torch
from pathlib import Path
from collections import Counter

root = Path("/kaggle/working/api_pentest_research")
data_dir = root / "finetuning" / "rebuilt_v3"

print("=== GPU ===")
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ))

print("\n=== Training data ===")
for name in ["train", "validation", "test"]:
    path = data_dir / f"{name}.jsonl"
    with path.open("r", encoding="utf-8") as f:
        count = sum(1 for _ in f)
    print(f"{name}: {count:,} rows | {path}")

print("\n=== Existing model artifacts ===")
for path in [
    root / "qwen_api_security_lora_resume_3000" / "checkpoint-4500",
    Path("/kaggle/working/MSK_API_Security_Model_step4500.zip")
]:
    print(f"{path}: {'FOUND' if path.exists() else 'NOT FOUND'}")

print("\n=== Installed training libraries ===")
for module_name in ["transformers", "peft", "trl", "bitsandbytes", "accelerate"]:
    try:
        module = __import__(module_name)
        print(module_name, getattr(module, "__version__", "version unavailable"))
    except Exception as e:
        print(module_name, "ERROR:", str(e)[:180])


In [ ]:

import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(
        f"GPU {i}: {props.name} | "
        f"VRAM: {props.total_memory / 1024**3:.2f} GB"
    )

print("\n=== NVIDIA GPU status ===")
result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.used,utilization.gpu",
        "--format=csv"
    ],
    capture_output=True,
    text=True
)

print(result.stdout if result.returncode == 0 else result.stderr)


In [ ]:

import json
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")

candidates = [
    root / "qwen_api_security_lora_resume_3000",
    root / "finetuning",
    root
]

keywords = [
    "training_args.bin",
    "trainer_state.json",
    "adapter_config.json",
    "config.json",
    "training_config.json",
    "train_config.json"
]

found = []

for base in candidates:
    if not base.exists():
        continue

    for path in base.rglob("*"):
        if path.is_file() and path.name in keywords:
            found.append(path)

# Avoid printing duplicate paths.
found = sorted(set(found))

print("=== Configuration files found ===")
for path in found[:100]:
    print(path)

print("\n=== Adapter configuration ===")
for path in found:
    if path.name == "adapter_config.json":
        try:
            with path.open("r", encoding="utf-8") as f:
                config = json.load(f)

            print(path)
            for key in [
                "base_model_name_or_path",
                "r",
                "lora_alpha",
                "lora_dropout",
                "target_modules",
                "task_type"
            ]:
                print(f"{key}: {config.get(key)}")
        except Exception as e:
            print("Could not read:", path, str(e))

print("\n=== Trainer state ===")
for path in found:
    if path.name == "trainer_state.json":
        try:
            with path.open("r", encoding="utf-8") as f:
                state = json.load(f)

            print(path)
            print("Global step:", state.get("global_step"))
            print("Best metric:", state.get("best_metric"))
            print("Log history entries:", len(state.get("log_history", [])))
            print("Last log entries:", state.get("log_history", [])[-3:])
        except Exception as e:
            print("Could not read:", path, str(e))

print("\nInspection complete. No files modified.")


In [ ]:

import torch
from pathlib import Path

path = Path(
    "/kaggle/working/api_pentest_research/"
    "qwen_api_security_lora_resume_3000/checkpoint-4500/training_args.bin"
)

args = torch.load(path, map_location="cpu", weights_only=False)

keys = [
    "output_dir",
    "num_train_epochs",
    "max_steps",
    "per_device_train_batch_size",
    "per_device_eval_batch_size",
    "gradient_accumulation_steps",
    "learning_rate",
    "lr_scheduler_type",
    "warmup_steps",
    "warmup_ratio",
    "weight_decay",
    "fp16",
    "bf16",
    "gradient_checkpointing",
    "logging_steps",
    "save_steps",
    "eval_steps",
    "dataloader_num_workers",
    "optim",
    "max_grad_norm",
    "seed",
    "report_to"
]

print("=== Previous training settings ===")
for key in keys:
    print(f"{key}: {getattr(args, key, 'not set')}")

print("\nConfiguration read successfully. No files modified.")


In [ ]:

import torch
import accelerate
import transformers
import subprocess
import shutil

print("=== Versions ===")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

print("\n=== Launch tools ===")
for name in ["torchrun", "accelerate"]:
    print(f"{name}: {shutil.which(name)}")

print("\n=== Distributed backend ===")
print("Distributed available:", torch.distributed.is_available())
print("NCCL available:", torch.distributed.is_nccl_available())

print("\n=== GPU visibility ===")
print("CUDA devices:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

print("\n=== Existing training script candidates ===")
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research")
for path in sorted(root.rglob("*.py")):
    if any(term in path.name.lower() for term in ["train", "finetune"]):
        print(path)

print("\nNo files modified.")


In [ ]:

from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/finetuning")

files = [
    root / "train_distributed.py",
    root / "train_distributed_fresh.py",
    root / "train_resume_3000.py",
]

keywords = [
    "load_dataset",
    "train.jsonl",
    "validation.jsonl",
    "Qwen2.5",
    "BitsAndBytesConfig",
    "LoraConfig",
    "TrainingArguments",
    "SFTTrainer",
    "Trainer(",
    "per_device_train_batch_size",
    "gradient_accumulation_steps",
    "max_steps",
    "learning_rate",
    "max_length",
    "max_seq_length",
    "torchrun",
    "save_strategy",
    "eval_strategy",
    "evaluation_strategy",
]

for path in files:
    print(f"\n{'=' * 30}\n{path.name}\n{'=' * 30}")

    if not path.exists():
        print("File not found")
        continue

    lines = path.read_text(encoding="utf-8").splitlines()
    selected = set()

    for i, line in enumerate(lines):
        if any(key.lower() in line.lower() for key in keywords):
            selected.update(range(max(0, i - 2), min(len(lines), i + 3)))

    if not selected:
        print("No matching configuration lines found.")
    else:
        previous = -2
        for i in sorted(selected):
            if i > previous + 1:
                print("...")
            print(f"{i + 1:4}: {lines[i]}")
            previous = i

print("\nInspection complete. No files modified.")


In [ ]:

from pathlib import Path

path = Path(
    "/kaggle/working/api_pentest_research/finetuning/"
    "train_distributed_fresh.py"
)

lines = path.read_text(encoding="utf-8").splitlines()

print(f"File: {path}")
print(f"Total lines: {len(lines)}")

for start, end in [(1, 75), (72, 130), (125, 195), (185, len(lines))]:
    print(f"\n{'=' * 20} Lines {start}-{min(end, len(lines))} {'=' * 20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i + 1:4}: {lines[i]}")

print("\nInspection only; no files modified.")


In [ ]:

import json
from pathlib import Path
from transformers import AutoTokenizer

data_path = Path(
    "/kaggle/working/api_pentest_research/finetuning/"
    "rebuilt_v3/train.jsonl"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

sample_size = 5000
lengths = []
log4j_lengths = []
long_log4j = []

with data_path.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= sample_size:
            break

        row = json.loads(line)
        input_text = row["input"]
        length = len(tokenizer(input_text, add_special_tokens=True)["input_ids"])
        lengths.append(length)

        if row["category"] == "LOG4J":
            log4j_lengths.append(length)
            if length > 512:
                long_log4j.append({
                    "length": length,
                    "input_chars": len(input_text),
                    "source_dataset": row["source_dataset"]
                })

lengths_sorted = sorted(lengths)

def percentile(values, p):
    return values[min(len(values) - 1, int((len(values) - 1) * p))]

print("Rows sampled:", len(lengths))
print("Mean tokens:", round(sum(lengths) / len(lengths), 1))
print("Median tokens:", percentile(lengths_sorted, 0.50))
print("90th percentile:", percentile(lengths_sorted, 0.90))
print("95th percentile:", percentile(lengths_sorted, 0.95))
print("99th percentile:", percentile(lengths_sorted, 0.99))
print("Maximum tokens:", max(lengths))

print("\nAt MAX_LENGTH=512:")
print("Rows over limit:", sum(n > 512 for n in lengths))
print("Percentage over limit:", round(100 * sum(n > 512 for n in lengths) / len(lengths), 2))

print("\nLOG4J rows in sample:", len(log4j_lengths))
print("LOG4J rows over 512:", len(long_log4j))
print("First 5 long LOG4J examples:", long_log4j[:5])


In [ ]:
import json
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer

ROOT = Path("/kaggle/working/api_pentest_research")
TRAIN_FILE = ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

rng = np.random.default_rng(42)
sample_size = 2000
reservoir = []

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f):
        row = json.loads(line)

        if len(reservoir) < sample_size:
            reservoir.append(row)
        else:
            j = int(rng.integers(0, line_number + 1))
            if j < sample_size:
                reservoir[j] = row

lengths = []
log4j_lengths = []
over_512 = 0
over_768 = 0
over_1024 = 0
log4j_total = 0
log4j_over_512 = 0
log4j_over_768 = 0
log4j_over_1024 = 0

for row in reservoir:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": row["category"]}
    ]

    token_ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    length = len(token_ids)
    lengths.append(length)

    over_512 += length > 512
    over_768 += length > 768
    over_1024 += length > 1024

    if row["category"] == "LOG4J":
        log4j_total += 1
        log4j_lengths.append(length)
        log4j_over_512 += length > 512
        log4j_over_768 += length > 768
        log4j_over_1024 += length > 1024

print("Sampled rows:", len(lengths))
print("Chat-template token statistics:")
print("Mean:", round(float(np.mean(lengths)), 1))
print("Median:", int(np.percentile(lengths, 50)))
print("P90:", int(np.percentile(lengths, 90)))
print("P95:", int(np.percentile(lengths, 95)))
print("P99:", int(np.percentile(lengths, 99)))
print("Maximum:", max(lengths))

print("\nTruncation estimates:")
for limit, count in [
    (512, over_512),
    (768, over_768),
    (1024, over_1024)
]:
    print(f"Over {limit}: {count} ({count / len(lengths) * 100:.2f}%)")

print("\nLOG4J rows sampled:", log4j_total)
if log4j_total:
    print("LOG4J mean length:", round(float(np.mean(log4j_lengths)), 1))
    print("LOG4J maximum length:", max(log4j_lengths))
    print("LOG4J over 512:", log4j_over_512)
    print("LOG4J over 768:", log4j_over_768)
    print("LOG4J over 1024:", log4j_over_1024)

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

ROOT = Path("/kaggle/working/api_pentest_research")
TRAIN_FILE = ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    row = json.loads(next(f))

messages = [
    {
        "role": "system",
        "content": "Classify the HTTP request. Return only its category."
    },
    {
        "role": "user",
        "content": row["input"]
    },
    {
        "role": "assistant",
        "content": row["category"]
    }
]

print("Transformers tokenizer class:", type(tokenizer).__name__)
print("Input characters:", len(row["input"]))
print("Category:", row["category"])

direct = tokenizer(
    row["input"],
    add_special_tokens=False
)

print("\nDirect tokenization:")
print("Return type:", type(direct))
print("Input IDs type:", type(direct["input_ids"]))
print("Token count:", len(direct["input_ids"]))

templated = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

print("\nChat-template tokenization:")
print("Return type:", type(templated))
print("Value type:", type(templated).__name__)
print("Value preview:", repr(templated)[:500])

if isinstance(templated, dict):
    print("Keys:", list(templated.keys()))
    if "input_ids" in templated:
        print("Input IDs type:", type(templated["input_ids"]))
        print("Input IDs shape:", getattr(templated["input_ids"], "shape", None))
        print("Actual token count:", len(templated["input_ids"]))
elif isinstance(templated, (list, tuple)):
    print("Sequence length:", len(templated))
    if templated and isinstance(templated[0], (list, tuple)):
        print("First nested sequence length:", len(templated[0]))

rendered = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print("\nRendered chat-template text:")
print("Character count:", len(rendered))
print("Preview:", repr(rendered[:300]))

In [ ]:
import json
import numpy as np
from pathlib import Path
from transformers import AutoTokenizer

ROOT = Path("/kaggle/working/api_pentest_research")
TRAIN_FILE = ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

rng = np.random.default_rng(42)
sample_size = 2000
reservoir = []

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f):
        row = json.loads(line)

        if len(reservoir) < sample_size:
            reservoir.append(row)
        else:
            j = int(rng.integers(0, line_number + 1))
            if j < sample_size:
                reservoir[j] = row

lengths = []
log4j_lengths = []

for row in reservoir:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": row["category"]}
    ]

    encoded = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False
    )

    lengths.append(len(encoded["input_ids"]))

    if row["category"] == "LOG4J":
        log4j_lengths.append(len(encoded["input_ids"]))

lengths = np.array(lengths)
log4j_lengths = np.array(log4j_lengths)

print("Sampled rows:", len(lengths))
print("Mean:", round(float(np.mean(lengths)), 1))
print("Median:", int(np.percentile(lengths, 50)))
print("P90:", int(np.percentile(lengths, 90)))
print("P95:", int(np.percentile(lengths, 95)))
print("P99:", int(np.percentile(lengths, 99)))
print("Maximum:", int(np.max(lengths)))

print("\nOverall truncation estimates:")
for limit in [512, 768, 1024]:
    count = int(np.sum(lengths > limit))
    print(f"Over {limit}: {count} ({count / len(lengths) * 100:.2f}%)")

print("\nLOG4J rows sampled:", len(log4j_lengths))
if len(log4j_lengths):
    print("LOG4J mean:", round(float(np.mean(log4j_lengths)), 1))
    print("LOG4J maximum:", int(np.max(log4j_lengths)))

    for limit in [512, 768, 1024]:
        count = int(np.sum(log4j_lengths > limit))
        print(f"LOG4J over {limit}: {count}")

In [ ]:
import torch
import subprocess

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    torch.cuda.set_device(i)
    free_bytes, total_bytes = torch.cuda.mem_get_info(i)

    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Total memory: {total_bytes / 1024**3:.2f} GiB")
    print(f"Free memory: {free_bytes / 1024**3:.2f} GiB")
    print(f"Allocated by PyTorch: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GiB")
    print(f"Reserved by PyTorch: {torch.cuda.memory_reserved(i) / 1024**3:.2f} GiB")

print("\nGPU processes:")
subprocess.run(["nvidia-smi"])

In [ ]:
from pathlib import Path
import importlib.metadata as metadata

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_distributed_fresh.py"

print("Script exists:", SCRIPT.exists())

if SCRIPT.exists():
    lines = SCRIPT.read_text(encoding="utf-8").splitlines()

    for start, end in [(1, 100), (101, 220)]:
        print(f"\n{'=' * 20} LINES {start}-{min(end, len(lines))} {'=' * 20}")
        for i in range(start - 1, min(end, len(lines))):
            print(f"{i + 1:03}: {lines[i]}")

print("\nRelevant installed package versions:")
for package in [
    "transformers",
    "accelerate",
    "peft",
    "bitsandbytes",
    "datasets",
    "torch"
]:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"
OUTPUT_DIR = ROOT / "qwen_api_security_lora_rebuilt_v3"

assert (ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl").exists()
assert (ROOT / "finetuning" / "rebuilt_v3" / "validation.jsonl").exists()
assert not OUTPUT_DIR.exists(), f"Output directory already exists: {OUTPUT_DIR}"

code = r'''
import os
import json
import torch
import torch.distributed as dist
from pathlib import Path
import peft.utils.save_and_load as _peft_save_load
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)

def _ddp_skip_tensor_parallel_sharding(model, state_dict, adapter_name):
    for module in model.modules():
        getter = getattr(module, "get_base_layer", None)
        if getter is None:
            continue
        base_layer = getter()
        if (
            getattr(base_layer, "_hf_tp_plan", None) is not None
            and getattr(base_layer, "_hf_device_mesh", None) is not None
        ):
            raise RuntimeError(
                "DDP workaround is only valid without tensor parallelism."
            )
    return None

_peft_save_load._maybe_shard_state_dict_for_tp = _ddp_skip_tensor_parallel_sharding

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ROOT = Path("/kaggle/working/api_pentest_research")
DATA_DIR = ROOT / "finetuning" / "rebuilt_v3"
OUTPUT_DIR = ROOT / "qwen_api_security_lora_rebuilt_v3"
MAX_LENGTH = 1024

local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

TRAIN_PATH = DATA_DIR / "train.jsonl"
VAL_PATH = DATA_DIR / "validation.jsonl"

SYSTEM_PROMPT = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

def read_text_dataset(path, tokenizer):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item["input"]},
                {"role": "assistant", "content": item["category"]},
            ]
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False,
            )
            rows.append({"text": text})
    return Dataset.from_list(rows)

class ExplicitCompletionCollator:
    def __init__(self, tokenizer, max_length):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.marker = "<|im_start|>assistant\n"

    def __call__(self, features):
        rows = []

        for feature in features:
            text = feature["text"]
            if self.marker not in text:
                raise ValueError("Assistant completion marker missing.")

            prompt_text, completion_text = text.rsplit(self.marker, 1)
            prompt_text += self.marker

            prompt_ids = self.tokenizer(
                prompt_text, add_special_tokens=False
            )["input_ids"]
            completion_ids = self.tokenizer(
                completion_text, add_special_tokens=False
            )["input_ids"]

            input_ids = prompt_ids + completion_ids
            labels = [-100] * len(prompt_ids) + completion_ids.copy()

            if len(input_ids) > self.max_length:
                excess = len(input_ids) - self.max_length
                if excess >= len(prompt_ids):
                    input_ids = input_ids[-self.max_length:]
                    labels = labels[-self.max_length:]
                else:
                    input_ids = input_ids[excess:]
                    labels = labels[excess:]

            rows.append((input_ids, labels))

        max_len = max(len(ids) for ids, _ in rows)
        pad_id = self.tokenizer.pad_token_id
        input_batch, label_batch, mask_batch = [], [], []

        for ids, labels in rows:
            padding = max_len - len(ids)
            input_batch.append(ids + [pad_id] * padding)
            label_batch.append(labels + [-100] * padding)
            mask_batch.append([1] * len(ids) + [0] * padding)

        return {
            "input_ids": torch.tensor(input_batch, dtype=torch.long),
            "labels": torch.tensor(label_batch, dtype=torch.long),
            "attention_mask": torch.tensor(mask_batch, dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map={"": local_rank},
)

base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "up_proj", "down_proj", "gate_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.config.use_cache = False
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

train_dataset = read_text_dataset(TRAIN_PATH, tokenizer)
val_dataset = read_text_dataset(VAL_PATH, tokenizer)
collator = ExplicitCompletionCollator(tokenizer, MAX_LENGTH)

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    logging_steps=25,
    eval_strategy="no",
    remove_unused_columns=False,
    ddp_find_unused_parameters=False,
    optim="adamw_torch",
    dataloader_num_workers=0,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=collator,
    processing_class=tokenizer,
)

if dist.is_initialized() and dist.get_rank() == 0:
    print("Distributed processes:", dist.get_world_size())
    print("Training examples:", len(train_dataset))
    print("Validation examples loaded:", len(val_dataset))
    print("Maximum sequence length:", MAX_LENGTH)
    print("Per-GPU batch size:", 1)
    print("Gradient accumulation:", 8)
    print("Effective batch size:", 16)
    print("Training from a fresh LoRA adapter.")
    print("Output directory:", OUTPUT_DIR)

trainer.train()
trainer.save_model(str(OUTPUT_DIR))
if trainer.is_world_process_zero():
    tokenizer.save_pretrained(str(OUTPUT_DIR))
'''

SCRIPT.write_text(code, encoding="utf-8")

print("Created script:", SCRIPT)
print("New output directory:", OUTPUT_DIR)
print("Script size:", SCRIPT.stat().st_size, "bytes")
print("Old checkpoint preserved:", (
    ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
).exists())
print("Old ZIP preserved:", Path("/kaggle/working/MSK_API_Security_Model_step4500.zip").exists())
print("\nTraining has NOT started.")

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"

source = SCRIPT.read_text(encoding="utf-8")
tree = ast.parse(source, filename=str(SCRIPT))

print("Syntax check: PASSED")

checks = {
    "Correct rebuilt_v3 dataset path": 'ROOT / "finetuning" / "rebuilt_v3"' in source,
    "Sequence length 1024": "MAX_LENGTH = 1024" in source,
    "Fresh output directory": "qwen_api_security_lora_rebuilt_v3" in source,
    "Gradient checkpointing enabled": "gradient_checkpointing=True" in source,
    "Old checkpoint not loaded": "PeftModel.from_pretrained" not in source,
    "Validation evaluation disabled currently": 'eval_strategy="no"' in source,
    "Both training and validation files referenced": "validation.jsonl" in source,
}

for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {label}")

print("\nNo model was loaded and no training was started.")

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

ROOT = Path("/kaggle/working/api_pentest_research")
TRAIN_FILE = ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl"
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

samples = []
with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if row["category"] == "LOG4J":
            samples.append(row)
            if len(samples) == 5:
                break

marker = "<|im_start|>assistant\n"

for i, row in enumerate(samples, 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": row["category"]},
    ]

    rendered = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_text, completion_text = rendered.rsplit(marker, 1)
    prompt_text += marker

    prompt_ids = tokenizer(
        prompt_text, add_special_tokens=False
    )["input_ids"]
    completion_ids = tokenizer(
        completion_text, add_special_tokens=False
    )["input_ids"]

    input_ids = prompt_ids + completion_ids
    truncated_ids = input_ids[-1024:]
    truncated_text = tokenizer.decode(truncated_ids)

    print(f"\n--- LOG4J sample {i} ---")
    print("Original token count:", len(input_ids))
    print("Tokens after truncation:", len(truncated_ids))
    print("Set-Cookie in original request:", "Set-Cookie" in row["input"])
    print("Set-Cookie survives truncation:", "Set-Cookie" in truncated_text)
    print("JNDI marker in original request:", "jndi:" in row["input"].lower())
    print("JNDI marker survives truncation:", "jndi:" in truncated_text.lower())
    print("Target category:", completion_text.strip())

In [ ]:
import json
from pathlib import Path
from transformers import AutoTokenizer

ROOT = Path("/kaggle/working/api_pentest_research")
TRAIN_FILE = ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl"

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

SYSTEM_PROMPT = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

longest = []

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        if row["category"] != "LOG4J":
            continue

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": row["input"]},
            {"role": "assistant", "content": row["category"]},
        ]

        encoded = tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=False,
        )
        length = len(encoded["input_ids"])
        longest.append((length, row))

longest.sort(key=lambda item: item[0], reverse=True)
print("LOG4J examples scanned:", len(longest))
print("\nTop 10 longest LOG4J examples:")

for rank, (length, row) in enumerate(longest[:10], 1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": row["input"]},
        {"role": "assistant", "content": row["category"]},
    ]

    ids = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
    )["input_ids"]

    kept = ids[-1024:]
    decoded = tokenizer.decode(kept).lower()

    print(
        rank,
        "tokens =", length,
        "| truncated =", length > 1024,
        "| Set-Cookie kept =", "set-cookie" in decoded,
        "| jndi kept =", "jndi:" in decoded,
    )

In [ ]:
import inspect
from transformers import TrainingArguments, Trainer

print("TrainingArguments signature:")
params = inspect.signature(TrainingArguments.__init__).parameters

for name in [
    "eval_strategy",
    "eval_steps",
    "per_device_eval_batch_size",
    "save_strategy",
    "load_best_model_at_end",
    "metric_for_best_model",
    "gradient_checkpointing",
]:
    print(f"{name}: {'SUPPORTED' if name in params else 'NOT SUPPORTED'}")

print("\nTrainer.evaluate available:", callable(getattr(Trainer, "evaluate", None)))
print("Trainer prediction available:", callable(getattr(Trainer, "predict", None)))

In [ ]:
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"

source = SCRIPT.read_text(encoding="utf-8")

source = source.replace(
    '    eval_strategy="no",',
    '    eval_strategy="steps",\n'
    '    eval_steps=500,'
)

source = source.replace(
    '    save_steps=500,',
    '    save_steps=500,'
)

source = source.replace(
    '    per_device_eval_batch_size=1,',
    '    per_device_eval_batch_size=1,'
)

source = source.replace(
    '    save_total_limit=2,',
    '    save_total_limit=2,\n'
    '    load_best_model_at_end=False,'
)

SCRIPT.write_text(source, encoding="utf-8")

print("Updated:", SCRIPT)
print("Validation strategy enabled:", 'eval_strategy="steps"' in source)
print("Validation interval:", "eval_steps=500" in source)
print("Evaluation dataset passed to Trainer:", "eval_dataset=val_dataset" in source)

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"

source = SCRIPT.read_text(encoding="utf-8")

old = '''    train_dataset=train_dataset,
    data_collator=collator,
    processing_class=tokenizer,
)'''

new = '''    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    processing_class=tokenizer,
)'''

if old not in source:
    raise RuntimeError(
        "Expected Trainer block not found. Script was not modified."
    )

source = source.replace(old, new, 1)
ast.parse(source, filename=str(SCRIPT))
SCRIPT.write_text(source, encoding="utf-8")

print("Syntax check: PASSED")
print("Validation dataset passed to Trainer:", "eval_dataset=val_dataset" in source)
print("Evaluation strategy enabled:", 'eval_strategy="steps"' in source)
print("Evaluation interval:", "eval_steps=500" in source)
print("Sequence length:", "MAX_LENGTH = 1024" in source)
print("New output directory:", "qwen_api_security_lora_rebuilt_v3" in source)
print("Training started: NO")

In [ ]:
import json
import ast
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
DATA = ROOT / "finetuning" / "rebuilt_v3"
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"
OUTPUT = ROOT / "qwen_api_security_lora_rebuilt_v3"

def count_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

train_count = count_lines(DATA / "train.jsonl")
val_count = count_lines(DATA / "validation.jsonl")
test_count = count_lines(DATA / "test.jsonl")

source = SCRIPT.read_text(encoding="utf-8")
ast.parse(source)

print("Dataset sizes:")
print("Train:", train_count)
print("Validation:", val_count)
print("Test:", test_count)
print("Total:", train_count + val_count + test_count)

world_size = 2
batch_size = 1
grad_accumulation = 8
effective_batch = world_size * batch_size * grad_accumulation
steps_per_epoch = (train_count + effective_batch - 1) // effective_batch

print("\nTraining estimates:")
print("Effective batch size:", effective_batch)
print("Optimizer steps for one epoch:", steps_per_epoch)
print("Checkpoint interval:", 500, "steps")
print("Validation interval:", 500, "steps")

print("\nScript checks:")
print("Syntax valid:", True)
print("Uses rebuilt_v3:", "rebuilt_v3" in source)
print("Maximum sequence length 1024:", "MAX_LENGTH = 1024" in source)
print("Validation dataset connected:", "eval_dataset=val_dataset" in source)
print("Fresh output directory:", str(OUTPUT) in source)
print("Output directory already exists:", OUTPUT.exists())
print("Old checkpoint exists:", (
    ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
).exists())
print("Old ZIP exists:", Path(
    "/kaggle/working/MSK_API_Security_Model_step4500.zip"
).exists())

In [ ]:
import json
import random
from collections import defaultdict
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
DATA = ROOT / "finetuning" / "rebuilt_v3"
SOURCE = DATA / "validation.jsonl"
DEST = DATA / "validation_sample_2000.jsonl"

assert SOURCE.exists()
assert not DEST.exists(), f"File already exists: {DEST}"

groups = defaultdict(list)

with open(SOURCE, "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        groups[row["category"]].append(row)

random.seed(42)
target_size = 2000
total = sum(len(rows) for rows in groups.values())

sample = []
for category, rows in groups.items():
    random.shuffle(rows)
    quota = round(target_size * len(rows) / total)
    sample.extend(rows[:quota])

random.shuffle(sample)
sample = sample[:target_size]

with open(DEST, "w", encoding="utf-8") as f:
    for row in sample:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

counts = defaultdict(int)
for row in sample:
    counts[row["category"]] += 1

print("Created:", DEST)
print("Sample rows:", len(sample))
print("Class counts:")
for category, count in sorted(counts.items()):
    print(f"{category}: {count}")

print("\nOriginal validation preserved:", SOURCE.exists())
print("Test set untouched:", (DATA / "test.jsonl").exists())

In [ ]:
from pathlib import Path
import ast

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"

source = SCRIPT.read_text(encoding="utf-8")

source = source.replace(
    'VAL_PATH = DATA_DIR / "validation.jsonl"',
    'VAL_PATH = DATA_DIR / "validation_sample_2000.jsonl"'
)

source = source.replace(
    '    eval_steps=500,',
    '    eval_steps=1000,'
)

source = source.replace(
    '    save_steps=500,',
    '    save_steps=1000,'
)

ast.parse(source)
SCRIPT.write_text(source, encoding="utf-8")

print("Syntax check: PASSED")
print("Training dataset: rebuilt_v3/train.jsonl")
print("Validation dataset: rebuilt_v3/validation_sample_2000.jsonl")
print("Evaluation interval: 1000 optimizer steps")
print("Checkpoint interval: 1000 optimizer steps")
print("Sequence length: 1024")
print("Training started: NO")

In [ ]:
import ast
import json
import torch
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"
DATA = ROOT / "finetuning" / "rebuilt_v3"
OUTPUT = ROOT / "qwen_api_security_lora_rebuilt_v3"

source = SCRIPT.read_text(encoding="utf-8")
ast.parse(source)

print("=== SCRIPT CONFIGURATION ===")
for check, passed in [
    ("Valid Python syntax", True),
    ("Correct training data", 'TRAIN_PATH = DATA_DIR / "train.jsonl"' in source),
    ("Sampled validation data", 'VAL_PATH = DATA_DIR / "validation_sample_2000.jsonl"' in source),
    ("1024-token context", "MAX_LENGTH = 1024" in source),
    ("Evaluation every 1000 steps", "eval_steps=1000" in source),
    ("Save every 1000 steps", "save_steps=1000" in source),
    ("Evaluation dataset connected", "eval_dataset=val_dataset" in source),
    ("Fresh output path configured", 'OUTPUT_DIR = ROOT / "qwen_api_security_lora_rebuilt_v3"' in source),
]:
    print(f"{'PASS' if passed else 'FAIL'}: {check}")

print("\n=== DATASET FILES ===")
for name in ["train.jsonl", "validation.jsonl", "validation_sample_2000.jsonl", "test.jsonl"]:
    path = DATA / name
    print(f"{name}: exists={path.exists()}, size={path.stat().st_size if path.exists() else 0}")

print("\n=== GPU STATUS ===")
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)}, "
        f"free={free / 1024**3:.2f} GiB / total={total / 1024**3:.2f} GiB"
    )

print("\n=== PRESERVATION CHECK ===")
print("New output directory exists:", OUTPUT.exists())
print("Old checkpoint exists:", (
    ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
).exists())
print("Old ZIP exists:", Path("/kaggle/working/MSK_API_Security_Model_step4500.zip").exists())

print("\nPreflight complete. Training has NOT started.")

In [ ]:
import os
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
SCRIPT = ROOT / "finetuning" / "train_rebuilt_v3.py"
OUTPUT = ROOT / "qwen_api_security_lora_rebuilt_v3"
LOG = ROOT / "finetuning" / "train_rebuilt_v3.log"

assert SCRIPT.exists(), "Training script missing"
assert not OUTPUT.exists(), "Output directory already exists; refusing to overwrite"
assert torch.cuda.device_count() == 2, "Expected exactly two visible GPUs"

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0,1"
env["TOKENIZERS_PARALLELISM"] = "false"
env["OMP_NUM_THREADS"] = "4"

with open(LOG, "w", encoding="utf-8") as log_file:
    process = subprocess.Popen(
        [
            "torchrun",
            "--standalone",
            "--nnodes=1",
            "--nproc_per_node=2",
            str(SCRIPT),
        ],
        cwd=str(ROOT),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Training process launched.")
print("PID:", process.pid)
print("Log file:", LOG)
print("Output directory:", OUTPUT)
print("Both GPUs requested:", env["CUDA_VISIBLE_DEVICES"])
print("\nTraining runs in the background. Check the log in the next step.")

In [ ]:
import subprocess
import time
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
LOG = ROOT / "finetuning" / "train_rebuilt_v3.log"

time.sleep(10)

print("=== TRAINING LOG (LAST 100 LINES) ===")
if LOG.exists():
    result = subprocess.run(
        ["tail", "-n", "100", str(LOG)],
        capture_output=True,
        text=True,
    )
    print(result.stdout or "(Log is currently empty)")
else:
    print("Log file not found.")

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi"])

print("\n=== TRAINING PROCESSES ===")
subprocess.run([
    "bash", "-lc",
    "ps -eo pid,ppid,stat,etime,cmd | grep -E 'torchrun|train_rebuilt_v3.py' | grep -v grep"
])

In [ ]:
import re
import subprocess
from pathlib import Path

LOG = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_rebuilt_v3.log"
)

if not LOG.exists():
    print("Training log not found.")
else:
    lines = LOG.read_text(errors="replace").splitlines()

    progress = None
    for line in reversed(lines):
        matches = re.findall(r"(\d+)/(\d+)\s+\[([^\]]+)\]", line)
        for step, total, timing in matches:
            if int(total) == 22953:
                progress = (int(step), int(total), timing)
                break
        if progress:
            break

    if progress:
        step, total, timing = progress
        percent = step / total * 100

        print("=== TRAINING PROGRESS ===")
        print(f"Completed steps: {step:,} / {total:,}")
        print(f"Progress: {percent:.2f}%")
        print(f"Last recorded timing: {timing}")
        print(f"Steps remaining: {total - step:,}")
    else:
        print("No training progress line found yet.")

    print("\n=== LATEST TRAINING LOG ===")
    for line in lines[-15:]:
        print(line)

    print("\n=== GPU STATUS ===")
    subprocess.run(["nvidia-smi"])

In [ ]:
import psutil
import time

matches = []

for p in psutil.process_iter(["pid", "cmdline"]):
    try:
        cmd = " ".join(p.info["cmdline"] or [])
        if (
            "train_rebuilt_v3.py" in cmd
            and p.pid != __import__("os").getpid()
        ):
            matches.append(p)
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

print("Matching training processes:", [p.pid for p in matches])

for p in matches:
    try:
        p.terminate()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

_, alive = psutil.wait_procs(matches, timeout=10)

for p in alive:
    try:
        p.kill()
    except (psutil.NoSuchProcess, psutil.AccessDenied):
        pass

print("Stopped the rebuilt_v3 training processes.")
print("Existing checkpoint and ZIP were not targeted.")

In [ ]:

from pathlib import Path
import subprocess
import json

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
OUT = ROOT / "qwen_api_security_lora_rebuilt_v3"

print("=== TRAINING PROCESS CHECK ===")
result = subprocess.run(
    ["bash", "-lc", "ps -eo pid,args | grep -E 'torchrun|train_rebuilt_v3.py' | grep -v grep"],
    capture_output=True, text=True
)
print(result.stdout.strip() or "No matching training processes found.")

print("\n=== EXISTING MODEL FILES ===")
for path in [
    ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500",
    ROOT / "MSK_API_Security_Model_step4500.zip",
    FT / "rebuilt_v3" / "train.jsonl",
    FT / "rebuilt_v3" / "validation.jsonl",
    FT / "rebuilt_v3" / "test.jsonl",
]:
    print(f"{'EXISTS' if path.exists() else 'MISSING'}: {path}")

print("\n=== NEW RUN CHECKPOINTS ===")
if OUT.exists():
    checkpoints = sorted(
        (p for p in OUT.glob("checkpoint-*") if p.is_dir()),
        key=lambda p: int(p.name.split("-")[-1])
    )
    print([p.name for p in checkpoints] or "No saved checkpoints found.")
else:
    print("Output directory does not exist.")

print("\n=== DATASET SIZES ===")
for name in ["train.jsonl", "validation.jsonl", "test.jsonl",
             "validation_sample_2000.jsonl"]:
    p = FT / "rebuilt_v3" / name
    if p.exists():
        with p.open("r", encoding="utf-8") as f:
            count = sum(1 for _ in f)
        print(f"{name}: {count:,} rows")


In [ ]:

from pathlib import Path
import re

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
script = FT / "train_rebuilt_v3.py"

print("=== ZIP LOCATION CHECK ===")
for p in [
    Path("/kaggle/working/MSK_API_Security_Model_step4500.zip"),
    ROOT / "MSK_API_Security_Model_step4500.zip",
]:
    print(f"{'EXISTS' if p.exists() else 'MISSING'}: {p}")

print("\n=== TRAINING SCRIPT: LOSS AND LABEL LOGIC ===")
if script.exists():
    lines = script.read_text(errors="replace").splitlines()
    patterns = re.compile(
        r"collator|labels|completion|assistant|loss|MAX_LENGTH|"
        r"truncat|DataCollator|TrainingArguments|compute_loss",
        re.I
    )
    shown = set()
    for i, line in enumerate(lines):
        if patterns.search(line):
            start, end = max(0, i - 2), min(len(lines), i + 3)
            for j in range(start, end):
                if j not in shown:
                    print(f"{j+1:4}: {lines[j]}")
                    shown.add(j)
            print("---")
else:
    print("Training script not found.")


In [ ]:

from pathlib import Path
from collections import Counter
import json
import re

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
script = FT / "train_rebuilt_v3.py"
train_path = FT / "rebuilt_v3" / "train.jsonl"

print("=== TRAINING STEP SETTINGS ===")
lines = script.read_text(errors="replace").splitlines()
for i, line in enumerate(lines):
    if 165 <= i <= 198:
        print(f"{i+1:4}: {line}")

print("\n=== TRAINING CLASS COUNTS ===")
counts = Counter()
with train_path.open("r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        counts[item.get("category", "UNKNOWN")] += 1

for category, count in counts.most_common():
    print(f"{category}: {count:,}")

print("\nTotal:", sum(counts.values()))
print("\n=== OLD CHECKPOINT AND ZIP ===")
for p in [
    ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500",
    Path("/kaggle/working/MSK_API_Security_Model_step4500.zip"),
]:
    print(f"{'EXISTS' if p.exists() else 'MISSING'}: {p}")


In [ ]:

from pathlib import Path
from collections import defaultdict, Counter
import random
import json

ROOT = Path("/kaggle/working/api_pentest_research")
SOURCE = ROOT / "finetuning" / "rebuilt_v3" / "train.jsonl"
DEST_DIR = ROOT / "finetuning" / "rebuilt_v4"
DEST = DEST_DIR / "train.jsonl"

SEED = 42
TARGETS = {
    "Benign": 2000,
    "Cookie Injection": 1000,
    "Directory Traversal": 1000,
    "LOG4J": 1000,
    "Log Forging": 1000,
    "RCE": 1000,
    "SQL Injection": 1000,
    "XSS": 1000,
}

rng = random.Random(SEED)
groups = defaultdict(list)

with SOURCE.open("r", encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        category = item.get("category")
        if category in TARGETS:
            groups[category].append(item)

for category, target in TARGETS.items():
    if len(groups[category]) < target:
        raise ValueError(
            f"Not enough examples for {category}: "
            f"{len(groups[category])} available, {target} needed"
        )

selected = []
for category, target in TARGETS.items():
    selected.extend(rng.sample(groups[category], target))

rng.shuffle(selected)
DEST_DIR.mkdir(parents=True, exist_ok=True)

with DEST.open("w", encoding="utf-8") as f:
    for item in selected:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Reduced dataset created:", DEST)
print("Total examples:", len(selected))
print("\nClass counts:")
for category, count in sorted(Counter(x["category"] for x in selected).items()):
    print(f"{category}: {count}")

print("\nOriginal dataset unchanged:", SOURCE.exists())
print("Validation unchanged:", (ROOT / "finetuning/rebuilt_v3/validation.jsonl").exists())
print("Test unchanged:", (ROOT / "finetuning/rebuilt_v3/test.jsonl").exists())


In [ ]:

from pathlib import Path

script = Path(
    "/kaggle/working/api_pentest_research/finetuning/train_rebuilt_v3.py"
)
lines = script.read_text(errors="replace").splitlines()

print("=== DATASET AND OUTPUT PATHS ===")
for i, line in enumerate(lines[:75], start=1):
    print(f"{i:3}: {line}")

print("\n=== TRAINING ARGUMENTS ===")
for i, line in enumerate(lines[165:205], start=166):
    print(f"{i:3}: {line}")


In [ ]:

from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
SCRIPT = FT / "train_rebuilt_v4.py"

source = (FT / "train_rebuilt_v3.py").read_text(encoding="utf-8")

source = source.replace(
    'DATA_DIR = ROOT / "finetuning" / "rebuilt_v3"',
    'DATA_DIR = ROOT / "finetuning" / "rebuilt_v4"'
)
source = source.replace(
    'OUTPUT_DIR = ROOT / "qwen_api_security_lora_rebuilt_v3"',
    'OUTPUT_DIR = ROOT / "qwen_api_security_lora_rebuilt_v4"'
)
source = source.replace(
    'VAL_PATH = DATA_DIR / "validation_sample_2000.jsonl"',
    'VAL_PATH = ROOT / "finetuning" / "rebuilt_v3" / "validation_sample_2000.jsonl"'
)
source = source.replace(
    '    num_train_epochs=1,',
    '    num_train_epochs=1,\n    max_steps=300,'
)
source = source.replace('    save_steps=1000,', '    save_steps=100,')
source = source.replace('    eval_steps=1000,', '    eval_steps=100,')

if "max_steps=300" not in source:
    raise RuntimeError("Could not verify the 300-step limit.")
if "save_steps=100" not in source or "eval_steps=100" not in source:
    raise RuntimeError("Checkpoint/evaluation settings were not updated.")
if "rebuilt_v4" not in source:
    raise RuntimeError("Reduced dataset path was not applied.")

SCRIPT.write_text(source, encoding="utf-8")

print("Created:", SCRIPT)
print("Training dataset:", ROOT / "finetuning" / "rebuilt_v4" / "train.jsonl")
print("Validation dataset:", ROOT / "finetuning" / "rebuilt_v3" / "validation_sample_2000.jsonl")
print("Output directory:", ROOT / "qwen_api_security_lora_rebuilt_v4")
print("Maximum optimizer steps: 300")
print("Checkpoint interval: 100 steps")
print("Evaluation interval: 100 steps")
print("Training started: NO")


In [ ]:

from pathlib import Path
import ast
import re

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
SCRIPT = FT / "train_rebuilt_v4.py"
TRAIN = FT / "rebuilt_v4" / "train.jsonl"
VAL = FT / "rebuilt_v3" / "validation_sample_2000.jsonl"
OUT = ROOT / "qwen_api_security_lora_rebuilt_v4"
OLD = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
ZIP = Path("/kaggle/working/MSK_API_Security_Model_step4500.zip")

source = SCRIPT.read_text(encoding="utf-8")
ast.parse(source)

checks = {
    "Script syntax valid": True,
    "Reduced training data exists": TRAIN.exists(),
    "Validation sample exists": VAL.exists(),
    "Old checkpoint exists": OLD.exists(),
    "Original ZIP exists": ZIP.exists(),
    "New output path is separate": OUT != OLD.parent,
    "Maximum steps set to 300": bool(re.search(r"max_steps\s*=\s*300", source)),
    "Save interval set to 100": bool(re.search(r"save_steps\s*=\s*100", source)),
    "Evaluation interval set to 100": bool(re.search(r"eval_steps\s*=\s*100", source)),
    "Reduced dataset path configured": 'DATA_DIR = ROOT / "finetuning" / "rebuilt_v4"' in source,
}

for label, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {label}")

print("\nOutput directory already exists:", OUT.exists())
print("Training started: NO")

if not all(checks.values()):
    print("\nPreflight failed. Do not start training.")
else:
    print("\nPreflight passed. Ready for the next check.")


In [ ]:

import os
import subprocess
from pathlib import Path

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
SCRIPT = FT / "train_rebuilt_v4.py"
OUT = ROOT / "qwen_api_security_lora_rebuilt_v4"
LOG = FT / "train_rebuilt_v4.log"

if not SCRIPT.exists():
    raise FileNotFoundError(SCRIPT)

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(
        f"Output directory is not empty: {OUT}. "
        "Stopping to avoid overwriting existing results."
    )

with LOG.open("w", encoding="utf-8") as log_file:
    env = os.environ.copy()
    env["CUDA_VISIBLE_DEVICES"] = "0,1"
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["OMP_NUM_THREADS"] = "4"

    process = subprocess.Popen(
        [
            "torchrun",
            "--standalone",
            "--nnodes=1",
            "--nproc_per_node=2",
            str(SCRIPT),
        ],
        cwd=str(ROOT),
        env=env,
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

print("Training launched.")
print("Launcher PID:", process.pid)
print("Maximum optimizer steps: 300")
print("Log file:", LOG)
print("New output directory:", OUT)
print("Old checkpoint and ZIP are not being modified.")


In [ ]:

import subprocess
from pathlib import Path

LOG = Path("/kaggle/working/api_pentest_research/finetuning/train_rebuilt_v4.log")

print("=== LATEST TRAINING LOG ===")
if LOG.exists():
    lines = LOG.read_text(errors="replace").splitlines()
    print("\n".join(lines[-35:]) or "Log is currently empty.")
else:
    print("Log file not found yet.")

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi"])

print("\n=== TRAINING PROCESSES ===")
subprocess.run([
    "bash", "-lc",
    "ps -eo pid,etime,args | grep -E 'torchrun|train_rebuilt_v4.py' | grep -v grep"
])


In [ ]:

import re
import subprocess
from pathlib import Path

LOG = Path("/kaggle/working/api_pentest_research/finetuning/train_rebuilt_v4.log")

lines = LOG.read_text(errors="replace").splitlines()
print("=== RECENT TRAINING PROGRESS ===")

progress_lines = [
    line for line in lines
    if re.search(r"\b\d+/300\b", line)
]
print("\n".join(progress_lines[-8:]) or "No new step updates yet.")

print("\n=== RECENT LOG MESSAGES ===")
print("\n".join(lines[-12:]))

print("\n=== TRAINING PROCESS ===")
subprocess.run([
    "bash", "-lc",
    "ps -eo pid,etime,args | grep -E 'torchrun.*train_rebuilt_v4|train_rebuilt_v4.py' | grep -v grep"
])


In [ ]:
from pathlib import Path
import re

log = Path("/kaggle/working/api_pentest_research/finetuning/train_rebuilt_v4.log")
lines = log.read_text(errors="replace").splitlines()

for line in reversed(lines):
    if re.search(r"\b\d+/300\b", line):
        print(line)
        break

print("\nLatest training metrics:")
for line in lines[-30:]:
    if "loss" in line.lower() or "eval_loss" in line.lower() or "error" in line.lower():
        print(line)

In [ ]:

from pathlib import Path
import subprocess

ROOT = Path("/kaggle/working/api_pentest_research")
OUT = ROOT / "qwen_api_security_lora_rebuilt_v4"
LOG = ROOT / "finetuning" / "train_rebuilt_v4.log"

print("=== CHECKPOINTS ===")
if OUT.exists():
    for p in sorted(OUT.glob("checkpoint-*")):
        if p.is_dir():
            files = list(p.iterdir())
            print(f"{p.name}: {len(files)} files")
            print("  ", [f.name for f in files[:12]])
else:
    print("Output directory not found.")

print("\n=== LATEST LOG ===")
lines = LOG.read_text(errors="replace").splitlines()
print("\n".join(lines[-35:]))

print("\n=== TRAINING PROCESS ===")
subprocess.run([
    "bash", "-lc",
    "ps -eo pid,etime,args | grep -E 'torchrun.*train_rebuilt_v4|train_rebuilt_v4.py' | grep -v grep"
])


In [ ]:

from pathlib import Path
import subprocess
import re
import os
import time

ROOT = Path("/kaggle/working/api_pentest_research")
FT = ROOT / "finetuning"
SCRIPT = FT / "train_rebuilt_v4.py"
LOG = FT / "train_rebuilt_v4.log"
OUT = ROOT / "qwen_api_security_lora_rebuilt_v4"

print("=== RUNNING TRAINING COMMANDS ===")
subprocess.run([
    "bash", "-lc",
    "ps -eo pid,ppid,lstart,etime,args | grep -E 'torchrun|train_rebuilt' | grep -v grep"
])

print("\n=== CURRENT SCRIPT SETTINGS ===")
if SCRIPT.exists():
    src = SCRIPT.read_text(errors="replace")
    for pattern in [r"max_steps\s*=\s*\d+", r"num_train_epochs\s*=\s*[^,\n]+",
                    r"save_steps\s*=\s*\d+", r"eval_steps\s*=\s*\d+",
                    r"OUTPUT_DIR\s*=.*", r"DATA_DIR\s*=.*"]:
        print(pattern, "=>", re.findall(pattern, src))
else:
    print("Script missing")

print("\n=== LOG FILE INFO ===")
if LOG.exists():
    print("Size:", LOG.stat().st_size, "bytes")
    print("Last modified:", time.ctime(LOG.stat().st_mtime))
    lines = LOG.read_text(errors="replace").splitlines()
    print("\n".join(lines[-12:]))
else:
    print("Log missing")

print("\n=== CHECKPOINT SEARCH ===")
for parent in [OUT, ROOT / "qwen_api_security_lora_rebuilt_v3"]:
    print(parent, "exists:", parent.exists())
    if parent.exists():
        print([str(p) for p in parent.glob("checkpoint-*")])

print("\n=== GPU STATUS ===")
subprocess.run(["nvidia-smi"])


In [ ]:

from pathlib import Path
import re

ROOT = Path("/kaggle/working/api_pentest_research")
LOG = ROOT / "finetuning" / "train_rebuilt_v4.log"
OUT = ROOT / "qwen_api_security_lora_rebuilt_v4"

lines = LOG.read_text(errors="replace").splitlines()

print("=== TRAINING STEP LINES ===")
for line in lines:
    if re.search(r"\b\d+/300\b", line):
        print(line)

print("\n=== EVALUATION METRICS / ERRORS ===")
matches = [
    line for line in lines
    if any(term in line.lower() for term in [
        "eval_loss", "eval_runtime", "eval_samples_per_second",
        "saving model checkpoint", "error", "traceback"
    ])
]
print("\n".join(matches[-25:]) or "No matching metrics or errors logged yet.")

print("\n=== OUTPUT DIRECTORY CONTENTS ===")
if OUT.exists():
    for p in sorted(OUT.rglob("*")):
        if p.is_file():
            print(p.relative_to(OUT), f"({p.stat().st_size:,} bytes)")
else:
    print("Output directory not found.")

print("\n=== LAST LOG LINES ===")
print("\n".join(lines[-15:]))


In [ ]:
from pathlib import Path
import subprocess

root = Path("/kaggle/working/api_pentest_research")
out = root / "qwen_api_security_lora_rebuilt_v4"
log = root / "finetuning" / "train_rebuilt_v4.log"

print("=== ACTIVE TRAINING PROCESSES ===")
result = subprocess.run(
    ["bash", "-lc",
     "ps -eo pid,etime,args | grep -E 'torchrun.*train_rebuilt_v4|train_rebuilt_v4.py' | grep -v grep"],
    capture_output=True, text=True
)
print(result.stdout.strip() or "No active training process")

print("\n=== CHECKPOINTS AND ADAPTER FILES ===")
if out.exists():
    files = [p for p in out.rglob("*") if p.is_file()]
    for p in files:
        print(p.relative_to(out), f"{p.stat().st_size:,} bytes")
    if not files:
        print("No files saved yet")
else:
    print("Output directory does not exist")

print("\n=== LATEST LOG MESSAGES ===")
lines = log.read_text(errors="replace").splitlines() if log.exists() else []
for line in lines[-60:]:
    if any(s in line.lower() for s in [
        "eval_loss", "eval_runtime", "saving", "checkpoint",
        "train_loss", "error", "traceback", "300/300"
    ]):
        print(line)

In [ ]:

from pathlib import Path
import json

ROOT = Path("/kaggle/working/api_pentest_research")
OUT = ROOT / "qwen_api_security_lora_rebuilt_v4"
TEST = ROOT / "finetuning" / "rebuilt_v3" / "test.jsonl"
LOG = ROOT / "finetuning" / "train_rebuilt_v4.log"

print("=== FINAL MODEL ===")
for p in [
    OUT / "adapter_model.safetensors",
    OUT / "adapter_config.json",
    OUT / "checkpoint-300" / "adapter_model.safetensors",
]:
    print(f"{'EXISTS' if p.exists() else 'MISSING'}: {p}")

print("\n=== ORIGINAL MODEL ===")
old = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
print("Original checkpoint exists:", old.exists())
print("Original ZIP exists:",
      Path("/kaggle/working/MSK_API_Security_Model_step4500.zip").exists())

print("\n=== TEST DATA ===")
print("Test file exists:", TEST.exists())
if TEST.exists():
    with TEST.open("r", encoding="utf-8") as f:
        print("Test rows:", sum(1 for _ in f))

print("\n=== FINAL TRAINING METRICS ===")
if LOG.exists():
    for line in LOG.read_text(errors="replace").splitlines():
        if "train_runtime" in line or "train_loss" in line or "eval_loss" in line:
            print(line)


In [ ]:

from pathlib import Path
import json

ROOT = Path("/kaggle/working/api_pentest_research")
TEST = ROOT / "finetuning" / "rebuilt_v3" / "test.jsonl"
OLD = ROOT / "qwen_api_security_lora_resume_3000" / "checkpoint-4500"
NEW = ROOT / "qwen_api_security_lora_rebuilt_v4"

print("=== TEST SAMPLE ===")
with TEST.open("r", encoding="utf-8") as f:
    for i in range(2):
        item = json.loads(next(f))
        print(f"\nSample {i+1}")
        print("Fields:", list(item.keys()))
        print("Category:", item.get("category"))
        print("Input preview:", item.get("input", "")[:700])

print("\n=== ADAPTER CONFIGS ===")
for label, path in [("Original", OLD), ("New", NEW)]:
    config_path = path / "adapter_config.json"
    print(f"\n{label}: {config_path} exists={config_path.exists()}")
    if config_path.exists():
        cfg = json.loads(config_path.read_text())
        for key in [
            "base_model_name_or_path", "r", "lora_alpha",
            "lora_dropout", "target_modules", "task_type"
        ]:
            print(f"{key}: {cfg.get(key)}")


In [ ]:

import json, gc, torch
from pathlib import Path
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

ROOT = Path("/kaggle/working/api_pentest_research")
TEST = ROOT / "finetuning/rebuilt_v3/test.jsonl"
OLD = ROOT / "qwen_api_security_lora_resume_3000/checkpoint-4500"
NEW = ROOT / "qwen_api_security_lora_rebuilt_v4"

with TEST.open(encoding="utf-8") as f:
    log4j = [json.loads(line) for line in f if json.loads(line)["category"] == "LOG4J"]

print("LOG4J test examples:", len(log4j))

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

def load_base():
    model = AutoModelForCausalLM.from_pretrained(
        BASE,
        quantization_config=quant,
        dtype=torch.float16,
        device_map={"": 0},
    )
    model.config.pad_token_id = tokenizer.pad_token_id
    return model

def evaluate(adapter_path, label):
    base = load_base()
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    detected = 0
    misses = Counter()
    batch_size = 8

    for start in range(0, len(log4j), batch_size):
        batch = log4j[start:start + batch_size]
        prompts = [
            tokenizer.apply_chat_template(
                [
                    {"role": "system", "content":
                     "You are an API security classifier. Return only one category: "
                     "Benign, Cookie Injection, Directory Traversal, LOG4J, "
                     "Log Forging, RCE, SQL Injection, or XSS."},
                    {"role": "user", "content": x["input"]},
                ],
                tokenize=False,
                add_generation_prompt=True,
            )
            for x in batch
        ]
        enc = tokenizer(
            prompts, return_tensors="pt", padding=True,
            truncation=True, max_length=1024
        ).to("cuda:0")

        with torch.inference_mode():
            out = model.generate(
                **enc, max_new_tokens=6, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        decoded = tokenizer.batch_decode(
            out[:, enc["input_ids"].shape[1]:],
            skip_special_tokens=True
        )
        for text in decoded:
            pred = next(
                (c for c in [
                    "Benign", "Cookie Injection", "Directory Traversal",
                    "LOG4J", "Log Forging", "RCE", "SQL Injection", "XSS"
                ] if text.strip().startswith(c)),
                "INVALID"
            )
            if pred == "LOG4J":
                detected += 1
            else:
                misses[pred] += 1

        if start % 160 == 0:
            print(f"{label}: processed {min(start + batch_size, len(log4j))}/{len(log4j)}")

    total = len(log4j)
    print(f"\n{label} LOG4J results")
    print(f"Detected: {detected}/{total}")
    print(f"Missed: {total - detected}/{total}")
    print(f"Recall: {detected / total:.4%}")
    print("Missed predictions:", dict(misses))

    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return detected

old_detected = evaluate(OLD, "Original")
new_detected = evaluate(NEW, "RebuiltV4")

print("\n=== FINAL LOG4J COMPARISON ===")
print(f"Original recall: {old_detected}/{len(log4j)} = {old_detected/len(log4j):.4%}")
print(f"New recall:      {new_detected}/{len(log4j)} = {new_detected/len(log4j):.4%}")
print(f"Change: {new_detected - old_detected:+d} additional LOG4J detections")


In [ ]:
import torch

print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"Allocated: {torch.cuda.memory_allocated(i) / 1024**3:.2f} GB")
    print(f"Reserved:  {torch.cuda.memory_reserved(i) / 1024**3:.2f} GB")
    print(f"Total:     {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GB")

In [ ]:
import json
import random
from pathlib import Path
from collections import defaultdict

ROOT = Path("/kaggle/working/api_pentest_research")
TEST = ROOT / "finetuning/rebuilt_v3/test.jsonl"
OUT = ROOT / "finetuning/rebuilt_v4/test_sample_2000.jsonl"

random.seed(42)
groups = defaultdict(list)

with TEST.open(encoding="utf-8") as f:
    for line in f:
        item = json.loads(line)
        groups[item["category"]].append(item)

total = sum(len(items) for items in groups.values())
sample = []

for category, items in groups.items():
    random.shuffle(items)
    count = max(1, round(2000 * len(items) / total))
    sample.extend(items[:count])

random.shuffle(sample)
sample = sample[:2000]

with OUT.open("w", encoding="utf-8") as f:
    for item in sample:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Sample saved:", OUT)
print("Total examples:", len(sample))
print("\nClass distribution:")
for category, count in sorted(
    {c: sum(x["category"] == c for x in sample) for c in groups}.items()
):
    print(f"{category}: {count}")

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import gc
import torch
import numpy as np
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

ROOT = Path("/kaggle/working/api_pentest_research")
DATA = ROOT / "finetuning/rebuilt_v4/test_sample_2000.jsonl"
OLD = ROOT / "qwen_api_security_lora_resume_3000/checkpoint-4500"
NEW = ROOT / "qwen_api_security_lora_rebuilt_v4"
OUT = ROOT / "finetuning/rebuilt_v4/evaluation"

OUT.mkdir(parents=True, exist_ok=True)

with DATA.open(encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

labels = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(BASE)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

prompts = [
    tokenizer.apply_chat_template(
        [
            {
                "role": "system",
                "content": "You are an API security classifier. Return only one category: Benign, Cookie Injection, Directory Traversal, LOG4J, Log Forging, RCE, SQL Injection, or XSS."
            },
            {"role": "user", "content": item["input"]}
        ],
        tokenize=False,
        add_generation_prompt=True
    )
    for item in records
]

y_true = [item["category"] for item in records]

def evaluate_adapter(adapter_path, name, gpu_id):
    torch.cuda.set_device(gpu_id)
    print(f"\n{name}: starting on GPU {gpu_id}", flush=True)

    quant = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

    base = AutoModelForCausalLM.from_pretrained(
        BASE,
        quantization_config=quant,
        dtype=torch.float16,
        device_map={"": gpu_id}
    )

    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    predictions = []
    batch_size = 8

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        enc = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024
        ).to(f"cuda:{gpu_id}")

        with torch.inference_mode():
            output = model.generate(
                **enc,
                max_new_tokens=8,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        generated = tokenizer.batch_decode(
            output[:, enc["input_ids"].shape[1]:],
            skip_special_tokens=True
        )

        for answer in generated:
            answer = answer.strip()
            prediction = next(
                (label for label in labels if answer.startswith(label)),
                "INVALID"
            )
            predictions.append(prediction)

        if (start + batch_size) % 200 == 0 or start + batch_size >= len(prompts):
            print(
                f"{name}: {min(start + batch_size, len(prompts))}/{len(prompts)}",
                flush=True
            )

    valid_pairs = [
        (actual, pred)
        for actual, pred in zip(y_true, predictions)
        if pred in labels
    ]

    result = {
        "model": name,
        "gpu": gpu_id,
        "total": len(y_true),
        "invalid": predictions.count("INVALID"),
        "accuracy": accuracy_score(y_true, predictions),
        "macro_f1": f1_score(y_true, predictions, labels=labels, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, predictions, labels=labels, average="weighted", zero_division=0),
        "classification_report": classification_report(
            y_true, predictions, labels=labels, zero_division=0, output_dict=True
        ),
        "confusion_matrix": confusion_matrix(y_true, predictions, labels=labels).tolist(),
        "predictions": predictions
    }

    output_path = OUT / f"{name.lower()}_sample2000.json"
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    print(f"\n{name} results")
    print(f"Accuracy:   {result['accuracy']:.4f}")
    print(f"Macro F1:   {result['macro_f1']:.4f}")
    print(f"Weighted F1:{result['weighted_f1']:.4f}")
    print(f"Invalid:    {result['invalid']}")
    print(classification_report(y_true, predictions, labels=labels, zero_division=0))
    print("Saved:", output_path, flush=True)

    del model, base
    gc.collect()
    torch.cuda.empty_cache()
    return result

with ThreadPoolExecutor(max_workers=2) as pool:
    old_future = pool.submit(evaluate_adapter, OLD, "Original", 0)
    new_future = pool.submit(evaluate_adapter, NEW, "RebuiltV4", 1)
    old_result = old_future.result()
    new_result = new_future.result()

print("\n=== FINAL COMPARISON ===")
print(f"{'Metric':<18}{'Original':>12}{'RebuiltV4':>14}")
for metric in ["accuracy", "macro_f1", "weighted_f1"]:
    print(f"{metric:<18}{old_result[metric]:>12.4f}{new_result[metric]:>14.4f}")

print("\nEvaluation completed. Both saved adapters remain unchanged.")

In [ ]:

import json
import shutil
import zipfile
from pathlib import Path
from datetime import datetime

ROOT = Path("/kaggle/working/api_pentest_research")
BACKUP_ROOT = Path("/kaggle/working/MSK_API_Security_Recovery")
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
PACKAGE = BACKUP_ROOT / f"recovery_{STAMP}"
PACKAGE.mkdir(parents=True, exist_ok=True)

def copy_file(src, dst):
    src = Path(src)
    if src.exists() and src.is_file():
        dst = PACKAGE / dst
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print("Saved:", src.name)
        return True
    print("NOT FOUND:", src)
    return False

def copy_folder(src, dst_name):
    src = Path(src)
    if src.exists() and src.is_dir():
        shutil.copytree(src, PACKAGE / dst_name, dirs_exist_ok=True)
        print("Saved folder:", src.name)
        return True
    print("NOT FOUND:", src)
    return False

# 1. Preserve the original model ZIP if available
copy_file(
    "/kaggle/working/MSK_API_Security_Model_step4500.zip",
    "original_model/MSK_API_Security_Model_step4500.zip"
)

# 2. Preserve the original checkpoint configuration and adapter
copy_folder(
    ROOT / "qwen_api_security_lora_resume_3000/checkpoint-4500",
    "original_model/checkpoint-4500"
)

# 3. Preserve the new trained adapter
copy_folder(
    ROOT / "qwen_api_security_lora_rebuilt_v4",
    "rebuilt_v4/adapter_and_checkpoints"
)

# 4. Preserve the training script
copy_file(
    ROOT / "finetuning/train_rebuilt_v4.py",
    "rebuilt_v4/train_rebuilt_v4.py"
)

# 5. Preserve the exact evaluation sample and both evaluation reports
copy_file(
    ROOT / "finetuning/rebuilt_v4/test_sample_2000.jsonl",
    "evaluation/test_sample_2000.jsonl"
)
copy_file(
    ROOT / "finetuning/rebuilt_v4/evaluation/original_sample2000.json",
    "evaluation/original_sample2000.json"
)
copy_file(
    ROOT / "finetuning/rebuilt_v4/evaluation/rebuiltv4_sample2000.json",
    "evaluation/rebuiltv4_sample2000.json"
)

# 6. Save a project handoff document
notes = r"""
# MSK API Security Model — Recovery Notes

## Base model
Qwen/Qwen2.5-1.5B-Instruct

## Original model
Project path:
api_pentest_research/qwen_api_security_lora_resume_3000/checkpoint-4500

Original ZIP:
MSK_API_Security_Model_step4500.zip

Do not overwrite the original checkpoint or ZIP.

## RebuiltV4
Adapter path:
api_pentest_research/qwen_api_security_lora_rebuilt_v4

Training script:
api_pentest_research/finetuning/train_rebuilt_v4.py

Training configuration:
- Fresh LoRA adapter
- Rank 16, alpha 32, dropout 0.05
- 4-bit NF4 quantization
- FP16 compute
- Effective batch size 16
- 300 training steps
- Save/evaluate every 100 steps
- Two Tesla T4 GPUs used for training

Reduced training dataset:
api_pentest_research/finetuning/rebuilt_v4/train.jsonl
9,000 examples: 2,000 Benign and 1,000 per attack class.

## Evaluation on the same 2,000-example sample

Original:
- Accuracy: 0.9920
- Macro F1: 0.9828
- Weighted F1: 0.9937
- Invalid outputs: 7

RebuiltV4:
- Accuracy: 0.9965
- Macro F1: 0.9924
- Weighted F1: 0.9981
- Invalid outputs: 7

Sample results favor RebuiltV4, but this is not yet a final model selection.

## LOG4J-only evaluation on 687 examples

Original:
- Detected: 684/687
- Recall: 99.5633%
- Three INVALID predictions

RebuiltV4:
- Detected: 683/687
- Recall: 99.4178%
- Three INVALID and one RCE prediction

The 2,000-example sample contained only 30 LOG4J examples. Do not infer perfect full-dataset performance from that small sample.

## Next steps
1. Restore or reopen the Kaggle project and verify the saved files.
2. Evaluate both adapters on the complete 45,912-example test set.
3. Use both Tesla T4 GPUs in parallel where suitable, after checking VRAM.
4. Compare accuracy, macro F1, weighted F1, per-class precision/recall/F1, invalid outputs, and confusion matrices.
5. Investigate Cookie Injection recall and invalid outputs.
6. Do not retrain until the full-test results are reviewed.
7. Keep the original model and ZIP unchanged.

## Important paths
Project root: /kaggle/working/api_pentest_research
New adapter: /kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4
Evaluation reports:
finetuning/rebuilt_v4/evaluation/

## Persistence warning
This recovery package is a copy of the listed artifacts, not a complete copy of every dataset or the entire project. Download the ZIP and save a Kaggle notebook version. The full original datasets are not included.
"""

(PACKAGE / "PROJECT_RECOVERY_NOTES.md").write_text(
    notes.strip() + "\n", encoding="utf-8"
)

manifest = {
    "created_at": datetime.now().isoformat(),
    "project_root": str(ROOT),
    "included": [
        "Original model ZIP, if present",
        "Original checkpoint-4500, if present",
        "RebuiltV4 adapter and checkpoints",
        "RebuiltV4 training script",
        "2,000-example test sample",
        "Both saved evaluation reports",
        "Project recovery notes"
    ],
    "not_included": [
        "Full training, validation, and test datasets",
        "Base model weights, which can be downloaded again"
    ]
}
(PACKAGE / "manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

zip_path = Path("/kaggle/working") / f"MSK_API_Security_Recovery_{STAMP}.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in PACKAGE.rglob("*"):
        if file.is_file():
            z.write(file, file.relative_to(PACKAGE.parent))

print("\n=== RECOVERY PACKAGE CREATED ===")
print("Folder:", PACKAGE)
print("ZIP:", zip_path)
print("ZIP size: %.2f MB" % (zip_path.stat().st_size / (1024**2)))

with zipfile.ZipFile(zip_path, "r") as z:
    bad = z.testzip()
    print("ZIP integrity:", "PASS" if bad is None else f"FAILED: {bad}")
    print("Files packaged:", len(z.namelist()))

print("\nDownload the ZIP from Kaggle's /kaggle/working output.")
print("Also save a Kaggle notebook version before closing the session.")


In [ ]:

from pathlib import Path
import json
import os
from datetime import datetime

WORK = Path("/kaggle/working")
ROOT = WORK / "api_pentest_research"
OUT = WORK / "MSK_API_Recovery_Inventory.json"

important = [
    WORK / "MSK_API_Security_Model_step4500.zip",
    WORK / "MSK_API_Security_Recovery_20261020_114027.zip",
    ROOT / "qwen_api_security_lora_resume_3000/checkpoint-4500",
    ROOT / "qwen_api_security_lora_rebuilt_v4",
    ROOT / "finetuning/train_rebuilt_v4.py",
    ROOT / "finetuning/rebuilt_v4/train.jsonl",
    ROOT / "finetuning/rebuilt_v4/test_sample_2000.jsonl",
    ROOT / "finetuning/rebuilt_v4/evaluation/original_sample2000.json",
    ROOT / "finetuning/rebuilt_v4/evaluation/rebuiltv4_sample2000.json",
]

def describe(path):
    result = {
        "path": str(path),
        "exists": path.exists(),
        "type": "directory" if path.is_dir() else "file"
    }
    if path.is_file():
        result["size_bytes"] = path.stat().st_size
    elif path.is_dir():
        result["file_count"] = sum(1 for p in path.rglob("*") if p.is_file())
    return result

inventory = {
    "created_at": datetime.now().isoformat(),
    "note": "Inventory only; no training/evaluation was run.",
    "important_artifacts": [describe(p) for p in important],
    "trainer_checkpoints": [],
    "working_directory_files": []
}

for base in [
    ROOT / "qwen_api_security_lora_resume_3000",
    ROOT / "qwen_api_security_lora_rebuilt_v4"
]:
    if base.exists():
        for p in sorted(base.glob("checkpoint-*")):
            inventory["trainer_checkpoints"].append(describe(p))

for p in WORK.iterdir():
    inventory["working_directory_files"].append(describe(p))

OUT.write_text(json.dumps(inventory, indent=2), encoding="utf-8")

print("Inventory saved:", OUT)
print("\nImportant artifacts:")
for item in inventory["important_artifacts"]:
    print(
        ("FOUND " if item["exists"] else "MISSING "),
        item["path"],
        f"({item.get('size_bytes', item.get('file_count', ''))})"
    )

print("\nTrainer checkpoints:")
for item in inventory["trainer_checkpoints"]:
    print(item["path"], "files:", item.get("file_count", "?"))

print("\nNo training or evaluation cells were executed.")


In [ ]:
import os
import json
import glob

base = "/kaggle/working/api_pentest_research"

paths = [
    base,
    base + "/qwen_api_security_lora_rebuilt_v4",
    base + "/finetuning/rebuilt_v4"
]

for path in paths:
    print("\nPATH:", path)
    print("Exists:", os.path.exists(path))

    if os.path.isdir(path):
        for name in sorted(os.listdir(path))[:30]:
            print(" ", name)

print("\n=== CHECKPOINTS AND TRAINING HISTORY ===")

for path in glob.glob(base + "/**/trainer_state.json", recursive=True):
    print("\nFILE:", path)

    try:
        with open(path, "r") as f:
            state = json.load(f)

        print("Global steps:", state.get("global_step"))
        print("Best checkpoint:", state.get("best_model_checkpoint"))
        print("Best metric:", state.get("best_metric"))
        print("Logged training history:")

        for item in state.get("log_history", [])[-10:]:
            print(item)

    except Exception as e:
        print("Could not read:", e)

print("\n=== ADAPTER FILES ===")

for path in glob.glob(base + "/**/adapter_config.json", recursive=True):
    print("\nAdapter:", os.path.dirname(path))

    with open(path, "r") as f:
        config = json.load(f)

    print("Rank:", config.get("r"))
    print("Alpha:", config.get("lora_alpha"))
    print("Base model:", config.get("base_model_name_or_path"))

In [ ]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for name in files:
        path = os.path.join(root, name)
        print(path, f"({os.path.getsize(path) / (1024 * 1024):.2f} MB)")

In [ ]:
import os

print("Input directory exists:", os.path.exists("/kaggle/input"))
print("Input directory contents:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "Not found")

print("\nWorking directory:", os.getcwd())
print("Working directory contents:", os.listdir("/kaggle/working")[:20])

In [ ]:
import os

paths = [
    "/kaggle/working/MSK_API_Security_Recovery",
    "/kaggle/working/api_pentest_research",
    "/kaggle/working/MSK_API_Recovery_Inventory.json"
]

for path in paths:
    print("\n", path)
    print("Exists:", os.path.exists(path))

    if os.path.isdir(path):
        for root, dirs, files in os.walk(path):
            level = root.replace(path, "").count(os.sep)
            if level > 3:
                dirs[:] = []
                continue

            print("  " * level + os.path.basename(root) + "/")

            for name in files[:15]:
                full_path = os.path.join(root, name)
                size = os.path.getsize(full_path) / (1024 * 1024)
                print("  " * (level + 1) + f"{name} ({size:.2f} MB)")

    elif os.path.isfile(path):
        with open(path, "r") as f:
            print(f.read()[:5000])

In [ ]:
import json
import os

base = "/kaggle/working/MSK_API_Security_Recovery/recovery_20261020_114027/evaluation"

files = [
    "original_sample2000.json",
    "rebuiltv4_sample2000.json"
]

for name in files:
    path = os.path.join(base, name)
    print("\n" + "=" * 60)
    print(name)

    with open(path, "r") as f:
        report = json.load(f)

    print(json.dumps(report, indent=2)[:12000])

In [ ]:
import os

paths = [
    "/kaggle/working/api_pentest_research/finetuning/train_rebuilt_v4.py",
    "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/train.jsonl",
    "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/test_sample_2000.jsonl",
    "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/evaluation"
]

for path in paths:
    print("\nPATH:", path, "| Exists:", os.path.exists(path))

    if os.path.isfile(path) and path.endswith(".py"):
        with open(path, "r") as f:
            print(f.read()[:18000])

    elif os.path.isfile(path) and path.endswith(".jsonl"):
        with open(path, "r") as f:
            for i, line in enumerate(f):
                if i >= 2:
                    break
                print(line[:1200])

    elif os.path.isdir(path):
        print("Files:", os.listdir(path))

In [ ]:
import os
import json

root = "/kaggle/working/api_pentest_research"

paths = [
    root + "/finetuning/rebuilt_v4/evaluation/original_sample2000.json",
    root + "/finetuning/rebuilt_v4/evaluation/rebuiltv4_sample2000.json",
    root + "/finetuning/rebuilt_v3/validation_sample_2000.jsonl",
    root + "/finetuning/rebuilt_v4/test_sample_2000.jsonl",
    root + "/qwen_api_security_lora_rebuilt_v4/checkpoint-200/trainer_state.json",
    root + "/qwen_api_security_lora_rebuilt_v4/checkpoint-300/trainer_state.json",
]

for path in paths:
    print("\n" + "=" * 80)
    print("FILE:", path)
    print("EXISTS:", os.path.exists(path))

    if not os.path.exists(path):
        continue

    if path.endswith(".json"):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        print("Top-level keys:", list(data.keys()))

        for key, value in data.items():
            if key in ["predictions", "y_true", "y_pred"]:
                print(key, "count:", len(value) if isinstance(value, list) else type(value).__name__)
                if isinstance(value, list):
                    print("First 3:", value[:3])
            elif isinstance(value, (dict, list)):
                print(key, json.dumps(value, ensure_ascii=False)[:2500])
            else:
                print(key, value)

    elif path.endswith(".jsonl"):
        count = 0
        labels = {}
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)
                count += 1
                label = item.get("category", item.get("label", "MISSING"))
                labels[label] = labels.get(label, 0) + 1
        print("Rows:", count)
        print("Label distribution:", labels)

    elif path.endswith("trainer_state.json"):
        with open(path, "r", encoding="utf-8") as f:
            state = json.load(f)

        print("Global step:", state.get("global_step"))
        print("Best checkpoint:", state.get("best_model_checkpoint"))
        print("Recent evaluation logs:")
        eval_logs = [
            x for x in state.get("log_history", [])
            if "eval_loss" in x
        ]
        for entry in eval_logs[-5:]:
            print(entry)

In [ ]:
import json
from pathlib import Path
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, classification_report

root = Path("/kaggle/working/api_pentest_research/finetuning")

val_path = root / "rebuilt_v3" / "validation_sample_2000.jsonl"
test_path = root / "rebuilt_v4" / "test_sample_2000.jsonl"
eval_dir = root / "rebuilt_v4" / "evaluation"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

val_rows = load_jsonl(val_path)
test_rows = load_jsonl(test_path)

def get_input(row):
    return row.get("input", row.get("text", ""))

val_inputs = {get_input(row) for row in val_rows}
test_inputs = {get_input(row) for row in test_rows}

print("VALIDATION rows:", len(val_rows))
print("TEST rows:", len(test_rows))
print("Unique validation inputs:", len(val_inputs))
print("Unique test inputs:", len(test_inputs))
print("Exact input overlap:", len(val_inputs & test_inputs))

labels = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

y_true = [row["category"] for row in test_rows]

for filename in ["original_sample2000.json", "rebuiltv4_sample2000.json"]:
    path = eval_dir / filename
    with open(path, "r", encoding="utf-8") as f:
        report = json.load(f)

    preds = report["predictions"]

    print("\n" + "=" * 60)
    print("MODEL:", report.get("model", filename))
    print("Prediction count:", len(preds))
    print("Invalid predictions:", sum(p not in labels for p in preds))

    if len(preds) != len(y_true):
        print("ERROR: Prediction count does not match test rows.")
        continue

    # Invalid outputs count as incorrect for accuracy.
    correct = sum(p == y and p in labels for p, y in zip(preds, y_true))
    accuracy = correct / len(y_true)

    # Treat invalid outputs as a separate prediction class for the audit.
    audited_preds = [p if p in labels else "INVALID" for p in preds]

    print(f"Recalculated accuracy: {accuracy:.4%}")
    print("Saved accuracy:", report.get("accuracy"))
    print("Saved macro F1:", report.get("macro_f1"))
    print("Recalculated macro F1 (valid classes only):",
          f1_score(y_true, audited_preds, labels=labels,
                   average="macro", zero_division=0))
    print("Prediction distribution:", Counter(audited_preds))
    print("\nPer-class report:")
    print(classification_report(
        y_true, audited_preds,
        labels=labels + ["INVALID"],
        zero_division=0,
        digits=4
    ))

In [ ]:
import os
import torch
from pathlib import Path

root = Path("/kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4")

for step in [200, 300]:
    path = root / f"checkpoint-{step}"
    print(f"\nCheckpoint {step}: {path}")
    print("Exists:", path.exists())

    if path.exists():
        for name in [
            "adapter_config.json",
            "adapter_model.safetensors",
            "model.safetensors",
            "pytorch_model.bin",
        ]:
            file_path = path / name
            if file_path.exists():
                print(name, f"{file_path.stat().st_size / (1024**2):.2f} MB")

print("\nGPU status:")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)}, "
            f"free {free / 1024**3:.2f} GB / "
            f"total {total / 1024**3:.2f} GB"
        )
else:
    print("CUDA unavailable")

In [ ]:
import importlib.metadata as md
import torch

for package in ["bitsandbytes", "transformers", "peft", "accelerate"]:
    try:
        print(f"{package}: {md.version(package)}")
    except md.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
import importlib.util
import traceback

print("bitsandbytes module:", importlib.util.find_spec("bitsandbytes"))

try:
    import bitsandbytes as bnb
    print("Imported bitsandbytes:", bnb.__version__)

    from transformers.utils import is_bitsandbytes_available
    print("Transformers detects bitsandbytes:", is_bitsandbytes_available())

    import torch
    print("CUDA available:", torch.cuda.is_available())

    print("4-bit linear layer:", bnb.nn.Linear4bit)

except Exception:
    traceback.print_exc()

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

base_name = "Qwen/Qwen2.5-1.5B-Instruct"

print("Loading base model in FP16...")

tokenizer = AutoTokenizer.from_pretrained(base_name)

model = AutoModelForCausalLM.from_pretrained(
    base_name,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

print("Model loaded successfully.")
print("Device:", next(model.parameters()).device)
print("Dtype:", next(model.parameters()).dtype)
print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

del model, tokenizer
torch.cuda.empty_cache()

print("Base-model loading test completed; GPU cache cleared.")

In [ ]:
import importlib.metadata as md
import peft.import_utils as iu
import inspect

print("PEFT version:", md.version("peft"))
print("torchao version:", md.version("torchao"))

print("\nPEFT torchao availability check:")
print(inspect.getsource(iu.is_torchao_available))

In [ ]:
import torch
import peft.tuners.lora.torchao as lora_torchao
from transformers import AutoModelForCausalLM
from peft import PeftModel

base_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4/checkpoint-200"

lora_torchao.is_torchao_available = lambda: False

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    base_name,
    dtype=torch.float16,
    device_map={"": 0},
)

print("Loading checkpoint-200 adapter...")

model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    low_cpu_mem_usage=False,
)

model.eval()

print("Adapter loaded successfully.")
print("GPU memory allocated:", round(torch.cuda.memory_allocated(0) / 1024**3, 2), "GB")

In [ ]:
import json
import torch

test_path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/test_sample_2000.jsonl"

system_prompt = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

with open(test_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()][:8]

tokenizer = globals().get("tokenizer")
if tokenizer is None:
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

for i, row in enumerate(rows, 1):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["input"]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output[0, inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    print(f"{i}. Actual: {row['category']} | Predicted: {prediction!r}")

In [ ]:
import json
import torch

test_path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/test_sample_2000.jsonl"

system_prompt = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

with open(test_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

categories = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

samples = {}
for row in rows:
    if row["category"] not in samples:
        samples[row["category"]] = row
    if len(samples) == len(categories):
        break

for category in categories:
    row = samples.get(category)
    if row is None:
        print(f"{category}: NO TEST EXAMPLE FOUND")
        continue

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": row["input"]},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:0")

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output[0, inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    print(f"Actual: {category} | Predicted: {prediction!r}")

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {torch.cuda.get_device_name(i)} | "
        f"Free: {free / 1024**3:.2f} GB / "
        f"Total: {total / 1024**3:.2f} GB | "
        f"Allocated by PyTorch: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB"
    )

In [ ]:
import gc
import torch

for name in ["model", "base_model", "tokenizer"]:
    if name in globals():
        del globals()[name]

gc.collect()

for i in range(torch.cuda.device_count()):
    with torch.cuda.device(i):
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

print("Released model objects and cleared unused GPU cache.")

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(
        f"GPU {i}: {free / 1024**3:.2f} GB free / "
        f"{total / 1024**3:.2f} GB total"
    )

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import gc
import torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import peft.tuners.lora.torchao as lora_torchao
from sklearn.metrics import accuracy_score, f1_score, classification_report
from collections import Counter

lora_torchao.is_torchao_available = lambda: False

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "/kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4/checkpoint-200"
TEST = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/test_sample_2000.jsonl"
OUT = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/evaluation/checkpoint200_2gpu_fp16.json"

LABELS = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

SYSTEM = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

with open(TEST, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

def evaluate_shard(gpu_id, start, end):
    torch.cuda.set_device(gpu_id)
    tokenizer = AutoTokenizer.from_pretrained(BASE)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE,
        dtype=torch.float16,
        device_map={"": gpu_id},
    )
    model = PeftModel.from_pretrained(
        base, ADAPTER, low_cpu_mem_usage=False
    )
    model.eval()

    result = []
    for idx in range(start, end):
        messages = [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": rows[idx]["input"]},
        ]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(f"cuda:{gpu_id}")

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        generated = output[0, inputs["input_ids"].shape[1]:]
        pred = tokenizer.decode(generated, skip_special_tokens=True).strip()
        result.append((idx, pred))

        if (idx - start + 1) % 100 == 0:
            print(f"GPU {gpu_id}: {idx - start + 1}/{end - start}")

    del model, base, tokenizer
    gc.collect()
    with torch.cuda.device(gpu_id):
        torch.cuda.empty_cache()
        torch.cuda.synchronize(gpu_id)

    return result

mid = len(rows) // 2
print(f"Evaluating {len(rows)} requests across two GPUs.")
print(f"GPU 0: requests 1–{mid}; GPU 1: requests {mid + 1}–{len(rows)}")

with ThreadPoolExecutor(max_workers=2) as pool:
    future0 = pool.submit(evaluate_shard, 0, 0, mid)
    future1 = pool.submit(evaluate_shard, 1, mid, len(rows))
    results = future0.result() + future1.result()

results.sort(key=lambda x: x[0])
predictions = [pred for _, pred in results]
y_true = [row["category"] for row in rows]

invalid = sum(p not in LABELS for p in predictions)
accuracy = sum(
    p == y and p in LABELS for p, y in zip(predictions, y_true)
) / len(y_true)

report = {
    "model": "RebuiltV4 checkpoint-200 FP16, two-GPU inference",
    "total": len(rows),
    "invalid": invalid,
    "accuracy": accuracy,
    "macro_f1": f1_score(y_true, predictions, labels=LABELS,
                          average="macro", zero_division=0),
    "weighted_f1": f1_score(y_true, predictions, labels=LABELS,
                            average="weighted", zero_division=0),
    "classification_report": classification_report(
        y_true, predictions, labels=LABELS,
        output_dict=True, zero_division=0
    ),
    "prediction_distribution": dict(Counter(predictions)),
    "predictions": predictions,
}

Path(OUT).parent.mkdir(parents=True, exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n=== CHECKPOINT-200 TWO-GPU RESULTS ===")
print("Accuracy:", round(accuracy * 100, 3), "%")
print("Macro F1:", round(report["macro_f1"], 4))
print("Weighted F1:", round(report["weighted_f1"], 4))
print("Invalid outputs:", invalid)
print(classification_report(
    y_true, predictions, labels=LABELS, zero_division=0, digits=4
))
print("Saved:", OUT)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import gc
import torch
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import peft.tuners.lora.torchao as lora_torchao
from sklearn.metrics import f1_score, classification_report
from collections import Counter

lora_torchao.is_torchao_available = lambda: False
torch.set_num_threads(2)

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER = "/kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4/checkpoint-300"
TEST = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/test_sample_2000.jsonl"
OUT = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/evaluation/checkpoint300_2gpu_fp16.json"
BATCH_SIZE = 4

LABELS = [
    "Benign", "Cookie Injection", "Directory Traversal", "LOG4J",
    "Log Forging", "RCE", "SQL Injection", "XSS"
]

SYSTEM = (
    "You are an API security classifier. "
    "Classify the HTTP request into exactly one of these categories: "
    "Benign, Cookie Injection, Directory Traversal, LOG4J, "
    "Log Forging, RCE, SQL Injection, or XSS. "
    "Return only the category name."
)

with open(TEST, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

def evaluate_shard(gpu_id, start, end):
    torch.cuda.set_device(gpu_id)

    tokenizer = AutoTokenizer.from_pretrained(BASE)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE,
        dtype=torch.float16,
        device_map={"": gpu_id},
    )
    model = PeftModel.from_pretrained(
        base, ADAPTER, low_cpu_mem_usage=False
    )
    model.eval()

    indexed_predictions = []
    shard_indices = list(range(start, end))
    total_batches = (len(shard_indices) + BATCH_SIZE - 1) // BATCH_SIZE

    for batch_num, offset in enumerate(
        range(0, len(shard_indices), BATCH_SIZE), 1
    ):
        indices = shard_indices[offset:offset + BATCH_SIZE]
        prompts = []

        for idx in indices:
            messages = [
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": rows[idx]["input"]},
            ]
            prompts.append(tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            ))

        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(f"cuda:{gpu_id}")

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=20,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
        decoded = tokenizer.batch_decode(
            new_tokens, skip_special_tokens=True
        )

        indexed_predictions.extend(zip(indices, [p.strip() for p in decoded]))

        if batch_num % 25 == 0 or batch_num == total_batches:
            completed = min(batch_num * BATCH_SIZE, len(shard_indices))
            print(f"GPU {gpu_id}: {completed}/{len(shard_indices)} requests")

    del model, base, tokenizer
    gc.collect()
    with torch.cuda.device(gpu_id):
        torch.cuda.synchronize(gpu_id)
        torch.cuda.empty_cache()

    return indexed_predictions

mid = len(rows) // 2
print(f"Checkpoint-300 evaluation: {len(rows)} requests")
print(f"GPU 0: {mid} requests | GPU 1: {len(rows) - mid} requests")
print(f"Batch size per GPU: {BATCH_SIZE}")

with ThreadPoolExecutor(max_workers=2) as pool:
    future0 = pool.submit(evaluate_shard, 0, 0, mid)
    future1 = pool.submit(evaluate_shard, 1, mid, len(rows))
    results = future0.result() + future1.result()

results.sort(key=lambda item: item[0])
predictions = [pred for _, pred in results]
y_true = [row["category"] for row in rows]

invalid = sum(pred not in LABELS for pred in predictions)
accuracy = sum(
    pred == actual and pred in LABELS
    for pred, actual in zip(predictions, y_true)
) / len(y_true)

report = {
    "model": "RebuiltV4 checkpoint-300 FP16, two-GPU batched inference",
    "total": len(rows),
    "invalid": invalid,
    "accuracy": accuracy,
    "macro_f1": f1_score(
        y_true, predictions, labels=LABELS,
        average="macro", zero_division=0
    ),
    "weighted_f1": f1_score(
        y_true, predictions, labels=LABELS,
        average="weighted", zero_division=0
    ),
    "classification_report": classification_report(
        y_true, predictions, labels=LABELS,
        output_dict=True, zero_division=0
    ),
    "prediction_distribution": dict(Counter(predictions)),
    "predictions": predictions,
}

Path(OUT).parent.mkdir(parents=True, exist_ok=True)
with open(OUT, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\n=== CHECKPOINT-300 RESULTS ===")
print(f"Accuracy: {accuracy * 100:.3f}%")
print(f"Macro F1: {report['macro_f1']:.4f}")
print(f"Weighted F1: {report['weighted_f1']:.4f}")
print("Invalid outputs:", invalid)
print(classification_report(
    y_true, predictions, labels=LABELS,
    zero_division=0, digits=4
))
print("Saved report:", OUT)

In [ ]:
import os
import json
from collections import Counter

base = "/kaggle/working/api_pentest_research/finetuning"

candidates = [
    os.path.join(base, "test_holdout.jsonl"),
    os.path.join(base, "validation_sft.jsonl"),
    os.path.join(base, "combined_sft.jsonl"),
    os.path.join(base, "train_final.jsonl"),
]

for path in candidates:
    if os.path.isfile(path):
        print(f"\nFILE: {path}")
        print(f"SIZE: {os.path.getsize(path) / (1024**2):.2f} MB")

        with open(path, "r", encoding="utf-8") as f:
            for i, line in enumerate(f):
                if i >= 2:
                    break
                try:
                    record = json.loads(line)
                    print(f"RECORD {i+1} KEYS: {list(record.keys())}")
                    print(f"RECORD {i+1} SAMPLE: {str(record)[:500]}")
                except Exception as e:
                    print("JSON ERROR:", str(e))

In [ ]:
import os
import json
from collections import Counter

base = "/kaggle/working/api_pentest_research/finetuning"
test_path = os.path.join(base, "test_holdout.jsonl")
train_path = os.path.join(base, "train_final.jsonl")
val_path = os.path.join(base, "validation_sft.jsonl")

def read_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

test = read_jsonl(test_path)

print("TEST RECORDS:", len(test))
print("TEST CATEGORIES:")
for category, count in sorted(Counter(r.get("category", "MISSING") for r in test).items()):
    print(f"  {category}: {count}")

print("\nTEST SOURCES:")
for source, count in Counter(r.get("source_dataset", "MISSING") for r in test).most_common():
    print(f"  {source}: {count}")

print("\nMISSING OR INVALID FIELDS:")
print("Missing input:", sum(not r.get("input") for r in test))
print("Missing category:", sum(not r.get("category") for r in test))
print("Missing binary label:", sum("binary_label" not in r for r in test))

test_inputs = [r["input"] for r in test]
print("\nDUPLICATE TEST INPUTS:", len(test_inputs) - len(set(test_inputs)))

test_set = set(test_inputs)

for name, path in [("TRAIN", train_path), ("VALIDATION", val_path)]:
    if not os.path.isfile(path):
        print(f"\n{name} FILE NOT FOUND:", path)
        continue

    overlap = set()
    rows = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            record = json.loads(line)
            rows += 1
            value = record.get("input")
            if value in test_set:
                overlap.add(value)

    print(f"\n{name} RECORDS:", rows)
    print(f"{name} UNIQUE INPUTS OVERLAPPING TEST:", len(overlap))
    print(f"{name} TEST OVERLAP PERCENT:", round(100 * len(overlap) / max(len(test_set), 1), 4))

In [ ]:
import os
import json
from collections import defaultdict, Counter

base = "/kaggle/working/api_pentest_research/finetuning"
test_path = os.path.join(base, "test_holdout.jsonl")
output_path = os.path.join(base, "test_holdout_deduplicated.jsonl")

groups = defaultdict(list)

with open(test_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            groups[record["input"]].append(record)

conflicts = []
deduplicated = []

for input_text, records in groups.items():
    labels = set(r["category"] for r in records)

    if len(labels) > 1:
        conflicts.append({
            "input": input_text,
            "labels": sorted(labels),
            "count": len(records)
        })
    else:
        deduplicated.append(records[0])

print("Original records:", sum(len(v) for v in groups.values()))
print("Unique inputs:", len(groups))
print("Inputs with conflicting labels:", len(conflicts))
print("Records retained after deduplication:", len(deduplicated))
print("Deduplicated category distribution:")
for category, count in sorted(Counter(r["category"] for r in deduplicated).items()):
    print(f"  {category}: {count}")

if conflicts:
    print("\nExamples of conflicting labels:")
    for item in conflicts[:5]:
        print(item["labels"], "| duplicate count:", item["count"])
else:
    with open(output_path, "w", encoding="utf-8") as f:
        for record in deduplicated:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print("\nSaved:", output_path)

In [ ]:
import os
import torch

base = "/kaggle/working/api_pentest_research/finetuning"
test_path = os.path.join(base, "test_holdout_deduplicated.jsonl")

checkpoints = {
    "checkpoint-200": os.path.join(
        base, "rebuilt_v4", "checkpoint-200"
    ),
    "checkpoint-300": os.path.join(
        base, "rebuilt_v4", "checkpoint-300"
    ),
    "final_adapter": "/kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4"
}

print("Deduplicated test file exists:", os.path.isfile(test_path))

for name, path in checkpoints.items():
    print(f"\n{name}: {path}")
    print("Exists:", os.path.isdir(path))
    if os.path.isdir(path):
        print("Adapter weights:", os.path.isfile(os.path.join(path, "adapter_model.safetensors")))
        print("Adapter config:", os.path.isfile(os.path.join(path, "adapter_config.json")))

print("\nPyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Free/total memory (GB): "
          f"{torch.cuda.mem_get_info(i)[0] / 1024**3:.2f}/"
          f"{torch.cuda.mem_get_info(i)[1] / 1024**3:.2f}")

In [ ]:
import os
import json
from collections import defaultdict, Counter

root = "/kaggle/working/api_pentest_research"
test_path = os.path.join(root, "finetuning", "test_holdout.jsonl")

print("=== FIND CHECKPOINTS ===")

for current, dirs, files in os.walk(root):
    dirs[:] = [d for d in dirs if d not in {".git", "__pycache__"}]

    if "adapter_model.safetensors" in files:
        print("ADAPTER:", current)

    if "trainer_state.json" in files and "checkpoint-" in current:
        print("CHECKPOINT:", current)

print("\n=== REBUILD CLEAN TEST FILE ===")

groups = defaultdict(list)

with open(test_path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            record = json.loads(line)
            groups[record["input"]].append(record)

clean = []
conflicts = []

for input_text, records in groups.items():
    labels = {r["category"] for r in records}

    if len(labels) > 1:
        conflicts.append((input_text, sorted(labels)))
    else:
        clean.append(records[0])

output_path = os.path.join(
    root, "finetuning", "test_holdout_deduplicated.jsonl"
)

with open(output_path, "w", encoding="utf-8") as f:
    for record in clean:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print("Original test records:", sum(map(len, groups.values())))
print("Unique inputs:", len(groups))
print("Conflicting inputs excluded:", len(conflicts))
print("Clean test records:", len(clean))
print("Clean category counts:", dict(Counter(r["category"] for r in clean)))
print("Saved test file:", output_path)
print("File exists:", os.path.isfile(output_path))

In [ ]:
import os
import json
import torch
import transformers
import peft

root = "/kaggle/working/api_pentest_research"
adapter = os.path.join(root, "qwen_api_security_lora_rebuilt_v4")
test_path = os.path.join(
    root, "finetuning", "test_holdout_deduplicated.jsonl"
)

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("PyTorch:", torch.__version__)
print("GPU count:", torch.cuda.device_count())

with open(os.path.join(adapter, "adapter_config.json"), "r") as f:
    config = json.load(f)

print("\nBase model:", config.get("base_model_name_or_path"))
print("LoRA rank:", config.get("r"))
print("LoRA alpha:", config.get("lora_alpha"))
print("Target modules:", config.get("target_modules"))

for step in (200, 300):
    path = os.path.join(adapter, f"checkpoint-{step}")
    print(f"\nCheckpoint {step} exists:", os.path.isdir(path))
    if os.path.isdir(path):
        print("Adapter weights:", os.path.isfile(
            os.path.join(path, "adapter_model.safetensors")
        ))

print("\nClean test file exists:", os.path.isfile(test_path))
print("Clean test file size (MB):", round(os.path.getsize(test_path) / 1024**2, 2))

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | "
          f"Free VRAM: {free / 1024**3:.2f}/{total / 1024**3:.2f} GB")

In [ ]:
import os
import json
import torch
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Compatibility workaround for the current Kaggle PEFT/torchao combination
import peft.tuners.lora.torchao as lora_torchao
lora_torchao.is_torchao_available = lambda: False

ROOT = "/kaggle/working/api_pentest_research"
ADAPTER = os.path.join(ROOT, "qwen_api_security_lora_rebuilt_v4")
TEST = os.path.join(ROOT, "finetuning", "test_holdout_deduplicated.jsonl")
OUT = os.path.join(ROOT, "finetuning", "rebuilt_v4", "full_test_evaluation")
os.makedirs(OUT, exist_ok=True)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
BATCH_SIZE = 4
MAX_NEW_TOKENS = 12

with open(TEST, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

labels = [r["category"] for r in records]
inputs = [r["input"] for r in records]

print("Test records:", len(records))
print("GPUs:", torch.cuda.device_count())
print("Output directory:", OUT)

tokenizer = AutoTokenizer.from_pretrained(ADAPTER, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

def run_gpu(gpu_id, checkpoint_path, indexed_inputs):
    torch.cuda.set_device(gpu_id)
    model = None

    try:
        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            torch_dtype=torch.float16,
            device_map={"": gpu_id},
            trust_remote_code=True
        )
        model = PeftModel.from_pretrained(
            base, checkpoint_path, is_trainable=False
        )
        model.eval()

        results = []
        for start in range(0, len(indexed_inputs), BATCH_SIZE):
            batch = indexed_inputs[start:start + BATCH_SIZE]
            batch_texts = [inputs[i] for i in batch]

            enc = tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024
            )
            enc = {k: v.to(f"cuda:{gpu_id}") for k, v in enc.items()}

            with torch.inference_mode():
                output = model.generate(
                    **enc,
                    max_new_tokens=MAX_NEW_TOKENS,
                    do_sample=False,
                    num_beams=1,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                    use_cache=True
                )

            prompt_len = enc["input_ids"].shape[1]
            generated = output[:, prompt_len:]
            decoded = tokenizer.batch_decode(
                generated, skip_special_tokens=True
            )

            for idx, text in zip(batch, decoded):
                prediction = text.strip().splitlines()[0].strip()
                prediction = prediction.strip("`'\" .")
                results.append((idx, prediction))

            completed = min(start + BATCH_SIZE, len(indexed_inputs))
            if completed % 500 == 0 or completed == len(indexed_inputs):
                print(
                    f"GPU {gpu_id}: {completed}/{len(indexed_inputs)} "
                    f"for {os.path.basename(checkpoint_path)}",
                    flush=True
                )

        return results

    finally:
        if model is not None:
            del model
        torch.cuda.empty_cache()

def evaluate_checkpoint(step):
    checkpoint = os.path.join(ADAPTER, f"checkpoint-{step}")
    output_file = os.path.join(OUT, f"checkpoint_{step}_predictions.jsonl")
    progress_file = os.path.join(OUT, f"checkpoint_{step}_progress.json")

    if os.path.isfile(output_file):
        with open(output_file, "r", encoding="utf-8") as f:
            saved = [json.loads(line) for line in f if line.strip()]
        if len(saved) == len(records):
            print(f"Checkpoint {step}: complete predictions already exist; skipping.")
            return output_file

    split = (len(inputs) + 1) // 2
    partitions = [
        list(range(0, split)),
        list(range(split, len(inputs)))
    ]

    print(f"\nStarting checkpoint {step} on both GPUs...", flush=True)
    with ThreadPoolExecutor(max_workers=2) as pool:
        futures = [
            pool.submit(run_gpu, gpu, checkpoint, partitions[gpu])
            for gpu in range(2)
        ]
        parts = [future.result() for future in futures]

    predictions = [None] * len(inputs)
    for part in parts:
        for idx, pred in part:
            predictions[idx] = pred

    with open(output_file, "w", encoding="utf-8") as f:
        for i, pred in enumerate(predictions):
            f.write(json.dumps({
                "input": inputs[i],
                "true_category": labels[i],
                "predicted_category": pred
            }, ensure_ascii=False) + "\n")

    with open(progress_file, "w", encoding="utf-8") as f:
        json.dump({
            "checkpoint": step,
            "records": len(records),
            "completed": sum(p is not None for p in predictions),
            "batch_size": BATCH_SIZE,
            "max_new_tokens": MAX_NEW_TOKENS
        }, f, indent=2)

    print("Saved:", output_file, flush=True)
    return output_file

for step in (200, 300):
    evaluate_checkpoint(step)

print("\nEVALUATION INFERENCE FINISHED.")
print("Predictions saved in:", OUT)

In [ ]:
import torch

for i in range(torch.cuda.device_count()):
    print(f"\nGPU {i}: {torch.cuda.get_device_name(i)}")
    x = torch.randn(4096, 4096, device=f"cuda:{i}")
    y = x @ x
    torch.cuda.synchronize(i)
    print("Device:", y.device)
    print("Test passed:", torch.isfinite(y).all().item())
    del x, y
    torch.cuda.empty_cache()

In [ ]:
import os
import torch
from transformers import AutoTokenizer

root = "/kaggle/working/api_pentest_research"
adapter = os.path.join(root, "qwen_api_security_lora_rebuilt_v4")
test_path = os.path.join(
    root, "finetuning", "test_holdout_deduplicated.jsonl"
)

print("CUDA devices:", torch.cuda.device_count())
print("Checkpoint 200:", os.path.isfile(
    os.path.join(adapter, "checkpoint-200", "adapter_model.safetensors")
))
print("Checkpoint 300:", os.path.isfile(
    os.path.join(adapter, "checkpoint-300", "adapter_model.safetensors")
))
print("Test file:", os.path.isfile(test_path))

tokenizer = AutoTokenizer.from_pretrained(adapter, trust_remote_code=True)
print("Tokenizer loaded:", tokenizer.__class__.__name__)
print("Padding side:", tokenizer.padding_side)
print("Pad token ID:", tokenizer.pad_token_id)
print("EOS token ID:", tokenizer.eos_token_id)

for i in range(2):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
import torch
from transformers import AutoTokenizer

adapter = "/kaggle/working/api_pentest_research/qwen_api_security_lora_rebuilt_v4"

tokenizer = AutoTokenizer.from_pretrained(adapter, trust_remote_code=True)
tokenizer.padding_side = "left"

print("Padding side:", tokenizer.padding_side)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
!nvidia-smi

In [ ]:
import os

path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/full_test_evaluation"

for name in os.listdir(path):
    file_path = os.path.join(path, name)
    print(name, round(os.path.getsize(file_path) / 1024**2, 2), "MB")

In [ ]:
import os
import glob

path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/full_test_evaluation"

print("Directory exists:", os.path.isdir(path))

if os.path.isdir(path):
    for file in glob.glob(path + "/*"):
        print(
            os.path.basename(file),
            round(os.path.getsize(file) / (1024 * 1024), 2),
            "MB"
        )
else:
    print("Evaluation output directory not found.")

In [ ]:
import os

path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/full_test_evaluation"

files = os.listdir(path)
print("Number of files:", len(files))

if not files:
    print("No prediction files saved yet.")
else:
    for name in files:
        full_path = os.path.join(path, name)
        print(name, "|", os.path.getsize(full_path), "bytes")

In [ ]:
import os
import json

path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/full_test_evaluation"

print("Output files:", os.listdir(path))

progress_path = os.path.join(
    path, "checkpoint_200_progress.json"
)

if os.path.isfile(progress_path):
    with open(progress_path, "r") as f:
        print(json.load(f))
else:
    print("No completed progress record yet.")

In [ ]:
import os
import json

path = "/kaggle/working/api_pentest_research/finetuning/rebuilt_v4/full_test_evaluation"

print("Files:", os.listdir(path))

for name in os.listdir(path):
    full_path = os.path.join(path, name)
    if os.path.isfile(full_path):
        print(f"{name}: {os.path.getsize(full_path) / 1024**2:.2f} MB")

progress = os.path.join(path, "checkpoint_200_progress.json")
if os.path.isfile(progress):
    with open(progress, "r") as f:
        print("\nProgress record:", json.load(f))
else:
    print("\nNo completed progress record yet.")

In [ ]:
!nvidia-smi

In [ ]:
import os
import tarfile
import json
import time

WORK = "/kaggle/working"
BACKUP_DIR = os.path.join(WORK, "MSK_PRESERVE_BACKUP")
os.makedirs(BACKUP_DIR, exist_ok=True)

sources = [
    os.path.join(WORK, "api_pentest_research"),
    os.path.join(WORK, "MSK_API_Security_Recovery"),
    os.path.join(WORK, "MSK_API_Recovery_Inventory.json"),
]

archive = os.path.join(BACKUP_DIR, "MSK_API_Security_FULL_BACKUP.tar.gz")
manifest = os.path.join(BACKUP_DIR, "backup_manifest.json")

excluded_dirs = {
    "__pycache__", ".cache", ".git", "node_modules"
}

files_added = []
missing_sources = []

def filter_members(tar, path):
    for name in os.listdir(path):
        full = os.path.join(path, name)
        if os.path.isdir(full) and name in excluded_dirs:
            continue
        yield full

print("Creating backup. This may take several minutes...")

with tarfile.open(archive, "w:gz") as tar:
    for source in sources:
        if not os.path.exists(source):
            missing_sources.append(source)
            print("NOT FOUND:", source)
            continue

        arcname = os.path.relpath(source, WORK)
        tar.add(source, arcname=arcname, recursive=True,
                filter=None)

        if os.path.isfile(source):
            files_added.append(source)
        else:
            for current, dirs, files in os.walk(source):
                dirs[:] = [d for d in dirs if d not in excluded_dirs]
                for name in files:
                    if name.endswith(".pyc"):
                        continue
                    files_added.append(os.path.join(current, name))

print("\nVerifying archive...")
with tarfile.open(archive, "r:gz") as tar:
    members = tar.getmembers()
    bad = tarfile.is_tarfile(archive)
    archive_files = sum(m.isfile() for m in members)

details = {
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "archive": archive,
    "archive_size_gb": round(os.path.getsize(archive) / 1024**3, 3),
    "archive_members": len(members),
    "archive_files": archive_files,
    "missing_sources": missing_sources,
    "sources": sources
}

with open(manifest, "w") as f:
    json.dump(details, f, indent=2)

print(json.dumps(details, indent=2))
print("\nBackup archive:", archive)
print("Archive readable:", bad)

In [ ]:
import os

paths = [
    "/content/drive/MyDrive",
    "/kaggle/working",
    "/kaggle/input"
]

for path in paths:
    print(f"{path}: {'EXISTS' if os.path.exists(path) else 'NOT FOUND'}")

if os.path.exists("/content/drive/MyDrive"):
    print("\nGoogle Drive appears to be mounted!")
    print(os.listdir("/content/drive/MyDrive")[:20])

In [ ]:
import os

for path in [
    "/content/drive/MyDrive",
    "/kaggle/working",
    "/kaggle/input"
]:
    print(f"{path}: {'EXISTS' if os.path.exists(path) else 'NOT FOUND'}")

In [ ]:
!pip install -q google-api-python-client google-auth

In [ ]:
import importlib.util

for package in ["googleapiclient", "google.auth"]:
    print(package, "installed:", importlib.util.find_spec(package) is not None)

In [ ]:
%pip install google-api-python-client google-auth

In [ ]:
import os
import tarfile

archive = "/kaggle/working/MSK_PRESERVE_BACKUP/MSK_API_Security_FULL_BACKUP.tar.gz"

assert os.path.isfile(archive), "Backup archive not found!"

with tarfile.open(archive, "r:gz") as tar:
    print("Archive integrity:", tar.getmembers() is not None)
    print("Files and folders:", len(tar.getmembers()))

print("Backup size (GB):", round(os.path.getsize(archive) / 1024**3, 2))
print("Backup path:", archive)